In [1]:
import os, sys, shutil, torch, psutil
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
vm = psutil.virtual_memory()
print(f"RAM {vm.total/1e9:.1f} GB | CPUs {os.cpu_count()}")
for d in ["/kaggle/working", "/kaggle/temp", "/tmp"]:
    if os.path.exists(d):
        du = shutil.disk_usage(d)
        print(f"Disk {d:<16} free {du.free/1e9:6.1f} GB / {du.total/1e9:6.1f} GB")

Python 3.12.13 | torch 2.10.0+cu128 | CUDA: True
RAM 33.7 GB | CPUs 4
Disk /kaggle/working  free   20.9 GB /   21.0 GB
Disk /tmp             free 1201.7 GB / 8758.4 GB


In [1]:
# ===== Cell R1 (NEW ACCOUNT): restore /kaggle/working from the private HF backup =====
!pip install -q -U "huggingface_hub>=1.3,<2.0" hf_xet
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"                 # fast parallel download
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download, HfApi
tok = UserSecretsClient().get_secret("HF_WRITE_TOKEN")      # the write token can also read
REPO = f"{HfApi(token=tok).whoami()['name']}/cs781-phase1-work"
snapshot_download(REPO, local_dir="/kaggle/working", token=tok)
!du -sh /kaggle/working/* && ls -la /kaggle/working/checkpoints/*/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 16.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 67.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 113 files:   0%|          | 0/113 [00:00<?, ?it/s]

2.4G	/kaggle/working/checkpoints
3.1G	/kaggle/working/data
132K	/kaggle/working/src
357M	/kaggle/working/stats
4.7M	/kaggle/working/submissions
3.6M	/kaggle/working/tokenizer
/kaggle/working/checkpoints/m1_dense_100m/:
total 243568
drwxr-xr-x 2 root root      4096 Oct  1 19:33 .
drwxr-xr-x 6 root root      4096 Oct  1 19:33 ..
-rw-r--r-- 1 root root 249383281 Oct  1 19:33 best.pt
-rw-r--r-- 1 root root       578 Oct  1 19:33 config.json
-rw-r--r-- 1 root root      6401 Oct  1 19:33 eval_log.jsonl
-rw-r--r-- 1 root root      7632 Oct  1 19:33 train_log.csv

/kaggle/working/checkpoints/m2_mla_moe/:
total 576268
drwxr-xr-x 2 root root      4096 Oct  1 19:33 .
drwxr-xr-x 6 root root      4096 Oct  1 19:33 ..
-rw-r--r-- 1 root root 590016635 Oct  1 19:33 best.pt
-rw-r--r-- 1 root root       574 Oct  1 19:33 config.json
-rw-r--r-- 1 root root      6399 Oct  1 19:33 eval_log.jsonl
-rw-r--r-- 1 root root     41332 Oct  1 19:33 stdout.log
-rw-r--r-- 1 root root      7921 Oct  1 19:33 train_log.

In [1]:
!pip install -q -U "huggingface_hub>=0.32" hf_xet orjson pyahocorasick

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 13.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 68.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 

In [2]:
import os, json, random, time, numpy as np, torch
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

HF_REPO = "Exploration-Lab/CS-781-Capstone"
WORK    = "/kaggle/working"                                           # persisted: small outputs only
SCRATCH = "/kaggle/temp" if os.path.exists("/kaggle/temp") else "/tmp"  # NOT persisted: big raw files
for d in ["src", "stats", "data", "tokenizer", "checkpoints", "submissions"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import whoami, HfFileSystem
print("Logged in as:", whoami(token=HF_TOKEN)["name"])
with HfFileSystem(token=HF_TOKEN).open(f"datasets/{HF_REPO}/Phase_1_Pretraining/Test/test.json", "rb") as f:
    print("Gated access OK:", f.read(60))

Logged in as: Utkarsh-1718-1
Gated access OK: b'{\n  "1": {\n    "text": "The issue involved in these petition'


In [3]:
# ===== Check persisted files after a session restart =====
import os
for d in ["src", "stats", "tokenizer", "data"]:
    p = f"{WORK}/{d}"
    files = sorted(os.listdir(p)) if os.path.exists(p) else []
    print(f"{d:10s} {len(files):3d} files:", files[:12], "..." if len(files) > 12 else "")
print("\nFinished shard sources:",
      [f for f in os.listdir(f"{WORK}/data") if f.startswith("shard_stats_")])

src          7 files: ['__pycache__', 'bpe_tokenizer.py', 'common.py', 'profile_corpus.py', 'provenance.py', 'shard_corpus.py', 'valset.py'] 
stats       24 files: ['log_acts.txt', 'log_high_courts.txt', 'log_supreme_court.txt', 'log_tribunals.txt', 'prov_log_acts.txt', 'prov_log_high_courts.txt', 'prov_log_supreme_court.txt', 'prov_log_tribunals.txt', 'provenance_acts.jsonl', 'provenance_high_courts.jsonl', 'provenance_supreme_court.jsonl', 'provenance_tribunals.jsonl'] ...
tokenizer    2 files: ['a1_bpe_v2_32k.json', 'hf_a1_bpe_v2_32k.json'] 
data        19 files: ['.cache', 'Phase_1_Pretraining', 'shard_config.json', 'shard_log_acts.txt', 'shard_log_high_courts.txt', 'shard_log_supreme_court.txt', 'shard_log_tribunals.txt', 'shard_stats_acts.json', 'shard_stats_supreme_court.json', 'train_acts.bin', 'train_acts_docs.npy', 'train_high_courts.bin'] ...

Finished shard sources: ['shard_stats_supreme_court.json', 'shard_stats_acts.json']


## Section 1: Data understanding
### 1.1 Corpus profiler module (`src/profile_corpus.py`)
One pass per source file, which is a JSON array with **one object per line**, so we parse line by line with constant memory. Per source it computes:
- document count and length distribution; documents ≥640 words (enough for a 512-word prompt plus a 128-word reference)
- exact-duplicate rate (blake2b hash of whitespace-normalised text)
- non-ASCII and newline rates (the test text is pure ASCII with no newlines)
- year and court distributions, and the record keys
- a **reservoir sample of 5,000 documents** (reused for tokenizer training)
- a **leakage diagnostic**: three 20-word fingerprints per test prompt, matched with Aho–Corasick

In [5]:
%%writefile /kaggle/working/src/profile_corpus.py
"""Single pass over one Phase-1 source file (JSON array, one object per line).
Outputs in out_dir: summary_<src>.json, wordlens_<src>.npy, sample_<src>.jsonl, log_<src>.txt"""
from __future__ import annotations
import hashlib, json, random, time, zlib
from collections import Counter
import numpy as np
try:
    import orjson; _loads = orjson.loads
except ImportError:
    _loads = json.loads
try:
    import ahocorasick
except ImportError:
    ahocorasick = None

PCTS = [0, 1, 5, 25, 50, 75, 95, 99, 100]
MIN_VAL_WORDS = 640  # 512-word prompt + 128-word reference


def profile_source(source, local_path, out_dir, needles=(), reservoir_k=5000,
                   seed=1337, log_every=5_000, max_parse_errors=100):
    log = open(f"{out_dir}/log_{source}.txt", "w", buffering=1)
    rng = random.Random(seed + zlib.crc32(source.encode()))
    automaton = None
    if needles and ahocorasick is not None:
        automaton = ahocorasick.Automaton()
        for needle, payload in needles:
            automaton.add_word(needle, payload)
        automaton.make_automaton()

    word_lens, char_lens = [], []
    n = errors = bytes_read = 0
    nonascii_docs = nonascii_extra_bytes = newlines = 0
    seen, dup_docs, dup_words = set(), 0, 0
    headnote_docs = headnote_words = 0
    keys, years, groups = Counter(), Counter(), Counter()
    reservoir, matches = [], []
    t0 = t_last_log = time.time()

    with open(local_path, "rb") as f:
        for raw in f:
            bytes_read += len(raw)
            s = raw.strip()
            if s in (b"[", b"]", b""):
                continue
            if s.endswith(b","):
                s = s[:-1]
            try:
                rec = _loads(s)
            except Exception as e:
                errors += 1
                log.write(f"PARSE ERROR #{errors} at doc~{n}: {e!r} | {s[:120]!r}\n")
                if errors > max_parse_errors:
                    raise RuntimeError(f"{source}: too many parse errors") from e
                continue

            text = rec.get("text") or ""
            if n < 2000:
                keys.update(rec.keys())
            toks = text.split()
            nw = len(toks)
            word_lens.append(nw); char_lens.append(len(text))
            newlines += text.count("\n")
            if not text.isascii():
                nonascii_docs += 1
                nonascii_extra_bytes += len(text.encode("utf-8")) - len(text)

            norm = " ".join(toks)
            h = hashlib.blake2b(norm.encode("utf-8"), digest_size=8).digest()
            if h in seen:
                dup_docs += 1; dup_words += nw
            else:
                seen.add(h)

            hn = rec.get("headnote")
            if hn:
                headnote_docs += 1; headnote_words += len(hn.split())
            years[str(rec.get("year") or (rec.get("date") or "")[:4])] += 1
            groups[rec.get("court") or rec.get("tribunal") or rec.get("category") or "?"] += 1

            if automaton is not None and norm:
                for _end, (tid, pos) in automaton.iter(norm):
                    if len(matches) < 20_000:
                        matches.append((tid, pos, n))

            item = {"i": n, "text": text,
                    "meta": {k: rec.get(k) for k in ("title", "court", "tribunal", "category", "year", "date")
                             if rec.get(k) is not None}}
            if len(reservoir) < reservoir_k:
                reservoir.append(item)
            else:
                j = rng.randrange(n + 1)
                if j < reservoir_k:
                    reservoir[j] = item

            n += 1
            if n % log_every == 0 or time.time() - t_last_log > 30:
                t_last_log = time.time(); dt = t_last_log - t0
                log.write(f"{source}: {n:,} docs | {bytes_read/1e9:.2f} GB | "
                          f"{bytes_read/1e6/dt:.1f} MB/s | {dt/60:.1f} min\n")

    wl = np.asarray(word_lens, dtype=np.int64)
    np.save(f"{out_dir}/wordlens_{source}.npy", wl.astype(np.int32))
    with open(f"{out_dir}/sample_{source}.jsonl", "w", encoding="utf-8") as g:
        for item in reservoir:
            g.write(json.dumps(item, ensure_ascii=False) + "\n")
    total_words = int(wl.sum()); long_mask = wl >= MIN_VAL_WORDS
    summary = {
        "source": source, "bytes": bytes_read, "docs": n, "parse_errors": errors,
        "total_words": total_words, "total_chars": int(np.sum(char_lens)),
        "words_mean": float(wl.mean()) if n else 0.0,
        "words_percentiles": dict(zip(map(str, PCTS), np.percentile(wl, PCTS).astype(int).tolist() if n else [])),
        "docs_ge_640_words": int(long_mask.sum()),
        "words_in_docs_ge_640": int(wl[long_mask].sum()),
        "frac_docs_nonascii": nonascii_docs / max(1, n),
        "nonascii_extra_bytes_per_1k_chars": 1000 * nonascii_extra_bytes / max(1, int(np.sum(char_lens))),
        "newlines_per_1k_words": 1000 * newlines / max(1, total_words),
        "exact_dup_docs": dup_docs, "exact_dup_frac_docs": dup_docs / max(1, n),
        "exact_dup_frac_words": dup_words / max(1, total_words),
        "headnote_docs": headnote_docs, "headnote_words": headnote_words,
        "record_keys": dict(keys.most_common()), "top_groups": groups.most_common(25),
        "n_groups": len(groups), "years": dict(sorted(years.items())),
        "test_fingerprint_hits": len(matches),
        "test_ids_matched": sorted({m[0] for m in matches}, key=lambda x: int(x) if str(x).isdigit() else str(x)),
        "test_matches_sample": matches[:200], "leak_check_ran": automaton is not None,
        "elapsed_min": (time.time() - t0) / 60,
    }
    with open(f"{out_dir}/summary_{source}.json", "w") as g:
        json.dump(summary, g, indent=1)
    log.write(f"DONE {source}: {n:,} docs in {summary['elapsed_min']:.1f} min\n"); log.close()
    return summary


def build_needles(test_dict, width=20, positions=(0, 246, 492)):
    """Three 20-word fingerprints per test prompt (start / middle / end)."""
    out = []
    for tid, v in test_dict.items():
        w = v["text"].split()
        for p in positions:
            if p + width <= len(w):
                out.append((" ".join(w[p:p + width]), (tid, p)))
    return out

Writing /kaggle/working/src/profile_corpus.py


### 1.2 Official test prompts
1,000 prompts, each exactly 512 whitespace words. We check the format (newlines, spacing, ASCII, how prompts end) because our **validation set must replicate it exactly**.

In [6]:
from huggingface_hub import hf_hub_download
from collections import Counter
test_path = hf_hub_download(HF_REPO, "Phase_1_Pretraining/Test/test.json", repo_type="dataset",
                            token=HF_TOKEN, local_dir=f"{WORK}/data")      # 3 MB -> persisted
test = json.load(open(test_path, encoding="utf-8"))
tt = [v["text"] for v in test.values()]
wc = np.array([len(t.split()) for t in tt])
print(f"{len(test)} prompts | words min/median/max: {wc.min()}/{int(np.median(wc))}/{wc.max()}")
print("Prompts containing a newline:", sum("\n" in t for t in tt))
print("Prompts with double spaces:  ", sum("  " in t for t in tt))
print("Prompts with non-ASCII:      ", sum(not t.isascii() for t in tt))
print("Prompts starting uppercase:  ", sum(t[:1].isupper() for t in tt))
print("Last-char distribution:", Counter(t.rstrip()[-1] for t in tt).most_common(8))

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

1000 prompts | words min/median/max: 512/512/512
Prompts containing a newline: 0
Prompts with double spaces:   0
Prompts with non-ASCII:       0
Prompts starting uppercase:   767
Last-char distribution: [('e', 186), ('d', 96), ('s', 94), ('t', 89), ('n', 85), ('r', 60), ('y', 59), ('f', 50)]


### 1.3 Download raw training files to scratch disk
About 19.2 GB across four sources (HC 10.7, Tribunals 6.5, SC 1.3, Acts 0.7 GB). Saved to **non-persisted** scratch: `/kaggle/working` is too small, and persisting it would slow every session stop.

In [7]:
import shutil
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
RAW_DIR = f"{SCRATCH}/raw"; os.makedirs(RAW_DIR, exist_ok=True)
free_gb = shutil.disk_usage(RAW_DIR).free / 1e9
print(f"Scratch: {RAW_DIR} | free {free_gb:.1f} GB")
assert free_gb > 22, "Not enough scratch disk -- send Claude this number"

SOURCES = {   # smallest first, so the first line gives a quick speed reading
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
}
LOCAL = {}
for src, rel in SOURCES.items():
    t = time.time()
    LOCAL[src] = hf_hub_download(HF_REPO, rel, repo_type="dataset", token=HF_TOKEN, local_dir=RAW_DIR)
    mb = os.path.getsize(LOCAL[src]) / 1e6; dt = time.time() - t
    print(f"{src:14s} {mb/1e3:6.2f} GB in {dt/60:5.1f} min ({mb/dt:6.1f} MB/s)", flush=True)

Scratch: /tmp/raw | free 1201.7 GB


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

acts             0.73 GB in   0.0 min ( 256.1 MB/s)


Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

... (3 outputs left out)

tribunals        6.48 GB in   0.5 min ( 219.6 MB/s)


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

high_courts     10.67 GB in   1.0 min ( 175.6 MB/s)


### 1.4 Profile all four sources in parallel
Four worker processes started with `spawn`. `fork` deadlocked because of inherited network threads. Progress is printed every 30 s. Takes about 10 min.

**Key results:** 2.90B words in total; HC has 25.7% duplicate words; test prompts are ASCII-normalised; 426/1000 test prompts share a 20-word span with training data (checked more precisely in 1.5).

In [8]:
import sys, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import pandas as pd
sys.path.insert(0, f"{WORK}/src")
import importlib, profile_corpus as pc; importlib.reload(pc)

STATS_DIR = f"{WORK}/stats"
needles = pc.build_needles(test)
print(f"{len(needles)} fingerprints | leak check available: {pc.ahocorasick is not None}")

t0 = time.time()
with ProcessPoolExecutor(max_workers=4, mp_context=mp.get_context("spawn")) as ex:
    futs = {s: ex.submit(pc.profile_source, s, LOCAL[s], STATS_DIR, needles) for s in SOURCES}
    while not all(f.done() for f in futs.values()):
        time.sleep(30)
        msg = []
        for s in SOURCES:
            p = f"{STATS_DIR}/log_{s}.txt"
            last = open(p).read().strip().splitlines()[-1:] if os.path.exists(p) else []
            msg.append("  " + (last[0] if last else f"{s}: starting..."))
        print(f"[{(time.time()-t0)/60:5.1f} min]\n" + "\n".join(msg), flush=True)
    summaries = {s: f.result() for s, f in futs.items()}
print(f"\nAll done in {(time.time()-t0)/60:.1f} min")

rows = []
for s, x in summaries.items():
    p = x["words_percentiles"]
    rows.append({"source": s, "GB": round(x["bytes"]/1e9, 2), "docs": x["docs"],
                 "words(M)": round(x["total_words"]/1e6, 1),
                 "p50 words": p["50"], "p95 words": p["95"], "max words": p["100"],
                 "docs>=640w": x["docs_ge_640_words"],
                 "dup docs %": round(100*x["exact_dup_frac_docs"], 2),
                 "dup words %": round(100*x["exact_dup_frac_words"], 2),
                 "non-ASCII docs %": round(100*x["frac_docs_nonascii"], 1),
                 "\\n per 1k words": round(x["newlines_per_1k_words"], 1),
                 "headnote words(M)": round(x["headnote_words"]/1e6, 1),
                 "parse errs": x["parse_errors"], "test ids hit": len(x["test_ids_matched"])})
display(pd.DataFrame(rows).set_index("source").T)
hit = set().union(*[set(x["test_ids_matched"]) for x in summaries.values()])
print(f"\nTest prompts with a verbatim 20-word match in training data: {len(hit)} / {len(test)}")
for s, x in summaries.items():
    print(f"\n[{s}] keys: {list(x['record_keys'])}\n[{s}] top groups: {x['top_groups'][:8]}")

3000 fingerprints | leak check available: True
[  0.5 min]
  acts: 5,000 docs | 0.18 GB | 11.4 MB/s | 0.3 min
  supreme_court: 10,000 docs | 0.25 GB | 11.5 MB/s | 0.4 min
  tribunals: 25,000 docs | 0.32 GB | 11.6 MB/s | 0.5 min
  high_courts: 115,000 docs | 0.32 GB | 11.3 MB/s | 0.5 min
[  1.0 min]
  acts: 15,000 docs | 0.59 GB | 11.5 MB/s | 0.8 min
  supreme_court: 25,000 docs | 0.57 GB | 11.8 MB/s | 0.8 min
  tribunals: 65,000 docs | 0.66 GB | 11.5 MB/s | 0.9 min
  high_courts: 230,000 docs | 0.67 GB | 11.4 MB/s | 1.0 min
[  1.5 min]
  DONE acts: 18,826 docs in 1.1 min
  supreme_court: 40,000 docs | 0.95 GB | 11.7 MB/s | 1.4 min
  tribunals: 105,000 docs | 1.05 GB | 11.8 MB/s | 1.5 min
  high_courts: 380,000 docs | 1.10 GB | 12.3 MB/s | 1.5 min
[  2.0 min]
  DONE acts: 18,826 docs in 1.1 min
  DONE supreme_court: 55,549 docs in 1.9 min
  tribunals: 145,000 docs | 1.43 GB | 12.1 MB/s | 2.0 min
  high_courts: 535,000 docs | 1.55 GB | 13.1 MB/s | 2.0 min
[  2.5 min]
  DONE acts: 18,826 

source,acts,supreme_court,tribunals,high_courts
GB,0.73,1.32,6.48,10.67
docs,18826.00,55549.00,869132.00,2536738.00
words(M),119.60,210.20,975.90,1594.90
p50 words,2623.00,2377.00,582.00,266.00
p95 words,22230.00,10946.00,3552.00,2089.00
max words,503777.00,416710.00,199106.00,242011.00
docs>=640w,15752.00,47257.00,400304.00,462559.00
dup docs %,0.50,0.34,0.57,4.67
dup words %,0.25,0.31,0.42,25.68
non-ASCII docs %,16.50,21.30,24.50,6.70



Test prompts with a verbatim 20-word match in training data: 426 / 1000

[acts] keys: ['title', 'category', 'year', 'url', 'text', 'word_count', 'text_length_chars']
[acts] top groups: [('Union of India - Act', 3932), ('State of Rajasthan - Act', 1190), ('State of  Tamilnadu- Act', 1078), ('State of Punjab - Act', 986), ('State of Uttar Pradesh - Act', 944), ('State of Madhya Pradesh - Act', 927), ('State of Odisha - Act', 861), ('State of Bihar - Act', 766)]

[supreme_court] keys: ['title', 'date', 'court', 'citations', 'bench', 'headnote', 'text', 'word_count', 'text_length_chars']
[supreme_court] top groups: [('Supreme Court of India', 55549)]

[tribunals] keys: ['text', 'word_count', 'char_count', 'title', 'date', 'year', 'tribunal']
[tribunals] top groups: [('Central Information', 249502), ('Consumer Disputes Redressal', 225029), ('Central Administrative Tribunal', 149781), ('Custom Excercise Service tax Tribunal', 100625), ('National Green Tribunal', 58153), ('Customs Excise and

In [9]:
%%writefile /kaggle/working/src/provenance.py
"""Where do the official test prompts come from? Checks whole-prompt containment in training docs
under candidate normalisations. Does NOT record the text following a prompt (diagnostic only)."""
from __future__ import annotations
import json, time, unicodedata
try:
    import orjson; _loads = orjson.loads
except ImportError:
    _loads = json.loads
import ahocorasick

PUNCT_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",
    "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u2033": '"', "\u2032": "'",
    "\u2013": "-", "\u2014": "-", "\u2012": "-", "\u2212": "-", "\u00ad": "",
    "\u2026": "...", "\u00a0": " ", "\u2022": "*", "\u00b7": ".",
})

def _collapse(s): return " ".join(s.split())

NORMALISERS = {
    "ws_only":        lambda t: _collapse(t),
    "drop_nonascii":  lambda t: _collapse(t.encode("ascii", "ignore").decode()),
    "nfkd_ascii":     lambda t: _collapse(unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()),
    "punctmap_ascii": lambda t: _collapse(unicodedata.normalize("NFKD", t.translate(PUNCT_MAP))
                                          .encode("ascii", "ignore").decode()),
}

def scan_source(source, local_path, out_dir, test, needles):
    A = ahocorasick.Automaton()
    for needle, payload in needles:
        A.add_word(needle, payload)
    A.make_automaton()
    log = open(f"{out_dir}/prov_log_{source}.txt", "w", buffering=1)
    out = open(f"{out_dir}/provenance_{source}.jsonl", "w")
    n = n_hit_docs = 0
    t0 = t_last = time.time()
    with open(local_path, "rb") as f:
        for raw in f:
            s = raw.strip()
            if s in (b"[", b"]", b""):
                continue
            if s.endswith(b","):
                s = s[:-1]
            try:
                rec = _loads(s)
            except Exception:
                continue
            text = rec.get("text") or ""
            n += 1
            ws = _collapse(text)
            if not ws:
                continue
            hits = {}
            for _e, (tid, pos) in A.iter(ws):
                hits.setdefault(tid, set()).add(pos)
            if not text.isascii():
                for _e, (tid, pos) in A.iter(NORMALISERS["punctmap_ascii"](text)):
                    hits.setdefault(tid, set()).add(pos)
            if not hits:
                continue
            n_hit_docs += 1
            cache = {}
            for tid, positions in hits.items():
                prompt = test[tid]["text"]
                row = {"tid": tid, "source": source, "doc_i": n - 1,
                       "fingerprints_hit": sorted(positions), "full_match": None,
                       "doc_words": len(ws.split())}
                for name, fn in NORMALISERS.items():
                    if name not in cache:
                        cache[name] = fn(text)
                    idx = cache[name].find(prompt)
                    if idx >= 0:
                        before, after = cache[name][:idx], cache[name][idx + len(prompt):]
                        row.update(full_match=name, words_before=len(before.split()),
                                   words_after=len(after.split()),
                                   starts_at_word_boundary=(idx == 0 or before.endswith(" ")),
                                   doc_nonascii=not text.isascii())
                        break
                out.write(json.dumps(row) + "\n")
            if time.time() - t_last > 30:
                t_last = time.time()
                log.write(f"{source}: {n:,} docs | {n_hit_docs:,} hit docs | {(t_last-t0)/60:.1f} min\n")
    out.close()
    log.write(f"DONE {source}: {n:,} docs, {n_hit_docs:,} hit docs in {(time.time()-t0)/60:.1f} min\n")
    log.close()
    return source, n, n_hit_docs

Writing /kaggle/working/src/provenance.py


In [10]:
import sys, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
sys.path.insert(0, f"{WORK}/src")
import importlib, provenance as pv, profile_corpus as pc
importlib.reload(pv)

needles = pc.build_needles(test)
t0 = time.time()
with ProcessPoolExecutor(max_workers=4, mp_context=mp.get_context("spawn")) as ex:
    futs = {s: ex.submit(pv.scan_source, s, LOCAL[s], STATS_DIR, test, needles) for s in SOURCES}
    while not all(f.done() for f in futs.values()):
        time.sleep(30)
        msg = []
        for s in SOURCES:
            p = f"{STATS_DIR}/prov_log_{s}.txt"
            last = open(p).read().strip().splitlines()[-1:] if os.path.exists(p) else []
            msg.append("  " + (last[0] if last else f"{s}: starting..."))
        print(f"[{(time.time()-t0)/60:5.1f} min]\n" + "\n".join(msg), flush=True)
    print({s: f.result() for s, f in futs.items()})

[  0.5 min]
  acts: starting...
  supreme_court: starting...
  tribunals: starting...
  high_courts: starting...
[  1.0 min]
  acts: 4,299 docs | 23 hit docs | 0.6 min
  supreme_court: 3,021 docs | 121 hit docs | 0.5 min
  tribunals: 5,035 docs | 30 hit docs | 0.5 min
  high_courts: 90,941 docs | 72 hit docs | 0.5 min
[  1.5 min]
  acts: 7,038 docs | 54 hit docs | 1.1 min
  supreme_court: 11,894 docs | 287 hit docs | 1.0 min
  tribunals: 45,471 docs | 141 hit docs | 1.0 min
  high_courts: 184,263 docs | 131 hit docs | 1.0 min
[  2.0 min]
  acts: 10,602 docs | 83 hit docs | 1.6 min
  supreme_court: 21,603 docs | 491 hit docs | 1.5 min
  tribunals: 66,428 docs | 188 hit docs | 1.5 min
  high_courts: 310,063 docs | 235 hit docs | 1.6 min
[  2.5 min]
  acts: 14,931 docs | 99 hit docs | 2.2 min
  supreme_court: 27,565 docs | 643 hit docs | 2.0 min
  tribunals: 80,104 docs | 215 hit docs | 2.0 min
  high_courts: 412,121 docs | 295 hit docs | 2.1 min
[  3.0 min]
  DONE acts: 18,826 docs, 124 

In [12]:
from collections import Counter
rows = [json.loads(l) for s in SOURCES for l in open(f"{STATS_DIR}/provenance_{s}.jsonl")]
full = [r for r in rows if r["full_match"]]
by_tid = {}
for r in full:
    by_tid.setdefault(r["tid"], []).append(r)
partial_only = {r["tid"] for r in rows} - set(by_tid)

print(f"Test prompts FULLY contained in >=1 training doc : {len(by_tid)} / {len(test)}")
print(f"Test prompts with only partial (20-word) overlap : {len(partial_only)}")
print(f"Test prompts with no overlap at all              : {len(test) - len(by_tid) - len(partial_only)}")

print("\nSource(s) containing each fully-matched prompt:")
for k, v in Counter(tuple(sorted({r['source'] for r in rs})) for rs in by_tid.values()).most_common():
    print(f"  {v:4d}  {k}")
print("\nNormalisation that produced the match:", Counter(r["full_match"] for r in full).most_common())
print("Matches in docs that contain non-ASCII:", sum(r["doc_nonascii"] for r in full), "/", len(full))

best = [min(rs, key=lambda r: r["words_before"]) for rs in by_tid.values()]
wb = np.array([r["words_before"] for r in best]); wa = np.array([r["words_after"] for r in best])
dw = np.array([r["doc_words"] for r in best])
print("\nwords BEFORE prompt  p0/p25/p50/p75/p100:", np.percentile(wb, [0, 25, 50, 75, 100]).astype(int))
print("words AFTER prompt   p0/p25/p50/p75/p100:", np.percentile(wa, [0, 25, 50, 75, 100]).astype(int))
print("relative start position (before/doc)  p25/p50/p75:", np.round(np.percentile(wb/dw, [25, 50, 75]), 2))
print("prompts with >=128 words after them:", int((wa >= 128).sum()), "/", len(wa))
print("prompts starting at a word boundary:", sum(r["starts_at_word_boundary"] for r in best), "/", len(best))
print("doc length of source docs p25/p50/p75:", np.percentile(dw, [25, 50, 75]).astype(int))

Test prompts FULLY contained in >=1 training doc : 9 / 1000
Test prompts with only partial (20-word) overlap : 420
Test prompts with no overlap at all              : 571

Source(s) containing each fully-matched prompt:
     9  ('supreme_court',)

Normalisation that produced the match: [('ws_only', 9)]
Matches in docs that contain non-ASCII: 0 / 9

words BEFORE prompt  p0/p25/p50/p75/p100: [ 845 1855 5024 7480 9412]
words AFTER prompt   p0/p25/p50/p75/p100: [ 306 1695 2846 4778 7234]
relative start position (before/doc)  p25/p50/p75: [0.46 0.48 0.67]
prompts with >=128 words after them: 9 / 9
prompts starting at a word boundary: 9 / 9
doc length of source docs p25/p50/p75: [ 4062 12770 12770]


In [13]:
# ===== Cell 11: What kind of text are the test prompts? (domain + where windows start) =====
# Uses only the 5,000-doc reservoir samples saved by the profiler (no raw files needed).
import json, random, re, unicodedata
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SRC_NAMES = ["acts", "supreme_court", "tribunals", "high_courts"]
rng = random.Random(1337)

# Same normalisation we plan to apply to training text: ASCII-fold + collapse whitespace.
PUNCT_MAP = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
                           "\u2013": "-", "\u2014": "-", "\u2026": "...", "\u00a0": " ", "\u00ad": ""})
def normalise(t):
    t = unicodedata.normalize("NFKD", t.translate(PUNCT_MAP)).encode("ascii", "ignore").decode()
    return " ".join(t.split())

# ---- 1. Build test-like windows (512 words) from each source's reservoir sample -----------
X, y, docs_by_src = [], [], {}
for src in SRC_NAMES:
    words_list = []
    for line in open(f"{STATS_DIR}/sample_{src}.jsonl", encoding="utf-8"):
        w = normalise(json.loads(line)["text"]).split()
        if len(w) >= 640:                      # same eligibility as test: 512 prompt + 128 ref
            words_list.append(w)
    docs_by_src[src] = words_list
    for w in words_list[:2000]:                # cap per source -> balanced classifier
        s = rng.randrange(0, len(w) - 640 + 1)
        X.append(" ".join(w[s:s + 512])); y.append(src)
print("windows per source:", Counter(y))

# ---- 2. Source classifier: TF-IDF (word 1-2 grams) + logistic regression ----------------
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1337)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000, sublinear_tf=True)
clf = LogisticRegression(max_iter=2000, C=4.0)
clf.fit(vec.fit_transform(Xtr), ytr)
pred_va = clf.predict(vec.transform(Xva))
print("\nHeld-out source-classification quality (how trustworthy the test estimate is):")
print(classification_report(yva, pred_va, digits=3))
print("confusion (rows=true, cols=pred, order=%s):\n" % clf.classes_,
      confusion_matrix(yva, pred_va, labels=clf.classes_))

# ---- 3. Apply to the 1000 official test prompts ------------------------------------------
test_texts = [v["text"] for v in test.values()]
P = clf.predict_proba(vec.transform(test_texts))
hard = Counter(clf.classes_[P.argmax(1)])
print("\nEstimated TEST source mix (argmax):",
      {str(k): f"{v/len(test_texts):.1%}" for k, v in hard.most_common()})
print("Estimated TEST source mix (mean prob):",
      {str(k): round(float(v), 3) for k, v in zip(clf.classes_, P.mean(0))})

# ---- 4. Do test windows start at sentence boundaries? -----------------------------------
# Compare the test's uppercase-first-word rate with (a) random word positions and
# (b) positions right after a sentence end, measured on the training samples.
def is_upper(w): return w[:1].isupper()
rand_up = sent_up = n_rand = n_sent = 0
for src in SRC_NAMES:
    for w in docs_by_src[src][:500]:
        for _ in range(20):                                    # random positions
            i = rng.randrange(1, len(w) - 1); rand_up += is_upper(w[i]); n_rand += 1
        ends = [i + 1 for i in range(len(w) - 1) if w[i].endswith((".", "?", "!"))
                and not re.fullmatch(r"(\d+|[A-Za-z])\.", w[i])]   # skip "6." / "J." style tokens
        for i in rng.sample(ends, min(20, len(ends))):
            sent_up += is_upper(w[i]); n_sent += 1
p_rand, p_sent = rand_up / n_rand, sent_up / n_sent
p_test = np.mean([is_upper(t.split()[0]) for t in test_texts])
frac = (p_test - p_rand) / max(1e-9, p_sent - p_rand)       # linear mixing estimate
print(f"\nUppercase first word: random position {p_rand:.1%} | after sentence end {p_sent:.1%} | TEST {p_test:.1%}")
print(f"=> estimated share of test windows starting at a sentence boundary: {min(max(frac, 0), 1):.0%}")
print("\nFirst 8 words of 12 test prompts:")
for t in test_texts[:12]:
    print("  ", " ".join(t.split()[:8]))

windows per source: Counter({'acts': 2000, 'supreme_court': 2000, 'tribunals': 2000, 'high_courts': 931})

Held-out source-classification quality (how trustworthy the test estimate is):
               precision    recall  f1-score   support

         acts      0.928     0.995     0.960       400
  high_courts      0.965     0.733     0.833       187
supreme_court      0.871     0.910     0.890       400
    tribunals      0.955     0.950     0.952       400

     accuracy                          0.922      1387
    macro avg      0.930     0.897     0.909      1387
 weighted avg      0.924     0.922     0.921      1387

confusion (rows=true, cols=pred, order=['acts' 'high_courts' 'supreme_court' 'tribunals']):
 [[398   0   2   0]
 [  5 137  38   7]
 [ 20   5 364  11]
 [  6   0  14 380]]

Estimated TEST source mix (argmax): {'supreme_court': '92.1%', 'acts': '4.3%', 'tribunals': '2.3%', 'high_courts': '1.3%'}
Estimated TEST source mix (mean prob): {'acts': 0.074, 'high_courts': 0.088, 

## Section 2: Tokenizer (A1 BPE v2)
Byte-level BPE, evolved from my Assignment-1 (Q5) tokenizer.

**Kept from A1:** the merge learner. It counts unique pre-tokenised chunks, keeps a pair→chunks index and a lazy max-heap, so each merge only updates the chunks that contain that pair.

**Changed for Phase 1:**
- **Lossless pre-tokeniser.** A GPT-2-style regex where each word carries its leading space (`" court"`). This replaces `nltk.word_tokenize` + `</w>`, which dropped whitespace and rewrote quotes, so generated text could not be decoded exactly. Now `decode(encode(x)) == x`.
- **Digit grouping.** Numbers are split into chunks of at most 3 digits, so citations and section numbers ("AIR 1984 SC 1020") don't each consume a vocabulary entry.
- **16 reserved special tokens:** a document separator, pad, mask, sep, cls and spares, for sequence packing now and task heads in Phases 2–7.
- **Fast encoding.** The merges learned by my Python code are exported to Hugging Face's Rust `tokenizers` engine for bulk encoding. An assertion checks that both encoders produce identical ids.

**Training data:** normalised reservoir samples (ASCII, single spaces), weighted towards the Supreme Court because about 90% of test prompts are SC-style. Validation documents never touch the tokenizer.

**Shared rules** (`src/common.py`), used by every later stage:
- `normalise()` makes training text match the test format.
- `split_of()` is a deterministic document-level train/val split based on a text hash, so duplicates always land in the same split.

In [14]:
%%writefile /kaggle/working/src/common.py
"""
common.py -- rules shared by every Phase-1 stage (tokenizer, sharding, validation, generation).

normalise(text): make training text look like the official test prompts -- ASCII only, single
    spaces, no newlines (0/1000 test prompts contain a newline, double space or non-ASCII).
split_of(norm_text, source): deterministic DOCUMENT-level train/val assignment from a hash of
    the full normalised text -> exact duplicates always land in the SAME split.
"""
from __future__ import annotations
import hashlib, unicodedata

# Typographic characters -> ASCII equivalents (applied before stripping accents)
PUNCT_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'", "\u2032": "'",
    "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u2033": '"',
    "\u2013": "-", "\u2014": "-", "\u2012": "-", "\u2212": "-", "\u2010": "-", "\u2011": "-",
    "\u2026": "...", "\u00a0": " ", "\u00ad": "", "\u2022": "*", "\u00b7": ".", "\u200b": "",
    "\ufeff": "",
})

SOURCES = ["acts", "supreme_court", "tribunals", "high_courts"]

# Share of documents held out for validation. SC is the test domain -> largest share (~1,100 docs).
VAL_FRAC = {"supreme_court": 0.02, "acts": 0.02, "tribunals": 0.002, "high_courts": 0.002}


def normalise(text: str) -> str:
    """ASCII-fold (map punctuation, strip accents, drop the rest) and collapse all whitespace."""
    if not text:
        return ""
    if not text.isascii():                                   # fast path: most docs are ASCII
        text = unicodedata.normalize("NFKD", text.translate(PUNCT_MAP))
        text = text.encode("ascii", "ignore").decode("ascii")
    return " ".join(text.split())                            # newlines/tabs/double spaces -> 1 space


def doc_hash(norm_text: str) -> int:
    """Stable 64-bit hash (blake2b; Python's hash() is randomised per process)."""
    return int.from_bytes(hashlib.blake2b(norm_text.encode(), digest_size=8).digest(), "big")


def split_of(norm_text: str, source: str) -> str:
    """'val' or 'train' -- identical on every run and machine."""
    return "val" if (doc_hash(norm_text) % 1_000_000) < VAL_FRAC[source] * 1_000_000 else "train"

Writing /kaggle/working/src/common.py


In [15]:
%%writefile /kaggle/working/src/bpe_tokenizer.py
"""
bpe_tokenizer.py -- A1 BPE v2: byte-level BPE for the Phase-1 legal LM.
Core learner from Assignment-1 (Q5) + lossless regex pre-tokeniser + digit grouping +
reserved specials + export to HF `tokenizers` (Rust) for fast bulk encoding.
"""
from __future__ import annotations
import heapq, json, re, time
from collections import Counter, defaultdict

# 16 reserved ids: doc separator, padding, masking/sep/cls for later phases, spares
SPECIALS = ["<|endoftext|>", "<|pad|>", "<|mask|>", "<|sep|>", "<|cls|>", "<|bos|>"] + \
           [f"<|extra_{i}|>" for i in range(10)]


def make_pattern(digit_group: int = 3) -> str:
    """Pre-tokenisation regex (text is already ASCII-normalised). A chunk is one of:
    contraction | optional space + letters | optional space + <=k digits |
    optional space + punctuation run | whitespace. Every character lands in exactly one chunk."""
    d = f"[0-9]{{1,{digit_group}}}" if digit_group > 1 else "[0-9]"
    return (r"'(?:s|t|re|ve|m|ll|d)| ?[A-Za-z]+| ?" + d +
            r"| ?[^\sA-Za-z0-9]+|\s+(?!\S)|\s+")


def _bytes_to_unicode():
    """GPT-2's reversible byte -> printable-unicode map (HF ByteLevel BPE stores tokens this way)."""
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))


class ByteBPE:
    def __init__(self, digit_group: int = 3, specials=SPECIALS):
        self.digit_group = digit_group
        self.pattern = make_pattern(digit_group)
        self._pat = re.compile(self.pattern)
        self.specials = list(specials)
        self.n_sp = len(self.specials)
        self.id_to_bytes = {self.n_sp + b: bytes([b]) for b in range(256)}   # byte b -> id n_sp+b
        self.merges: list[tuple[int, int, int]] = []    # (left, right, new) in rank order
        self._rank, self._cache, self.train_info = {}, {}, {}

    # ---------------------------------------------------------------- training (A1 algorithm)
    def train(self, texts, vocab_size: int, min_freq: int = 2, verbose: bool = True, log_every: int = 2000):
        t0 = time.time()
        counts, n_chars = Counter(), 0
        for t in texts:                                   # 1) count unique chunks
            n_chars += len(t)
            counts.update(self._pat.findall(t))
        types = [(w, f) for w, f in counts.items() if f >= min_freq]   # hapaxes: little signal, lots of cost
        words = [[self.n_sp + b for b in w.encode("utf-8")] for w, _ in types]
        freqs = [f for _, f in types]
        if verbose:
            print(f"[bpe] {n_chars/1e6:.1f}M chars | {len(counts):,} unique chunks | "
                  f"{len(types):,} kept (freq>={min_freq}) | {time.time()-t0:.0f}s", flush=True)

        pair_freq, pair_words = defaultdict(int), defaultdict(set)   # 2) pair stats + index
        for wi, w in enumerate(words):
            for p in zip(w, w[1:]):
                pair_freq[p] += freqs[wi]
                pair_words[p].add(wi)
        heap = [(-f, p) for p, f in pair_freq.items()]
        heapq.heapify(heap)

        bytes_to_id = {v: k for k, v in self.id_to_bytes.items()}
        next_id = self.n_sp + 256
        while next_id < vocab_size and heap:              # 3) greedy merges
            negf, p = heapq.heappop(heap)
            if pair_freq.get(p, 0) != -negf:              # lazy deletion: stale heap entry
                continue
            if -negf < min_freq:
                break
            a, b = p
            nb = self.id_to_bytes[a] + self.id_to_bytes[b]
            nid = bytes_to_id.get(nb)
            if nid is None:                               # same byte string can arise from 2 merges
                nid, next_id = next_id, next_id + 1
                self.id_to_bytes[nid] = nb
                bytes_to_id[nb] = nid
            self.merges.append((a, b, nid))

            for wi in list(pair_words.pop(p, ())):        # only chunks containing the pair
                w = words[wi]
                nw, i, changed = [], 0, False
                while i < len(w):                         # merge left-to-right, non-overlapping
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        nw.append(nid); i += 2; changed = True
                    else:
                        nw.append(w[i]); i += 1
                if not changed:
                    continue
                f = freqs[wi]
                old_c, new_c = Counter(zip(w, w[1:])), Counter(zip(nw, nw[1:]))
                for q in old_c.keys() | new_c.keys():     # incremental pair-count update
                    d = new_c[q] - old_c[q]
                    if d:
                        pair_freq[q] += d * f
                        if d > 0:
                            pair_words[q].add(wi)
                        if pair_freq[q] > 0:
                            heapq.heappush(heap, (-pair_freq[q], q))
                        else:
                            pair_freq.pop(q, None)
                words[wi] = nw
            pair_freq.pop(p, None)
            if verbose and len(self.merges) % log_every == 0:
                print(f"[bpe] merges {len(self.merges):,} | vocab {next_id:,}/{vocab_size:,} | "
                      f"last freq {-negf:,} | {time.time()-t0:.0f}s", flush=True)

        self._build_rank()
        self.train_info = {"vocab_size": self.vocab_size, "merges": len(self.merges), "chars": n_chars,
                           "unique_chunks": len(counts), "kept_chunks": len(types), "min_freq": min_freq,
                           "digit_group": self.digit_group, "seconds": round(time.time() - t0, 1)}
        if verbose:
            print(f"[bpe] done: {self.train_info}", flush=True)
        return self

    def _build_rank(self):
        self._rank = {(a, b): (r, n) for r, (a, b, n) in enumerate(self.merges)}
        self._cache = {}

    @property
    def vocab_size(self):
        return self.n_sp + len(self.id_to_bytes)

    # ---------------------------------------------------------------- encode / decode (reference)
    def _encode_chunk(self, chunk: str):
        """Apply merges by rank: repeatedly merge the lowest-rank adjacent pair (GPT-2 style)."""
        ids = self._cache.get(chunk)
        if ids is not None:
            return ids
        w = [self.n_sp + b for b in chunk.encode("utf-8")]
        while len(w) > 1:
            best = None
            for i in range(len(w) - 1):
                r = self._rank.get((w[i], w[i + 1]))
                if r is not None and (best is None or r[0] < best[0]):
                    best = r
            if best is None:
                break
            r, nid = best
            a, b = self.merges[r][0], self.merges[r][1]
            nw, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                    nw.append(nid); i += 2
                else:
                    nw.append(w[i]); i += 1
            w = nw
        if len(self._cache) < 1_000_000:
            self._cache[chunk] = w
        return w

    def encode(self, text: str):
        out = []
        for chunk in self._pat.findall(text):
            out.extend(self._encode_chunk(chunk))
        return out

    def decode(self, ids):
        return b"".join(self.specials[i].encode() if i < self.n_sp else self.id_to_bytes[i]
                        for i in ids).decode("utf-8", errors="replace")

    # ---------------------------------------------------------------- persistence
    def save(self, path):
        json.dump({"format": "A1-BPE-v2", "digit_group": self.digit_group, "pattern": self.pattern,
                   "specials": self.specials,
                   "tokens": {str(i): b.hex() for i, b in self.id_to_bytes.items()},
                   "merges": self.merges, "train_info": self.train_info}, open(path, "w"))

    @classmethod
    def load(cls, path):
        d = json.load(open(path))
        obj = cls(digit_group=d["digit_group"], specials=d["specials"])
        obj.id_to_bytes = {int(i): bytes.fromhex(h) for i, h in d["tokens"].items()}
        obj.merges = [tuple(m) for m in d["merges"]]
        obj.train_info = d.get("train_info", {})
        obj._build_rank()
        return obj

    # ---------------------------------------------------------------- fast Rust encoder
    def to_hf(self):
        """Same vocab + merges + pre-tokeniser, applied by HF `tokenizers` (Rust)."""
        from tokenizers import Tokenizer, Regex, models, pre_tokenizers, decoders, AddedToken
        b2u = _bytes_to_unicode()
        tok_str = lambda bs: "".join(b2u[x] for x in bs)
        vocab = {s: i for i, s in enumerate(self.specials)}
        vocab.update({tok_str(bs): i for i, bs in self.id_to_bytes.items()})
        merges = [(tok_str(self.id_to_bytes[a]), tok_str(self.id_to_bytes[b])) for a, b, _ in self.merges]
        tk = Tokenizer(models.BPE(vocab=vocab, merges=merges, fuse_unk=False))
        tk.pre_tokenizer = pre_tokenizers.Sequence([
            pre_tokenizers.Split(Regex(self.pattern), behavior="isolated"),   # our regex chunks
            pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
        ])
        tk.decoder = decoders.ByteLevel()
        tk.add_special_tokens([AddedToken(s, special=True, normalized=False) for s in self.specials])
        return tk


def fertility(tok, texts, encode=None):
    """Tokens per whitespace word and bytes per token."""
    enc = encode or tok.encode
    n_tok = n_words = n_bytes = 0
    for t in texts:
        n_tok += len(enc(t)); n_words += len(t.split()); n_bytes += len(t.encode())
    return {"tokens_per_word": n_tok / max(1, n_words), "bytes_per_token": n_bytes / max(1, n_tok),
            "tokens": n_tok, "words": n_words}


def verify_hf_equivalence(bpe, hf_tok, texts, max_report=3):
    """Count docs where the Python and Rust encoders disagree (or Rust fails to round-trip)."""
    bad = 0
    for t in texts:
        a = bpe.encode(t)
        b = hf_tok.encode(t, add_special_tokens=False).ids
        if a != b or hf_tok.decode(b) != t:
            bad += 1
            if bad <= max_report:
                k = next((i for i in range(min(len(a), len(b))) if a[i] != b[i]), min(len(a), len(b)))
                print(f"MISMATCH at token {k}: ours {a[k:k+5]} vs rust {b[k:k+5]} | {t[:80]!r}")
    return bad

Writing /kaggle/working/src/bpe_tokenizer.py


In [17]:
# ===== Cell 14: Train the A1-BPE-v2 tokenizer, verify it, measure it, save it =====
import sys, json, time, importlib
import numpy as np
sys.path.insert(0, f"{WORK}/src")
import common, bpe_tokenizer as bt
importlib.reload(common); importlib.reload(bt)

STATS_DIR   = f"{WORK}/stats"
TOK_DIR     = f"{WORK}/tokenizer"
VOCAB_SIZE  = 32_000       # good compression; < 65,536 so ids fit in uint16 shards
DIGIT_GROUP = 3            # numbers -> <=3-digit chunks
# SC is the test domain (~90% of prompts) -> used in full; Acts docs are huge -> capped
CAP_WORDS = {"supreme_court": None, "acts": 8_000_000, "tribunals": None, "high_courts": None}

# ---- 1. Load samples -> normalise -> tokenizer-train vs held-out ----------------------------
train_texts, heldout = [], {s: [] for s in common.SOURCES}
for s in common.SOURCES:
    n_words = 0
    for k, line in enumerate(open(f"{STATS_DIR}/sample_{s}.jsonl", encoding="utf-8")):
        t = common.normalise(json.loads(line)["text"])
        if not t:
            continue
        if common.split_of(t, s) == "val" or k % 20 == 0:   # val docs + every 20th doc held out
            heldout[s].append(t)
            continue
        if CAP_WORDS[s] and n_words >= CAP_WORDS[s]:
            continue
        train_texts.append(t)
        n_words += len(t.split())
    print(f"{s:14s} tokenizer-train words: {n_words/1e6:6.1f}M | held-out docs: {len(heldout[s])}")

# ---- 2. Train with our A1-derived learner -------------------------------------------------
t0 = time.time()
bpe = bt.ByteBPE(digit_group=DIGIT_GROUP).train(train_texts, vocab_size=VOCAB_SIZE, min_freq=2)
bpe.save(f"{TOK_DIR}/a1_bpe_v2_{VOCAB_SIZE//1000}k.json")
print(f"trained in {(time.time()-t0)/60:.1f} min | vocab {bpe.vocab_size}")

# ---- 3. Export to Rust and PROVE both encoders agree -----------------------------------------
hf = bpe.to_hf()
hf.save(f"{TOK_DIR}/hf_a1_bpe_v2_{VOCAB_SIZE//1000}k.json")
check = [" ".join(t.split()[:3000]) for s in common.SOURCES for t in heldout[s][:75]]
bad = bt.verify_hf_equivalence(bpe, hf, check)
print(f"Python vs Rust encoder mismatches: {bad} / {len(check)} docs")
assert bad == 0, "encoders disagree -- do NOT tokenize the corpus until this is fixed"
rt_bad = sum(bpe.decode(bpe.encode(t)) != t for t in check[:100])
print(f"decode(encode(x)) != x : {rt_bad} / {min(100, len(check))}")
assert rt_bad == 0

# ---- 4. Fertility per source + token length of the official test prompts ---------------------
enc = lambda t: hf.encode(t, add_special_tokens=False).ids
print("\nFertility on held-out docs (lower = better compression):")
for s in common.SOURCES:
    f = bt.fertility(None, heldout[s], encode=enc)
    print(f"  {s:14s} {f['tokens_per_word']:.3f} tokens/word | {f['bytes_per_token']:.2f} bytes/token")
tl = np.array([len(enc(v["text"])) for v in test.values()])
print(f"\nTest prompts (512 words): tokens min/median/p99/max = "
      f"{tl.min()}/{int(np.median(tl))}/{int(np.percentile(tl, 99))}/{tl.max()}")
print(f"  -> prompt + 128-word continuation needs ~{int(tl.max() + 128 * tl.mean() / 512)} tokens of context")

# ---- 5. Rust encoder throughput (predicts how long Section 3 takes) ---------------------------
sample = [t for s in common.SOURCES for t in heldout[s]][:2000]
t0 = time.time(); n_tok = sum(len(e.ids) for e in hf.encode_batch(sample, add_special_tokens=False))
print(f"\nRust encode_batch: {n_tok/(time.time()-t0)/1e6:.2f}M tokens/s")

# ---- 6. Reference point: GPT-2's tokenizer (tokenizer files only, no model weights) -----------
# GPT-2 tokenizer reference via the `tokenizers` library (no transformers import needed)
from tokenizers import Tokenizer as _T
g2 = _T.from_pretrained("gpt2")
gf = bt.fertility(None, heldout["supreme_court"], encode=lambda t: g2.encode(t).ids)
of = bt.fertility(None, heldout["supreme_court"], encode=enc)
print(f"SC fertility: ours {of['tokens_per_word']:.3f} vs GPT-2 {gf['tokens_per_word']:.3f} "
      f"({100*(1-of['tokens_per_word']/gf['tokens_per_word']):+.1f}% fewer tokens)")

# ---- 7. How legal text gets segmented -------------------------------------------------------
demo = "The learned counsel for the appellant relied on AIR 1984 SC 1020 and Section 302 IPC."
print("\n", [bpe.decode([i]) for i in bpe.encode(demo)])

acts           tokenizer-train words:    8.0M | held-out docs: 342
supreme_court  tokenizer-train words:   17.7M | held-out docs: 353
tribunals      tokenizer-train words:    5.1M | held-out docs: 263
high_courts    tokenizer-train words:    2.9M | held-out docs: 260
[bpe] 201.6M chars | 287,832 unique chunks | 141,379 kept (freq>=2) | 18s
[bpe] merges 2,000 | vocab 2,272/32,000 | last freq 6,378 | 36s
[bpe] merges 4,000 | vocab 4,272/32,000 | last freq 2,276 | 39s
[bpe] merges 6,000 | vocab 6,272/32,000 | last freq 1,172 | 40s
[bpe] merges 8,000 | vocab 8,272/32,000 | last freq 714 | 41s
[bpe] merges 10,000 | vocab 10,272/32,000 | last freq 481 | 42s
[bpe] merges 12,000 | vocab 12,272/32,000 | last freq 346 | 42s
[bpe] merges 14,000 | vocab 14,272/32,000 | last freq 262 | 43s
[bpe] merges 16,000 | vocab 16,272/32,000 | last freq 205 | 43s
[bpe] merges 18,000 | vocab 18,272/32,000 | last freq 163 | 43s
[bpe] merges 20,000 | vocab 20,272/32,000 | last freq 134 | 44s
[bpe] merges 22,000 


Rust encode_batch: 0.78M tokens/s


tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

SC fertility: ours 1.244 vs GPT-2 1.282 (+2.9% fewer tokens)

 ['The', ' learned', ' counsel', ' for', ' the', ' appellant', ' relied', ' on', ' AIR', ' 198', '4', ' SC', ' 102', '0', ' and', ' Section', ' 302', ' IPC', '.']


## Section 3: Corpus construction (clean → dedup → decontaminate → split → sample → tokenise → shard)
One streaming pass per source (4 processes in parallel). For each document:
1. **Normalise** to the test format: ASCII-fold, collapse whitespace, no newlines (`common.normalise`).
2. **Length filter**: drop near-empty records and very short orders (HC and Tribunals: < 150 words; median HC doc is only 266 words).
3. **Exact dedup** on a hash of the normalised text. High Courts had 25.7% duplicate words.
4. **Decontaminate**: drop the 9 Supreme Court documents that contain an official test prompt (found in Section 1.5).
5. **Document-level split** (`common.split_of`, hash based): SC and Acts 2% validation, HC and Tribunals 0.2%. Duplicates always fall in the same split, so there is no train/val leakage.
6. **Sample** the two huge sources (HC and Tribunals: keep 40% by hash) to fit our compute budget. SC and Acts are kept in full; SC is ~90% of the test domain.
7. **Tokenise** with A1-BPE-v2 (Rust engine), append `<|endoftext|>`, and write **uint16** shards per source plus document-offset indexes. SC headnotes are added as extra documents.

Mixing between sources is decided **at training time** by sampling weights (natural / SC-heavy / two-stage curriculum), so changing the mix never requires re-tokenising.

Validation (Section 3.2) comes only from held-out documents:
- a **generation set** in the exact test format (512-word prompt starting at a sentence boundary, 128-word reference);
- **validation-loss shards** for bits-per-word curves.

In [18]:
%%writefile /kaggle/working/src/shard_corpus.py
"""
shard_corpus.py -- one pass per source: clean -> dedup -> decontaminate -> split -> sample ->
tokenize -> write uint16 token shards.
Outputs per source s: train_<s>.bin (uint16, docs end with <|endoftext|>=0), train_<s>_docs.npy
(doc start offsets), val_<s>.jsonl (normalised held-out docs), shard_stats_<s>.json.
"""
from __future__ import annotations
import json, os, time
import numpy as np
try:
    import orjson; _loads = orjson.loads
except ImportError:
    _loads = json.loads
import common

EOT_ID = 0          # <|endoftext|> is special id 0 in A1-BPE-v2


def _load_encoder(tok_hf_path, tok_py_path):
    """Rust tokenizer if available, else pure-Python A1-BPE-v2 (slow fallback)."""
    try:
        from tokenizers import Tokenizer
        tk = Tokenizer.from_file(tok_hf_path)
        return lambda texts: [e.ids for e in tk.encode_batch(texts, add_special_tokens=False)]
    except Exception:
        from bpe_tokenizer import ByteBPE
        b = ByteBPE.load(tok_py_path)
        return lambda texts: [b.encode(t) for t in texts]


def process_source(source, local_path, out_dir, tok_hf_path, tok_py_path, cfg,
                   contaminated=frozenset(), batch_docs=256):
    t0 = t_last = time.time()
    encode = _load_encoder(tok_hf_path, tok_py_path)
    log = open(f"{out_dir}/shard_log_{source}.txt", "w", buffering=1)
    bin_f = open(f"{out_dir}/train_{source}.bin", "wb")
    val_f = open(f"{out_dir}/val_{source}.jsonl", "w")
    min_words, keep_frac = cfg["min_words"][source], cfg["keep_frac"][source]
    use_headnote = cfg.get("include_headnote", True)

    st = dict(docs_read=0, empty=0, too_short=0, duplicate=0, contaminated=0, val_docs=0,
              sampled_out=0, train_docs=0, headnote_docs=0, train_words=0, train_tokens=0)
    seen, doc_starts, offset, pending = set(), [], 0, []

    def flush():
        """Tokenize buffered docs in one Rust batch call and append them to the .bin."""
        nonlocal offset
        if not pending:
            return
        for ids in encode(pending):
            ids.append(EOT_ID)                                   # document separator
            arr = np.asarray(ids, dtype=np.uint16)
            bin_f.write(arr.tobytes())
            doc_starts.append(offset); offset += len(arr)
            st["train_tokens"] += len(arr)
        pending.clear()

    with open(local_path, "rb") as f:
        doc_i = -1
        for raw in f:
            s = raw.strip()
            if s in (b"[", b"]", b""):
                continue
            if s.endswith(b","):
                s = s[:-1]
            try:
                rec = _loads(s)
            except Exception:
                continue
            doc_i += 1                                           # same numbering as provenance.py
            st["docs_read"] += 1

            text = common.normalise(rec.get("text") or "")      # 1) test-format normalisation
            if not text:
                st["empty"] += 1; continue
            nw = text.count(" ") + 1                             # single-spaced -> word count
            if nw < min_words:                                   # 2) length filter
                st["too_short"] += 1; continue
            h = common.doc_hash(text)
            if h in seen:                                        # 3) exact dedup
                st["duplicate"] += 1; continue
            seen.add(h)
            if doc_i in contaminated:                            # 4) test-prompt source doc
                st["contaminated"] += 1; continue
            if common.split_of(text, source) == "val":           # 5) doc-level split
                st["val_docs"] += 1
                val_f.write(json.dumps({"doc_i": doc_i, "text": text}) + "\n")
                continue
            if keep_frac < 1.0 and ((h >> 20) % 10_000) >= keep_frac * 10_000:   # 6) sampling
                st["sampled_out"] += 1; continue

            st["train_docs"] += 1; st["train_words"] += nw
            pending.append(text)
            hn = rec.get("headnote") if use_headnote else None   # SC headnotes as extra docs
            if hn:
                hn = common.normalise(hn)
                if hn and hn.count(" ") + 1 >= 20:
                    st["headnote_docs"] += 1; st["train_words"] += hn.count(" ") + 1
                    pending.append(hn)
            if len(pending) >= batch_docs:                       # 7) tokenize + write
                flush()
            if time.time() - t_last > 30:
                t_last = time.time()
                log.write(f"{source}: read {st['docs_read']:,} | train docs {st['train_docs']:,} | "
                          f"tokens {st['train_tokens']/1e6:.1f}M | {(t_last-t0)/60:.1f} min\n")
    flush()
    bin_f.close(); val_f.close()
    np.save(f"{out_dir}/train_{source}_docs.npy", np.asarray(doc_starts, dtype=np.int64))
    st["elapsed_min"] = round((time.time() - t0) / 60, 2)
    st["tokens_per_word"] = st["train_tokens"] / max(1, st["train_words"])
    st["bin_bytes"] = os.path.getsize(f"{out_dir}/train_{source}.bin")
    json.dump(st, open(f"{out_dir}/shard_stats_{source}.json", "w"), indent=1)
    log.write(f"DONE {source}: {st}\n"); log.close()
    return source, st

Writing /kaggle/working/src/shard_corpus.py


In [20]:
%%writefile /kaggle/working/src/valset.py
"""
valset.py -- validation data from held-out documents.
1) Generation set in the EXACT test format: 512-word prompt starting at a sentence/numbered-
   paragraph boundary + the next 128 words as reference -> val_prompts.json / val_refs.json.
2) Validation-loss shards val_<source>.bin for bits-per-word curves.
"""
from __future__ import annotations
import json, random, re
import numpy as np

PROMPT_WORDS, REF_WORDS = 512, 128

# Tokens ending in "." that do NOT end a sentence in Indian legal text
ABBREV = {"vs.", "v.", "no.", "nos.", "ors.", "anr.", "co.", "ltd.", "pvt.", "dr.", "mr.", "mrs.",
          "ms.", "sri.", "smt.", "shri.", "s.", "ss.", "art.", "arts.", "sec.", "secs.", "rs.",
          "viz.", "i.e.", "e.g.", "etc.", "cl.", "para.", "paras.", "p.", "pp.", "j.", "jj.",
          "st.", "r.", "o.", "govt.", "dept.", "addl.", "asstt.", "hon'ble", "u/s.", "w.e.f.",
          "cr.", "crl.", "civ.", "misc.", "spl.", "a.i.r.", "s.c.", "s.c.c.", "vol.", "ed.", "approx."}
_single_letter_dot = re.compile(r"^[A-Za-z]\.$")          # initials like "K."


def sentence_starts(words):
    """Word indices that plausibly start a sentence or numbered paragraph."""
    out = [0]
    for i in range(1, len(words)):
        prev, w = words[i - 1], words[i]
        if not prev.endswith((".", "?", "!", ":")):
            continue
        if prev.lower() in ABBREV or _single_letter_dot.match(prev):
            continue
        if w[0].isupper() or w[0].isdigit() or w[0] in "(\"'[":
            out.append(i)
    return out


def build_generation_set(val_jsonl_by_source, n_per_source, seed=1337):
    """One random sentence-boundary window per held-out doc, until n windows per source."""
    rng = random.Random(seed)
    prompts, refs, meta = {}, {}, {}
    for src, path in val_jsonl_by_source.items():
        docs = [json.loads(l) for l in open(path)]
        rng.shuffle(docs)
        k = 0
        for d in docs:
            if k >= n_per_source.get(src, 0):
                break
            w = d["text"].split()
            if len(w) < PROMPT_WORDS + REF_WORDS:
                continue
            cands = [i for i in sentence_starts(w) if i + PROMPT_WORDS + REF_WORDS <= len(w)]
            if not cands:
                continue
            i = rng.choice(cands)
            vid = f"{src[:2]}_{k:04d}"
            prompts[vid] = {"text": " ".join(w[i:i + PROMPT_WORDS])}
            refs[vid] = {"text": " ".join(w[i + PROMPT_WORDS:i + PROMPT_WORDS + REF_WORDS])}
            meta[vid] = {"source": src, "doc_i": d["doc_i"], "start_word": i, "doc_words": len(w)}
            k += 1
    return prompts, refs, meta


def build_val_loss_shards(val_jsonl_by_source, out_dir, encode_batch, max_tokens_by_source, eot_id=0):
    """Tokenize held-out docs into val_<src>.bin; returns token/word counts (for bits per word)."""
    stats = {}
    for src, path in val_jsonl_by_source.items():
        texts = [json.loads(l)["text"] for l in open(path)]
        cap = max_tokens_by_source.get(src, 1_000_000)
        n_tok = n_words = 0
        with open(f"{out_dir}/val_{src}.bin", "wb") as f:
            for i in range(0, len(texts), 256):
                batch = texts[i:i + 256]
                for t, ids in zip(batch, encode_batch(batch)):
                    ids = list(ids) + [eot_id]
                    f.write(np.asarray(ids, dtype=np.uint16).tobytes())
                    n_tok += len(ids); n_words += t.count(" ") + 1
                if n_tok >= cap:
                    break
        stats[src] = {"tokens": n_tok, "words": n_words, "tokens_per_word": n_tok / max(1, n_words)}
    return stats

Overwriting /kaggle/working/src/valset.py


In [4]:
# ===== Cell 17: Build per-source token shards (4 parallel workers) =====
import os, sys, json, time, shutil, importlib, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import pandas as pd
sys.path.insert(0, f"{WORK}/src")
import common, shard_corpus as sc
importlib.reload(common); importlib.reload(sc)

DATA_DIR  = f"{WORK}/data"
STATS_DIR = f"{WORK}/stats"
TOK_HF    = f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json"
TOK_PY    = f"{WORK}/tokenizer/a1_bpe_v2_32k.json"
FORCE     = False                        # True = rebuild even if shards already exist

CFG = {
    "min_words": {"acts": 20, "supreme_court": 50, "tribunals": 150, "high_courts": 150},
    "keep_frac": {"acts": 1.0, "supreme_court": 1.0, "tribunals": 0.4, "high_courts": 0.4},
    "include_headnote": True,
}
json.dump({"cfg": CFG, "val_frac": common.VAL_FRAC, "tokenizer": os.path.basename(TOK_HF)},
          open(f"{DATA_DIR}/shard_config.json", "w"), indent=1)           # reproducibility record

done = all(os.path.exists(f"{DATA_DIR}/shard_stats_{s}.json") for s in common.SOURCES)
if done and not FORCE:
    print("Shards already built -> skipping (set FORCE=True to rebuild)")
else:
    # ---- only redo sources whose shards are not complete ------------------------------------
    todo = [s for s in common.SOURCES
            if FORCE or not os.path.exists(f"{DATA_DIR}/shard_stats_{s}.json")]
    print("sources to (re)build:", todo)

    # ---- raw files: reuse if still on scratch disk, otherwise download (only what's needed) --
    from huggingface_hub import hf_hub_download
    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
    RAW_DIR = f"{SCRATCH}/raw"
    LOCAL = {}
    for s in todo:
        rel = f"Phase_1_Pretraining/Train/train_{s}.json"
        p = f"{RAW_DIR}/{rel}"
        t = time.time()
        LOCAL[s] = p if os.path.exists(p) else hf_hub_download(
            HF_REPO, rel, repo_type="dataset", token=HF_TOKEN, local_dir=RAW_DIR)
        print(f"  {s}: ready in {(time.time()-t)/60:.1f} min", flush=True)

    # ---- documents that contain an official test prompt (Section 1.5) ------------------------
    contam = {s: set() for s in common.SOURCES}
    for s in common.SOURCES:
        for l in open(f"{STATS_DIR}/provenance_{s}.jsonl"):
            r = json.loads(l)
            if r["full_match"]:
                contam[s].add(r["doc_i"])

    # ---- workers x 1 Rust thread each (no CPU oversubscription on 4 cores) --------------------
    os.environ["RAYON_NUM_THREADS"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "true"
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=len(todo), mp_context=mp.get_context("spawn")) as ex:
        futs = {s: ex.submit(sc.process_source, s, LOCAL[s], DATA_DIR, TOK_HF, TOK_PY, CFG,
                             frozenset(contam[s])) for s in todo}
        while not all(f.done() for f in futs.values()):
            time.sleep(60)
            msg = []
            for s in todo:
                p = f"{DATA_DIR}/shard_log_{s}.txt"
                last = open(p).read().strip().splitlines()[-1:] if os.path.exists(p) else []
                msg.append("  " + (last[0][:150] if last else f"{s}: starting..."))
            print(f"[{(time.time()-t0)/60:5.1f} min]\n" + "\n".join(msg), flush=True)
        for f in futs.values():
            f.result()                                   # re-raise any worker error
    print(f"Sharding finished in {(time.time()-t0)/60:.1f} min")

# ---- summary table (from saved stats, works on re-runs too) ---------------------------------
st = {s: json.load(open(f"{DATA_DIR}/shard_stats_{s}.json")) for s in common.SOURCES}
df = pd.DataFrame(st).T
df["train_tokens(M)"] = (df["train_tokens"] / 1e6).round(1)
df["bin GB"] = (df["bin_bytes"] / 1e9).round(2)
display(df[["docs_read", "empty", "too_short", "duplicate", "contaminated", "val_docs", "sampled_out",
            "train_docs", "headnote_docs", "train_tokens(M)", "tokens_per_word", "bin GB", "elapsed_min"]])
print(f"TOTAL train tokens: {df['train_tokens'].sum()/1e9:.2f} B")

sources to (re)build: ['tribunals', 'high_courts']


Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

  tribunals: ready in 0.7 min


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

  high_courts: ready in 0.9 min
[  1.3 min]
  tribunals: read 2,176 | train docs 847 | tokens 5.6M | 0.5 min
  high_courts: read 40,438 | train docs 13,056 | tokens 8.5M | 0.5 min
[  2.3 min]
  tribunals: read 16,493 | train docs 6,400 | tokens 20.3M | 1.5 min
  high_courts: read 124,252 | train docs 40,448 | tokens 25.4M | 1.5 min
[  3.3 min]
  tribunals: read 37,827 | train docs 14,848 | tokens 37.4M | 2.6 min
  high_courts: read 208,088 | train docs 66,816 | tokens 42.6M | 2.5 min
[  4.3 min]
  tribunals: read 60,039 | train docs 23,296 | tokens 52.3M | 3.6 min
  high_courts: read 296,748 | train docs 86,528 | tokens 59.4M | 3.5 min
[  5.3 min]
  tribunals: read 77,678 | train docs 29,952 | tokens 66.4M | 4.6 min
  high_courts: read 398,761 | train docs 108,831 | tokens 75.5M | 4.5 min
[  6.3 min]
  tribunals: read 91,748 | train docs 35,328 | tokens 80.4M | 5.7 min
  high_courts: read 474,957 | train docs 128,256 | tokens 87.0M | 5.5 min
[  7.3 min]
  tribunals: read 105,731 | trai

,docs_read,empty,too_short,duplicate,contaminated,val_docs,sampled_out,train_docs,headnote_docs,train_tokens(M),tokens_per_word,bin GB,elapsed_min
acts,18826.0,4.0,4.0,91.0,0.0,392.0,0.0,18335.0,0.0,147.6,1.264021,0.30,14.91
supreme_court,55549.0,21.0,87.0,156.0,3.0,1114.0,0.0,54168.0,10614.0,262.6,1.227415,0.53,21.76
tribunals,869132.0,0.0,49015.0,4355.0,0.0,1619.0,489151.0,324992.0,0.0,547.8,1.451300,1.10,38.32
high_courts,2536738.0,0.0,677770.0,107992.0,0.0,3369.0,1049411.0,698196.0,0.0,608.8,1.359852,1.22,40.87


TOTAL train tokens: 1.57 B


### 3.2 Validation sets (held-out documents only)
- **Generation set** in the exact test format: a 512-word prompt starting at a sentence or numbered-paragraph boundary, plus the next 128 words as the reference. There are 1,000 Supreme Court windows (the test domain) and 150 from each other source. Saved as `val_prompts.json` (same schema as `test.json`) and `val_refs.json`. It is used to score BERTScore / ROUGE-L / BLEU-4 locally before each Codabench submission.
- **Validation-loss shards** (`val_<source>.bin`) give per-source loss and **bits per word** during training. Bits per word stays comparable across tokenizers and architectures; per-token perplexity does not.
- A check compares the validation prompts with the official test prompts: token length and the rate of capitalised first words.

In [5]:
# ===== Cell 18: Validation sets from held-out documents (CPU, ~2 min) =====
import sys, json, importlib
import numpy as np
from collections import Counter
sys.path.insert(0, f"{WORK}/src")
import common, valset
importlib.reload(valset)
from tokenizers import Tokenizer

DATA_DIR = f"{WORK}/data"
tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
enc_batch = lambda texts: [e.ids for e in tk.encode_batch(texts, add_special_tokens=False)]
vj = {s: f"{DATA_DIR}/val_{s}.jsonl" for s in common.SOURCES}

# ---- 1. Generation set in the exact test format (SC = test domain -> most windows) ---------
N = {"supreme_court": 1000, "acts": 150, "tribunals": 150, "high_courts": 150}
P, R, M = valset.build_generation_set(vj, N, seed=1337)
json.dump(P, open(f"{DATA_DIR}/val_prompts.json", "w"))      # same schema as test.json
json.dump(R, open(f"{DATA_DIR}/val_refs.json", "w"))         # references (128 words each)
json.dump(M, open(f"{DATA_DIR}/val_meta.json", "w"))
assert all(len(v["text"].split()) == 512 for v in P.values())
assert all(len(v["text"].split()) == 128 for v in R.values())
print("generation windows per source:", Counter(m["source"] for m in M.values()))

# ---- 2. Does our validation set look like the official test set? ------------------------
test = json.load(open(f"{WORK}/data/Phase_1_Pretraining/Test/test.json"))
def profile(prompts):
    t = [v["text"] for v in prompts]
    n_tok = np.array([len(x) for x in enc_batch(t)])
    return {"median_tokens": int(np.median(n_tok)), "p99_tokens": int(np.percentile(n_tok, 99)),
            "upper_first_word": round(float(np.mean([x[:1].isupper() for x in t])), 3)}
sc_val = [P[k] for k in P if M[k]["source"] == "supreme_court"]
print("TEST   :", profile(list(test.values())))
print("VAL-SC :", profile(sc_val))

# ---- 3. Validation-loss shards (for bits-per-word curves during training) ----------------
st = valset.build_val_loss_shards(vj, DATA_DIR, enc_batch,
                                  {"supreme_court": 3_000_000, "acts": 1_000_000,
                                   "tribunals": 1_000_000, "high_courts": 1_000_000})
json.dump(st, open(f"{DATA_DIR}/val_loss_stats.json", "w"), indent=1)
for s, v in st.items():
    print(f"  val_{s}.bin: {v['tokens']/1e6:.2f}M tokens | {v['tokens_per_word']:.3f} tokens/word")

k = next(iter(P))
print("\nExample prompt end  :", "..." + P[k]["text"][-150:])
print("Example reference   :", R[k]["text"][:150] + "...")

generation windows per source: Counter({'supreme_court': 944, 'acts': 150, 'tribunals': 150, 'high_courts': 150})
TEST   : {'median_tokens': 610, 'p99_tokens': 758, 'upper_first_word': 0.767}
VAL-SC : {'median_tokens': 617, 'p99_tokens': 762, 'upper_first_word': 0.824}
  val_acts.bin: 2.18M tokens | 1.288 tokens/word
  val_supreme_court.bin: 3.85M tokens | 1.232 tokens/word
  val_tribunals.bin: 1.46M tokens | 1.460 tokens/word
  val_high_courts.bin: 1.25M tokens | 1.331 tokens/word

Example prompt end  : ...onal registration for a period not exceeding one year if he satisfies the Council that he holds such a degree or diploma. (4) Notwithstanding anything
Example reference   : contained in sub-section (3) the name of every person which on the day immediately preceding the date of commencement of this Act stands entered in th...


## Section 4: Model code + training engine (M1: dense decoder baseline)
**M1 = modern dense decoder-only Transformer, about 100M parameters.** This is the controlled baseline every other architecture is compared against at the same token budget.
- Pre-norm blocks, **RMSNorm** (computed in fp32), **SwiGLU** MLP, **RoPE** positions, **tied** input/output embeddings.
- **GQA** (12 query heads, 4 KV heads): 3× smaller KV cache for generation.
- **QK-norm**: RMSNorm on queries and keys, which bounds attention logits. That matters because the T4 has **no bf16**, so we train in fp16.
- Attention via `scaled_dot_product_attention`. On the T4 (sm_75) this uses the **memory-efficient kernel**, since FlashAttention-2 needs sm_80+. Memory is O(T); the T×T matrix is never materialised.
- **KV cache** for generation: the prompt is prefilled once, then each decoding step reuses the cached K/V.

**Training engine (`train.py`)**, shared by all architectures:
- fp16 autocast with GradScaler, **DDP on 2×T4** via `torchrun`, gradient accumulation, `torch.compile`.
- **WSD learning-rate schedule** (warmup → stable → decay) combined with a **data curriculum**: a broad legal mix, switching to an SC-heavy anneal exactly when the LR starts decaying.
- Batches are a pure function of (seed, step, rank), so **resume is exact**. Atomic checkpoints (`latest.pt` full state, `best.pt` by SC validation loss). Stops cleanly before Kaggle's 12-hour limit.
- Every eval logs per-source validation loss and bits per word.

In [6]:
%%writefile /kaggle/working/src/lm_common.py
"""
lm_common.py -- building blocks shared by all Phase-1 architectures + KV-cached generation.
RMSNorm (fp32 stats) | RoPE | QK-norm (fp16 stability) | SwiGLU | SDPA attention (memory-
efficient kernel on T4) | KV cache: prompt processed once, then one token per step.
"""
from __future__ import annotations
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                              # fp32 statistics
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return xf.to(x.dtype) * self.weight.to(x.dtype)


def rope_cache(seq_len: int, head_dim: int, theta: float = 10000.0, device=None):
    """cos/sin tables of shape (seq_len, head_dim/2), fp32."""
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    ang = torch.outer(torch.arange(seq_len, device=device).float(), inv)
    return ang.cos(), ang.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotate-half formulation, computed in fp32."""
    h = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :h], xf[..., h:]
    c, s = cos[None, None], sin[None, None]
    return torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)


@dataclass
class KVCache:
    k: torch.Tensor          # (B, n_kv_heads, max_T, head_dim)
    v: torch.Tensor


class Attention(nn.Module):
    """Multi-head attention with grouped-query K/V (n_kv_heads <= n_heads), QK-norm, RoPE."""

    def __init__(self, d_model, n_heads, n_kv_heads, qk_norm=True):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.h, self.kv, self.hd = n_heads, n_kv_heads, d_model // n_heads
        self.wq = nn.Linear(d_model, n_heads * self.hd, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.hd, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.hd, bias=False)
        self.wo = nn.Linear(n_heads * self.hd, d_model, bias=False)
        self.qk_norm = qk_norm
        if qk_norm:
            self.q_norm, self.k_norm = RMSNorm(self.hd), RMSNorm(self.hd)

    def forward(self, x, cos, sin, cache: KVCache | None = None, pos: int = 0):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.wk(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.wv(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        if self.qk_norm:
            q, k = self.q_norm(q), self.k_norm(k)
        q = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T])
        k = apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
        if cache is not None:                                        # write new K/V, read all
            cache.k[:, :, pos:pos + T] = k
            cache.v[:, :, pos:pos + T] = v
            k, v = cache.k[:, :, :pos + T], cache.v[:, :, :pos + T]
        if self.kv != self.h:                                        # GQA: share K/V across groups
            rep = self.h // self.kv
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        # causal mask needed only when several queries are processed at once (training/prefill
        # at pos=0); decoding uses T=1, which may attend to everything cached.
        assert T == 1 or pos == 0, "chunked prefill (T>1 at pos>0) is not supported"
        y = F.scaled_dot_product_attention(q, k, v, is_causal=(T > 1))
        return self.wo(y.transpose(1, 2).reshape(B, T, self.h * self.hd))


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden, bias=False)   # gate
        self.w3 = nn.Linear(d_model, hidden, bias=False)   # up
        self.w2 = nn.Linear(hidden, d_model, bias=False)   # down

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


def count_params(model: nn.Module) -> int:
    """Unique parameters (tied embedding counted once)."""
    return sum(p.numel() for p in {id(p): p for p in model.parameters()}.values())


# --------------------------------------------------------------------------- generation
def _apply_sampling(logits, temperature, top_k, top_p):
    if temperature <= 0:
        return logits.argmax(-1, keepdim=True)                       # greedy
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1).values[..., -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p and top_p < 1.0:                                         # nucleus sampling
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = sl.softmax(-1).cumsum(-1)
        remove = cp - sl.softmax(-1) > top_p                          # always keep the top token
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(-1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1)


@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.0, top_k=0, top_p=1.0,
             repetition_penalty=1.0, banned_ids=(0,), no_repeat_ngram=0):
    """KV-cached generation for a batch of EQUAL-length prompts. idx: (B, T) -> (B, max_new_tokens).
    banned_ids: <|endoftext|> by default so the model cannot stop before 128 words."""
    model.eval()
    B, T = idx.shape
    max_T = T + max_new_tokens
    # cache dtype must match the K/V activations: fp16 under autocast, else the weights' dtype
    cdt = torch.float16 if torch.is_autocast_enabled() else next(model.parameters()).dtype
    caches = model.make_caches(B, max_T, idx.device, dtype=cdt)
    logits = model(idx, caches=caches, pos=0)[:, -1].float()         # prefill the prompt once
    out, seen = [], idx.clone()
    for i in range(max_new_tokens):
        if repetition_penalty != 1.0:                                 # CTRL-style penalty
            g = logits.gather(1, seen)
            logits.scatter_(1, seen, torch.where(g > 0, g / repetition_penalty, g * repetition_penalty))
        if banned_ids:
            logits[:, list(banned_ids)] = float("-inf")
        if no_repeat_ngram and seen.size(1) >= no_repeat_ngram:       # block repeated n-grams
            n = no_repeat_ngram
            for b in range(B):
                s = seen[b].tolist(); prefix = tuple(s[-(n - 1):])
                bad = [s[j + n - 1] for j in range(len(s) - n + 1) if tuple(s[j:j + n - 1]) == prefix]
                if bad:
                    logits[b, bad] = float("-inf")
        nxt = _apply_sampling(logits, temperature, top_k, top_p)      # (B, 1)
        out.append(nxt)
        seen = torch.cat([seen, nxt], 1)
        if i + 1 < max_new_tokens:                                    # one cached step
            logits = model(nxt, caches=caches, pos=T + i)[:, -1].float()
    return torch.cat(out, 1)

Writing /kaggle/working/src/lm_common.py


In [7]:
%%writefile /kaggle/working/src/model_dense.py
"""
model_dense.py -- M1: dense decoder-only Transformer (the controlled baseline).
Pre-norm: x = x + Attn(RMSNorm(x)); x = x + SwiGLU(RMSNorm(x)). GQA + QK-norm + RoPE,
tied input/output embeddings.
"""
from __future__ import annotations
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from lm_common import RMSNorm, Attention, SwiGLU, KVCache, rope_cache


@dataclass
class DenseConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 4
    ffn_hidden: int = 2048          # ~8/3 * d_model, multiple of 64 (tensor cores)
    max_seq_len: int = 2048         # RoPE table length (training uses 1024)
    rope_theta: float = 10000.0
    qk_norm: bool = True
    tie_embeddings: bool = True
    init_std: float = 0.02


class Block(nn.Module):
    def __init__(self, c: DenseConfig):
        super().__init__()
        self.norm1 = RMSNorm(c.d_model)
        self.attn = Attention(c.d_model, c.n_heads, c.n_kv_heads, c.qk_norm)
        self.norm2 = RMSNorm(c.d_model)
        self.mlp = SwiGLU(c.d_model, c.ffn_hidden)

    def forward(self, x, cos, sin, cache=None, pos=0):
        x = x + self.attn(self.norm1(x), cos, sin, cache, pos)
        return x + self.mlp(self.norm2(x))


class DenseGPT(nn.Module):
    def __init__(self, c: DenseConfig):
        super().__init__()
        self.cfg = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layers)])
        self.norm_f = RMSNorm(c.d_model)
        self.lm_head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        if c.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight                  # weight tying
        cos, sin = rope_cache(c.max_seq_len, c.d_model // c.n_heads, c.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        # GPT-2 trick: shrink residual-output projections by 1/sqrt(2 * n_layers)
        for n, p in self.named_parameters():
            if n.endswith("wo.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, c.init_std / math.sqrt(2 * c.n_layers))

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, self.cfg.init_std)

    def make_caches(self, B, max_T, device, dtype=torch.float16):
        hd, kv = self.cfg.d_model // self.cfg.n_heads, self.cfg.n_kv_heads
        return [KVCache(torch.zeros(B, kv, max_T, hd, device=device, dtype=dtype),
                        torch.zeros(B, kv, max_T, hd, device=device, dtype=dtype))
                for _ in self.blocks]

    def forward(self, idx, targets=None, caches=None, pos: int = 0):
        x = self.tok_emb(idx)
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope_cos, self.rope_sin, None if caches is None else caches[i], pos)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits                                              # (B, T, V)
        return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1),
                               ignore_index=-1)

Writing /kaggle/working/src/model_dense.py


In [8]:
%%writefile /kaggle/working/src/data.py
"""
data.py -- batches from per-source uint16 shards with a (possibly time-varying) mixture.
* memory-mapped shards (no RAM load); each sample reads seq_len+1 tokens
* sampling = pure function of (seed, step, rank, micro) -> exact resume, no loader state,
  DDP ranks never draw the same windows
* recipes: natural | sc_heavy | curriculum (broad mix -> SC-heavy anneal during LR decay)
"""
from __future__ import annotations
import numpy as np
import torch

SOURCES = ["acts", "supreme_court", "tribunals", "high_courts"]

MIXTURES = {
    "sc_heavy":  {"supreme_court": 0.50, "high_courts": 0.20, "tribunals": 0.15, "acts": 0.15},
    "broad":     {"supreme_court": 0.30, "high_courts": 0.30, "tribunals": 0.25, "acts": 0.15},
    "sc_anneal": {"supreme_court": 0.75, "acts": 0.10, "high_courts": 0.10, "tribunals": 0.05},
}


def mixture_weights(recipe: str, frac: float, n_tokens: dict, switch_at: float = 0.8):
    """Source weights at training progress `frac` in [0, 1]."""
    if recipe == "natural":
        tot = sum(n_tokens.values())
        return {s: n_tokens[s] / tot for s in SOURCES}
    if recipe == "curriculum":
        return MIXTURES["broad"] if frac < switch_at else MIXTURES["sc_anneal"]
    return MIXTURES[recipe]


class ShardSampler:
    def __init__(self, data_dir: str, split: str = "train", seq_len: int = 1024):
        self.seq_len = seq_len
        self.arrs = {s: np.memmap(f"{data_dir}/{split}_{s}.bin", dtype=np.uint16, mode="r")
                     for s in SOURCES}
        self.n_tokens = {s: len(a) for s, a in self.arrs.items()}

    def batch(self, weights: dict, B: int, seed: int, step: int, rank: int, micro: int):
        rng = np.random.default_rng([seed, step, rank, micro])       # deterministic stream
        names = list(weights)
        p = np.array([weights[s] for s in names], dtype=np.float64); p /= p.sum()
        srcs = rng.choice(len(names), size=B, p=p)                    # source per sequence
        T = self.seq_len
        buf = np.empty((B, T + 1), dtype=np.int64)
        for i, si in enumerate(srcs):
            a = self.arrs[names[si]]
            off = int(rng.integers(0, len(a) - T - 1))                # random window
            buf[i] = a[off:off + T + 1]
        t = torch.from_numpy(buf)
        return t[:, :-1], t[:, 1:], np.bincount(srcs, minlength=len(names))


def fixed_val_windows(data_dir: str, seq_len: int, n_windows: int):
    """Deterministic, spread-out validation windows per source: {src: LongTensor (N, T+1)}."""
    out = {}
    for s in SOURCES:
        a = np.memmap(f"{data_dir}/val_{s}.bin", dtype=np.uint16, mode="r")
        n = min(n_windows, (len(a) - 1) // (seq_len + 1))
        idx = np.linspace(0, len(a) - seq_len - 2, n).astype(np.int64)
        out[s] = torch.from_numpy(np.stack([a[i:i + seq_len + 1].astype(np.int64) for i in idx]))
    return out

Writing /kaggle/working/src/data.py


In [9]:
%%writefile /kaggle/working/src/train.py
"""
train.py -- Phase-1 pre-training loop (single GPU or DDP on Kaggle 2xT4).
Launch:  !torchrun --standalone --nproc_per_node=2 /kaggle/working/src/train.py --config <run.json>
fp16 autocast + GradScaler | DDP + grad accumulation (no_sync) | torch.compile | WSD or cosine LR
| curriculum mixture | per-source val loss + bits/word | atomic latest.pt / best.pt | exact
auto-resume | clean stop before the Kaggle session limit.
"""
from __future__ import annotations
import argparse, csv, json, math, os, sys, time
from contextlib import nullcontext
import torch
import torch.distributed as dist

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from data import ShardSampler, fixed_val_windows, mixture_weights, SOURCES   # noqa: E402
from lm_common import count_params                                         # noqa: E402


def build_model(arch: str, model_cfg: dict):
    """Model registry -- M2-M4 are added here later."""
    if arch == "dense":
        from model_dense import DenseGPT, DenseConfig
        return DenseGPT(DenseConfig(**model_cfg))
    raise ValueError(f"unknown arch {arch}")


def lr_at(step, total, cfg):
    """Warmup, then WSD (constant -> cosine decay over the last decay_frac) or plain cosine."""
    peak, floor = cfg["lr"], cfg["lr"] * cfg["min_lr_frac"]
    warm = max(1, int(cfg["warmup_frac"] * total))
    if step < warm:
        return peak * (step + 1) / warm
    if cfg["schedule"] == "cosine":
        prog = (step - warm) / max(1, total - warm)
        return floor + 0.5 * (peak - floor) * (1 + math.cos(math.pi * prog))
    decay_start = int((1 - cfg["decay_frac"]) * total)
    if step < decay_start:
        return peak
    prog = (step - decay_start) / max(1, total - decay_start)
    return floor + 0.5 * (peak - floor) * (1 + math.cos(math.pi * prog))


@torch.no_grad()
def evaluate(model, val_windows, tokens_per_word, device, micro_bs):
    """Mean loss per source on fixed windows -> loss, perplexity, bits per word."""
    model.eval()
    res = {}
    for s, W in val_windows.items():
        tot, n = 0.0, 0
        for i in range(0, len(W), micro_bs):
            b = W[i:i + micro_bs].to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(b[:, :-1], b[:, 1:])
            tot += loss.item() * len(b); n += len(b)
        nats = tot / max(1, n)
        res[s] = {"loss": nats, "ppl": math.exp(min(nats, 20)),
                  "bits_per_word": nats * tokens_per_word.get(s, 1.3) / math.log(2)}
    model.train()
    return res


def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)                     # never leaves a half-written checkpoint


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    cfg = json.load(open(ap.parse_args().config))

    # ---- distributed setup -----------------------------------------------------------------
    ddp = int(os.environ.get("WORLD_SIZE", "1")) > 1
    if ddp:
        dist.init_process_group("nccl")
        rank, world = dist.get_rank(), dist.get_world_size()
        local = int(os.environ["LOCAL_RANK"])
    else:
        rank, world, local = 0, 1, 0
    torch.cuda.set_device(local)
    device = torch.device("cuda", local)
    master = rank == 0
    torch.manual_seed(cfg["seed"])
    out_dir = cfg["out_dir"]
    os.makedirs(out_dir, exist_ok=True)
    log = (lambda *a: print(*a, flush=True)) if master else (lambda *a: None)

    # ---- batch geometry --------------------------------------------------------------------
    T, mbs = cfg["seq_len"], cfg["micro_bs"]
    accum = cfg["global_bs_seqs"] // (mbs * world)
    assert accum >= 1 and accum * mbs * world == cfg["global_bs_seqs"], "global batch must divide evenly"
    tokens_per_step = cfg["global_bs_seqs"] * T
    total_steps = cfg["max_tokens"] // tokens_per_step

    # ---- data ------------------------------------------------------------------------------
    sampler = ShardSampler(cfg["data_dir"], "train", T)
    val_windows = fixed_val_windows(cfg["data_dir"], T, cfg["val_windows"])
    tpw = {s: v["tokens_per_word"] for s, v in
           json.load(open(f"{cfg['data_dir']}/val_loss_stats.json")).items()}

    # ---- model / optimiser (weight decay on matrices only) ----------------------------------
    raw = build_model(cfg["arch"], cfg["model"]).to(device)
    n_params = count_params(raw)
    decay = [p for n, p in raw.named_parameters() if p.dim() >= 2]
    no_decay = [p for n, p in raw.named_parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": cfg["weight_decay"]},
                             {"params": no_decay, "weight_decay": 0.0}],
                            lr=cfg["lr"], betas=tuple(cfg["betas"]), eps=1e-8, fused=True)
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 14)

    # ---- resume ----------------------------------------------------------------------------
    start_step, tokens_seen, best_val, train_hours = 0, 0, float("inf"), 0.0
    ck_path = f"{out_dir}/latest.pt"
    if os.path.exists(ck_path):
        ck = torch.load(ck_path, map_location=device, weights_only=False)
        raw.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start_step, tokens_seen = ck["step"], ck["tokens_seen"]
        best_val, train_hours = ck.get("best_val", float("inf")), ck.get("train_hours", 0.0)
        log(f"RESUMED from step {start_step} ({tokens_seen/1e6:.0f}M tokens, {train_hours:.2f} h so far)")

    model = raw
    if cfg.get("compile", True):
        try:
            model = torch.compile(raw)
        except Exception as e:
            log("torch.compile failed, running eager:", repr(e)[:200])
    if ddp:
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[local])

    log(f"arch={cfg['arch']} params={n_params/1e6:.1f}M | world={world} micro_bs={mbs} accum={accum} "
        f"| {tokens_per_step/1e3:.0f}k tokens/step | {total_steps} steps = {cfg['max_tokens']/1e9:.2f}B tokens")
    json.dump({**cfg, "n_params": n_params, "total_steps": total_steps},
              open(f"{out_dir}/run_config.json", "w"), indent=1)
    csv_path = f"{out_dir}/train_log.csv"
    if master and not os.path.exists(csv_path):
        with open(csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["step", "tokens_M", "loss", "lr", "grad_norm", "scale",
                                    "tok_per_s", "sc_frac", "hours"])

    def save_latest(step):
        if master:
            atomic_save({"model": raw.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                         "step": step, "tokens_seen": tokens_seen, "best_val": best_val,
                         "train_hours": train_hours + (time.time() - t_start) / 3600, "cfg": cfg}, ck_path)

    # ---- training loop ---------------------------------------------------------------------
    model.train()
    t_start = t_log = t_ck = time.time()
    tok_since_log, n_acc = 0, 0
    loss_acc = torch.zeros((), device=device)          # accumulate on GPU: no per-micro-step sync
    stop = False
    for step in range(start_step, total_steps):
        frac = step / total_steps
        lr = lr_at(step, total_steps, cfg)
        for g in opt.param_groups:
            g["lr"] = lr
        w = mixture_weights(cfg["mixture"], frac, sampler.n_tokens, 1 - cfg["decay_frac"])

        for micro in range(accum):
            x, y, _ = sampler.batch(w, mbs, cfg["seed"], step, rank, micro)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            sync = nullcontext() if (not ddp or micro == accum - 1) else model.no_sync()
            with sync:                                  # all-reduce grads only on last micro-step
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = model(x, y) / accum
                scaler.scale(loss).backward()
            loss_acc += loss.detach(); tok_since_log += x.numel() * world
        n_acc += 1
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw.parameters(), cfg["grad_clip"]).item()
        scaler.step(opt); scaler.update()               # skips the step if grads overflowed
        opt.zero_grad(set_to_none=True)
        tokens_seen += tokens_per_step

        # ---- logging -------------------------------------------------------------------------
        if master and ((step + 1) % cfg["log_every"] == 0 or step == start_step):
            dt = time.time() - t_log
            row = [step + 1, round(tokens_seen / 1e6, 1), round(loss_acc.item() / n_acc, 4), f"{lr:.2e}",
                   round(gnorm, 3), int(scaler.get_scale()), int(tok_since_log / dt),
                   round(w["supreme_court"], 2), round(train_hours + (time.time() - t_start) / 3600, 3)]
            with open(csv_path, "a", newline="") as f:
                csv.writer(f).writerow(row)
            log(f"step {row[0]:6d}/{total_steps} | {row[1]:7.1f}M tok | loss {row[2]:.4f} | lr {row[3]} | "
                f"gnorm {row[4]:.2f} | scale {row[5]} | {row[6]/1e3:.1f}k tok/s | SC w {row[7]} | {row[8]:.2f} h")
            t_log, tok_since_log, n_acc = time.time(), 0, 0
            loss_acc.zero_()

        # ---- evaluation + best checkpoint ------------------------------------------------------
        last = step + 1 == total_steps
        if (step + 1) % cfg["eval_every"] == 0 or last:
            if master:
                res = evaluate(raw, val_windows, tpw, device, mbs)
                sc = res["supreme_court"]["loss"]
                with open(f"{out_dir}/eval_log.jsonl", "a") as f:
                    f.write(json.dumps({"step": step + 1, "tokens_M": tokens_seen / 1e6, **res}) + "\n")
                log("  EVAL " + " | ".join(f"{s[:5]} loss {r['loss']:.3f} bpw {r['bits_per_word']:.3f}"
                                           for s, r in res.items()))
                if sc < best_val:
                    best_val = sc
                    atomic_save({"model": {k: v.half() for k, v in raw.state_dict().items()},
                                 "arch": cfg["arch"], "model_cfg": cfg["model"], "step": step + 1,
                                 "val": res}, f"{out_dir}/best.pt")
                    log(f"  new best SC val loss {sc:.4f} -> best.pt")
            if ddp:
                dist.barrier()

        # ---- periodic checkpoint + time limit (decisions all-reduced -> ranks stay in lockstep)
        elapsed_h = (time.time() - t_start) / 3600
        flags = torch.tensor([float(elapsed_h > cfg["time_limit_hours"]),
                              float((time.time() - t_ck) / 60 > cfg["ckpt_every_min"])], device=device)
        if ddp:
            dist.all_reduce(flags, op=dist.ReduceOp.MAX)
        stop, want_ck = flags[0].item() > 0, flags[1].item() > 0
        if last or stop or want_ck:
            save_latest(step + 1)
            t_ck = time.time()
            log(f"  checkpoint saved at step {step + 1}")
            if ddp:
                dist.barrier()
        if stop:
            log(f"Time limit reached ({elapsed_h:.2f} h) -> stopped cleanly at step {step + 1}; "
                f"re-run the same command to resume.")
            break

    if master and not stop:
        log(f"TRAINING COMPLETE: {tokens_seen/1e9:.2f}B tokens, best SC val loss {best_val:.4f}")
    if ddp:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/src/train.py


In [3]:
# ===== Cell 23: Self-tests for the model code (GPU session, ~2-3 min) =====
import sys, time, math, importlib, torch
sys.path.insert(0, f"{WORK}/src")
import lm_common, model_dense, data
for m in (lm_common, model_dense, data):
    importlib.reload(m)
from model_dense import DenseGPT, DenseConfig
from lm_common import count_params, generate
from data import ShardSampler, mixture_weights
dev = "cuda"
print("GPU:", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0),
      "| torch", torch.__version__, "| GPUs:", torch.cuda.device_count())

# 1) the real M1 configuration must respect the 50M-300M rule
m1 = DenseGPT(DenseConfig())
n = count_params(m1)
print(f"[1] M1 parameters: {n/1e6:.1f}M (embedding {m1.tok_emb.weight.numel()/1e6:.1f}M, tied)")
assert 50e6 <= n <= 300e6
del m1

# 2) KV-cached decoding must reproduce the full forward pass (fp32, tiny model)
torch.manual_seed(0)
tiny_cfg = DenseConfig(d_model=128, n_layers=2, n_heads=4, n_kv_heads=2, ffn_hidden=256, max_seq_len=256)
tiny = DenseGPT(tiny_cfg).to(dev).float().eval()
x = torch.randint(16, 32000, (2, 40), device=dev)
with torch.no_grad():
    full = tiny(x)
    caches = tiny.make_caches(2, 40, dev, dtype=torch.float32)
    parts = [tiny(x[:, :30], caches=caches, pos=0)] + \
            [tiny(x[:, t:t + 1], caches=caches, pos=t) for t in range(30, 40)]
err = (torch.cat(parts, 1) - full).abs().max().item()
print(f"[2] KV-cache vs full forward: max |diff| = {err:.2e}")
assert err < 1e-3

# 3) generation API: correct shape, <|endoftext|> (id 0) never produced
with torch.autocast("cuda", dtype=torch.float16):
    g = generate(tiny, x[:, :30], 25, temperature=0.8, top_k=50, top_p=0.9,
                 repetition_penalty=1.1, no_repeat_ngram=3)
print(f"[3] generate -> {tuple(g.shape)}, contains EOT: {bool((g == 0).any())}")
assert g.shape == (2, 25) and not (g == 0).any()

# 4) fp16 forward/backward of the real M1 at training shape; initial loss ~ ln(32000) = 10.37
m1 = DenseGPT(DenseConfig()).to(dev)
sampler = ShardSampler(f"{WORK}/data", "train", 1024)
xb, yb, cnt = sampler.batch(mixture_weights("curriculum", 0.0, sampler.n_tokens), 4, 1337, 0, 0, 0)
xb, yb = xb.to(dev), yb.to(dev)
with torch.autocast("cuda", dtype=torch.float16):
    loss = m1(xb, yb)
loss.backward()
print(f"[4] initial loss {loss.item():.3f} (expected ~10.4) | peak mem {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
assert 9.5 < loss.item() < 11.5
del m1; torch.cuda.empty_cache()

# 5) a small model must overfit ONE real batch (end-to-end learning check, fp16 + GradScaler)
small = DenseGPT(DenseConfig(d_model=256, n_layers=4, n_heads=4, n_kv_heads=2, ffn_hidden=704)).to(dev)
opt = torch.optim.AdamW(small.parameters(), lr=2e-3)
scaler = torch.amp.GradScaler("cuda")
xs, ys = xb[:2, :256], yb[:2, :256]
t0 = time.time()
for it in range(300):
    with torch.autocast("cuda", dtype=torch.float16):
        l = small(xs, ys)
    scaler.scale(l).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
print(f"[5] overfit one batch: loss {l.item():.3f} after 300 steps ({time.time()-t0:.0f}s)")
assert l.item() < 1.0

# 6) data sampler: decode a training window to confirm tokens <-> text
from tokenizers import Tokenizer
tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
print("[6] sample window:", repr(tk.decode(xb[0, :60].tolist()))[:200])
print("    shard sizes (M tokens):", {s: round(v / 1e6) for s, v in sampler.n_tokens.items()})
print("\nALL SELF-TESTS PASSED")

GPU: Tesla T4 | capability (7, 5) | torch 2.10.0+cu128 | GPUs: 2
[1] M1 parameters: 100.1M (embedding 24.6M, tied)
[2] KV-cache vs full forward: max |diff| = 7.15e-07
[3] generate -> (2, 25), contains EOT: False
[4] initial loss 10.515 (expected ~10.4) | peak mem 4.68 GB
[5] overfit one batch: loss 0.008 after 300 steps (7s)
[6] sample window: ', and was responsible to, the company for the conduct of business of the company, as well as the company, shall be deemed to be guilty of the offence and shall be liable to be proceeded against and p
    shard sizes (M tokens): {'acts': 148, 'supreme_court': 263, 'tribunals': 548, 'high_courts': 609}

ALL SELF-TESTS PASSED


In [4]:
# ===== Cell 24: Throughput benchmark on one T4 (decides micro-batch + token budget) =====
import gc
def bench(model_kwargs, mbs, use_compile, steps=8, T=1024):
    """Tokens/s, peak memory and MFU for one training configuration on a single GPU."""
    torch.manual_seed(0)
    model = DenseGPT(DenseConfig(**model_kwargs)).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, fused=True)
    scaler = torch.amp.GradScaler("cuda")
    m = torch.compile(model) if use_compile else model
    x = torch.randint(16, 32000, (mbs, T), device=dev)
    torch.cuda.reset_peak_memory_stats()
    for i in range(steps + 3):                        # first 3 iterations = warm-up / compile
        if i == 3:
            torch.cuda.synchronize(); t0 = time.time()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = m(x, x)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    tps = mbs * T * steps / (time.time() - t0)
    mem = torch.cuda.max_memory_allocated() / 1e9
    N = count_params(model)
    flops_tok = 6 * N + 12 * model.cfg.n_layers * model.cfg.d_model * T   # params + attention FLOPs
    del model, m, opt, x; gc.collect(); torch.cuda.empty_cache()
    return tps, mem, flops_tok * tps / 65e12, N           # MFU vs T4 fp16 peak (65 TFLOPS)

print(f"{'config':30s} {'mbs':>4s} {'compile':>8s} {'tok/s':>9s} {'mem GB':>7s} {'MFU':>6s} {'h/1B tok (2xT4)':>16s}")
for name, kw in [("M1 d768 L12 (~100M)", {}),
                 ("M1-wide d1024 L12 (~166M)", dict(d_model=1024, n_heads=16, n_kv_heads=4, ffn_hidden=2752))]:
    for mbs in (8, 16):
        for c in (False, True):
            try:
                tps, mem, mfu, N = bench(kw, mbs, c)
                print(f"{name:30s} {mbs:4d} {str(c):>8s} {tps:9.0f} {mem:7.2f} {mfu:6.1%} "
                      f"{1e9 / (tps * 1.85) / 3600:16.1f}")        # ~1.85x from 2 GPUs (DDP)
            except torch.cuda.OutOfMemoryError:
                print(f"{name:30s} {mbs:4d} {str(c):>8s}   OOM"); gc.collect(); torch.cuda.empty_cache()

config                          mbs  compile     tok/s  mem GB    MFU  h/1B tok (2xT4)
M1 d768 L12 (~100M)               8    False     14311   10.44  15.7%             10.5


W0928 05:33:40.418000 58 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


M1 d768 L12 (~100M)               8     True     23643    5.96  26.0%              6.4
M1 d768 L12 (~100M)              16    False   OOM
M1 d768 L12 (~100M)              16     True     23825    9.44  26.2%              6.3
M1-wide d1024 L12 (~166M)         8    False      9827   12.96  17.3%             15.3
M1-wide d1024 L12 (~166M)         8     True     14606    7.89  25.7%             10.3
M1-wide d1024 L12 (~166M)        16    False   OOM
M1-wide d1024 L12 (~166M)        16     True     14724   12.32  25.9%             10.2


## Section 5: Pre-training runs
All architectures are trained under the **same controlled recipe** so differences come from the architecture:
- **Tokens:** 700M per model (compute-limited, ≈ 0.35 tokens-per-parameter of the Chinchilla-optimal 20 for 100M params)
- **Batch:** 256 sequences × 1,024 tokens = 262k tokens per step → 2,670 steps; micro-batch 8 per GPU × 2 GPUs × 16 accumulation
- **Optimiser:** AdamW (β = 0.9, 0.95; weight decay 0.1 on matrices), grad clip 1.0, fp16 + GradScaler, `torch.compile` (1.65× measured speed-up)
- **LR:** WSD, i.e. 3% warmup → constant 6e-4 → cosine decay to 6e-5 over the final 20%
- **Data:** curriculum. A broad legal mix (SC 30%, HC 30%, Tribunals 25%, Acts 15%) for the first 80%, then an **SC-heavy anneal** (SC 75%) during the LR decay, because about 90% of test prompts are Supreme Court text.
- **Evaluation** every 200 steps: per-source validation loss and bits per word. `best.pt` is selected by SC validation loss.

Benchmark (1×T4, 1,024-token sequences): M1 100M → 14.3k tok/s eager, **23.6k tok/s compiled** (26% MFU); a 166M wide variant runs at 14.6k tok/s.

In [1]:
# ===== Quick resume: M1 training after a session restart =====
import os, json
WORK = "/kaggle/working"                                     # the only thing Cell 25 needs from Cell 3
ck_dir = f"{WORK}/checkpoints/m1_dense_100m"

# ---- 1. confirm the checkpoint survived the restart --------------------------------------
if os.path.exists(f"{ck_dir}/latest.pt"):
    print(f"latest.pt found ({os.path.getsize(f'{ck_dir}/latest.pt')/1e9:.2f} GB) -> training will RESUME")
else:
    print("No latest.pt -> training would start from step 0 (check persistence settings!)")
print("files:", sorted(os.listdir(ck_dir)) if os.path.exists(ck_dir) else "run folder missing")

# ---- 2. re-launch with the SAME saved config; train.py resumes from latest.pt ------------
cfg_path = f"{ck_dir}/config.json"
!torchrun --standalone --nproc_per_node=2 {WORK}/src/train.py --config {cfg_path}

latest.pt found (1.20 GB) -> training will RESUME
files: ['best.pt', 'config.json', 'eval_log.jsonl', 'latest.pt', 'run_config.json', 'train_log.csv']
W0928 07:18:26.560000 86 torch/distributed/run.py:852] 
W0928 07:18:26.560000 86 torch/distributed/run.py:852] *****************************************
W0928 07:18:26.560000 86 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0928 07:18:26.560000 86 torch/distributed/run.py:852] *****************************************
[W928 07:18:27.162025709 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W928 07:18:28.956504407 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W928 07:18:29.529930561 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
RESU

In [2]:
# ===== Cell 25: Launch M1 (dense baseline) on 2xT4 with DDP =====
# Re-running this cell after an interruption RESUMES from latest.pt automatically.
import json, os

RUN = "m1_dense_100m"
run_cfg = {
    # ---- model -------------------------------------------------------------------------
    "arch": "dense", "model": {},            # DenseConfig defaults -> 100.1M params
    # ---- paths -------------------------------------------------------------------------
    "run_name": RUN, "out_dir": f"{WORK}/checkpoints/{RUN}", "data_dir": f"{WORK}/data",
    # ---- batch / budget (identical for every architecture) ------------------------------
    "seq_len": 1024, "micro_bs": 8, "global_bs_seqs": 256, "max_tokens": 700_000_000,
    # ---- optimisation ------------------------------------------------------------------
    "lr": 6e-4, "min_lr_frac": 0.1, "warmup_frac": 0.03, "schedule": "wsd", "decay_frac": 0.2,
    "weight_decay": 0.1, "betas": [0.9, 0.95], "grad_clip": 1.0, "compile": True, "seed": 1337,
    # ---- data curriculum (broad mix -> SC-heavy anneal during LR decay) ------------------
    "mixture": "curriculum",
    # ---- logging / eval / checkpoints --------------------------------------------------
    "log_every": 20, "eval_every": 200, "val_windows": 64,
    "ckpt_every_min": 15, "time_limit_hours": 11.0,   # stop cleanly before Kaggle's 12 h cap
}
os.makedirs(run_cfg["out_dir"], exist_ok=True)
cfg_path = f"{run_cfg['out_dir']}/config.json"
json.dump(run_cfg, open(cfg_path, "w"), indent=1)

# torchrun starts one process per GPU; output streams into this cell
!torchrun --standalone --nproc_per_node=2 {WORK}/src/train.py --config {cfg_path}

W0928 07:20:32.946000 802 torch/distributed/run.py:852] 
W0928 07:20:32.946000 802 torch/distributed/run.py:852] *****************************************
W0928 07:20:32.946000 802 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0928 07:20:32.946000 802 torch/distributed/run.py:852] *****************************************
[W928 07:20:33.418284777 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W928 07:20:35.143609310 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W928 07:20:35.208141090 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
RESUMED from step 849 (223M tokens, 1.51 h so far)
arch=dense params=100.1M | world=2 micro_bs=8 accum=16 | 262k tokens/step | 2670 steps = 0.70B token

## Section 6: Generation → local evaluation → Codabench submission
**Generation** (`generation.py`, `infer.py`):
- KV-cached decoding: the prompt is processed once, then one token per step.
- Prompts are kept to their last 824 tokens, so prompt + 200 generated tokens fit the 1,024-token training context. Only a few of the 1,000 test prompts are longer than that (max 952 tokens).
- Prompts are sorted by length and batched; each batch is cut to its shortest prompt from the left (drops only the oldest few tokens), so no padding or masking is needed.
- `<|endoftext|>` is banned, so the model can't stop early. The first token must start with a space, so the continuation starts a new word, as the reference does.
- Output is normalised and cut to **exactly 128 words**.

**Decoding study:** greedy vs sampling, repetition penalty (applied only to generated tokens, never to the prompt, so party names and sections can be reused), and no-repeat n-gram blocking. Measured on a fixed validation subset (200 SC + 50 of each other source).

**Local metrics** (`eval_local.py`): BERTScore F1 (roberta-large, the library default), ROUGE-L F1, BLEU-4, plus a repetition rate. The official scorer's exact BERTScore settings aren't published, so these numbers are for **relative** comparison.

**Baseline:** "copy the last 128 words of the prompt". It gives a floor score and serves as a format check for the first Codabench upload.

In [1]:
!pip install -q -U "huggingface_hub>=1.3,<2.0" hf_xet orjson pyahocorasick

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 7.0 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 28.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [8]:
# ===== Cell 26: evaluation packages (restart the kernel afterwards) =====
!pip install -q -U "huggingface_hub>=1.3,<2.0" bert-score rouge-score sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 13.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [3]:
%%writefile /kaggle/working/src/generation.py
"""
generation.py -- KV-cached decoding with the controls used in the Phase-1 decoding study.
(Supersedes lm_common.generate; training code is untouched.)
"""
from __future__ import annotations
import torch


def _apply_sampling(logits, temperature, top_k, top_p):
    """Greedy if temperature <= 0; otherwise temperature + optional top-k / top-p sampling."""
    if temperature <= 0:
        return logits.argmax(-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1).values[..., -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p and top_p < 1.0:                                         # nucleus sampling
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = sl.softmax(-1).cumsum(-1)
        remove = cp - sl.softmax(-1) > top_p                          # always keep the top token
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(-1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1)


@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.0, top_k=0, top_p=1.0,
             repetition_penalty=1.0, banned_ids=(0,), no_repeat_ngram=0,
             first_allowed=None, ngram_window=384):
    """KV-cached generation for a batch of EQUAL-length prompts. idx: (B, T) -> (B, max_new_tokens).
    banned_ids: <|endoftext|> by default, so the model cannot stop before 128 words.
    repetition_penalty: applied only to tokens GENERATED so far (not the prompt, whose party
      names / sections the continuation should be free to reuse).
    first_allowed: bool (V,) mask of tokens allowed as the FIRST generated token.
    ngram_window: no-repeat-ngram looks back this many tokens (bounds the Python cost)."""
    model.eval()
    B, T = idx.shape
    cdt = torch.float16 if torch.is_autocast_enabled() else next(model.parameters()).dtype
    caches = model.make_caches(B, T + max_new_tokens, idx.device, dtype=cdt)
    logits = model(idx, caches=caches, pos=0)[:, -1].float()         # prefill the prompt once
    out, seen = [], idx.clone()
    for i in range(max_new_tokens):
        if repetition_penalty != 1.0 and i > 0:                       # CTRL-style, generated only
            gen = seen[:, T:]
            g = logits.gather(1, gen)
            logits.scatter_(1, gen, torch.where(g > 0, g / repetition_penalty, g * repetition_penalty))
        if banned_ids:
            logits[:, list(banned_ids)] = float("-inf")
        if i == 0 and first_allowed is not None:                      # start a new word
            logits[:, ~first_allowed] = float("-inf")
        if no_repeat_ngram and seen.size(1) >= no_repeat_ngram:       # block repeated n-grams
            n = no_repeat_ngram
            for b in range(B):
                s = seen[b, -ngram_window:].tolist(); prefix = tuple(s[-(n - 1):])
                bad = [s[j + n - 1] for j in range(len(s) - n + 1) if tuple(s[j:j + n - 1]) == prefix]
                if bad:
                    logits[b, bad] = float("-inf")
        nxt = _apply_sampling(logits, temperature, top_k, top_p)      # (B, 1)
        out.append(nxt)
        seen = torch.cat([seen, nxt], 1)
        if i + 1 < max_new_tokens:                                    # one cached decoding step
            logits = model(nxt, caches=caches, pos=T + i)[:, -1].float()
    return torch.cat(out, 1)

Writing /kaggle/working/src/generation.py


In [4]:
%%writefile /kaggle/working/src/infer.py
"""
infer.py -- checkpoint -> 128-word continuations -> Codabench submission.
1) tokenize prompt, keep the LAST max_prompt tokens (prompt + generation <= 1,024 trained ctx)
2) sort by length, batch, cut each batch to its shortest prompt from the LEFT (no padding)
3) KV-cached generation, <|endoftext|> banned, first token starts with a space
4) decode, normalise, keep exactly 128 words; short rows get a second generation pass
"""
from __future__ import annotations
import json, os, sys, zipfile
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common                                   # noqa: E402
from generation import generate                 # noqa: E402

N_WORDS = 128


def load_model(ckpt_path, device="cuda"):
    """best.pt (fp16 weights + arch/model_cfg) -> fp32 module; inference runs under fp16 autocast."""
    from train import build_model
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model = build_model(ck["arch"], ck["model_cfg"])
    model.load_state_dict({k: v.float() for k, v in ck["model"].items()})
    return model.to(device).eval(), ck


def load_tokenizer(path):
    from tokenizers import Tokenizer
    return Tokenizer.from_file(path)


def space_start_mask(tk, vocab_size, device):
    """Bool (V,): True where the token's text starts with a space."""
    m = torch.zeros(vocab_size, dtype=torch.bool)
    for i in range(16, vocab_size):                   # ids < 16 are special tokens
        m[i] = tk.decode([i]).startswith(" ")
    return m.to(device)


@torch.no_grad()
def continue_prompts(model, tk, prompts: dict, device="cuda", batch_size=32,
                     max_ctx=1024, max_new_tokens=200, first_mask=None, **gen_kw):
    """prompts: {id: text} -> ({id: exactly-128-word continuation}, stats)."""
    ids = {k: tk.encode(v, add_special_tokens=False).ids for k, v in prompts.items()}
    max_prompt = max_ctx - max_new_tokens
    ids = {k: v[-max_prompt:] for k, v in ids.items()}
    order = sorted(ids, key=lambda k: len(ids[k]))            # similar lengths share a batch
    out, n_trunc = {}, 0
    for b in range(0, len(order), batch_size):
        keys = order[b:b + batch_size]
        L = min(len(ids[k]) for k in keys)
        n_trunc += sum(len(ids[k]) > L for k in keys)
        x = torch.tensor([ids[k][-L:] for k in keys], device=device)
        with torch.autocast("cuda", dtype=torch.float16):
            y = generate(model, x, max_new_tokens, first_allowed=first_mask, **gen_kw)
        for k, row in zip(keys, y.tolist()):
            out[k] = common.normalise(tk.decode(row))
    short = [k for k in out if len(out[k].split()) < N_WORDS]   # rare: extend short rows
    for k in short:
        ctx = (ids[k] + tk.encode(" " + out[k], add_special_tokens=False).ids)[-max_prompt:]
        with torch.autocast("cuda", dtype=torch.float16):
            y = generate(model, torch.tensor([ctx], device=device), max_new_tokens, **gen_kw)
        out[k] = common.normalise(out[k] + " " + tk.decode(y[0].tolist()))
    final = {k: " ".join(v.split()[:N_WORDS]) for k, v in out.items()}
    stats = {"n": len(final), "left_truncated_in_batch": n_trunc, "needed_second_pass": len(short),
             "min_words": min(len(v.split()) for v in final.values())}
    return final, stats


def write_submission(preds: dict, test_ids, out_dir, name):
    """Strict checks, then prediction.json at the ROOT of <name>.zip."""
    test_ids = set(map(str, test_ids))
    assert set(preds) == test_ids, f"ID mismatch: missing {len(test_ids - set(preds))}, extra {len(set(preds) - test_ids)}"
    bad = [k for k, v in preds.items() if len(v.split()) != N_WORDS]
    assert not bad, f"{len(bad)} predictions are not exactly {N_WORDS} words"
    os.makedirs(out_dir, exist_ok=True)
    js = {k: {"text": preds[k]} for k in sorted(preds, key=lambda s: int(s) if s.isdigit() else s)}
    pj = f"{out_dir}/prediction.json"
    json.dump(js, open(pj, "w"), ensure_ascii=True)
    zp = f"{out_dir}/{name}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")          # no folder inside the zip
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    return zp

Writing /kaggle/working/src/infer.py


In [5]:
%%writefile /kaggle/working/src/eval_local.py
"""
eval_local.py -- local proxy for the Phase-1 metrics on our validation set:
BERTScore F1 (roberta-large, library default), ROUGE-L F1 (stemmed), BLEU-4 (sacrebleu corpus).
Official scorer settings are unknown -> use for RELATIVE comparisons on identical prompts.
"""
from __future__ import annotations
import numpy as np


def _txt(v):
    return v["text"] if isinstance(v, dict) else v


def score(preds: dict, refs: dict, device="cuda", bert_model="roberta-large", batch_size=32,
          by_group: dict | None = None):
    keys = sorted(preds)
    cands = [_txt(preds[k]) for k in keys]
    gold = [_txt(refs[k]) for k in keys]
    from bert_score import score as bert_score
    _, _, F = bert_score(cands, gold, model_type=bert_model, lang="en", batch_size=batch_size,
                         device=device, verbose=False)
    F = F.numpy()
    from rouge_score import rouge_scorer
    rs = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rl = np.array([rs.score(g, c)["rougeL"].fmeasure for c, g in zip(cands, gold)])
    import sacrebleu
    out = {"n": len(keys), "bertscore_f1": float(F.mean()), "rougeL_f1": float(rl.mean()),
           "bleu4": float(sacrebleu.corpus_bleu(cands, [gold]).score)}
    if by_group:                                          # e.g. per-source BERTScore
        for g in sorted(set(by_group[k] for k in keys)):
            idx = [i for i, k in enumerate(keys) if by_group[k] == g]
            out[f"bertscore_f1[{g}]"] = float(F[idx].mean())
    return out


def repetition_rate(texts, n=3):
    """Mean fraction of repeated word 3-grams per text (degeneration check)."""
    r = []
    for t in texts:
        w = _txt(t).split()
        grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
        r.append(1 - len(set(grams)) / max(1, len(grams)))
    return float(np.mean(r))

Writing /kaggle/working/src/eval_local.py


In [6]:
# ===== Cell 30: setup + trivial baseline (floor score) + format-check submission =====
import sys, os, json, time, importlib, torch
WORK = "/kaggle/working"; DATA = f"{WORK}/data"; SUB = f"{WORK}/submissions"
sys.path.insert(0, f"{WORK}/src")
import generation, infer, eval_local
for m in (generation, infer, eval_local):
    importlib.reload(m)

tk   = infer.load_tokenizer(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
test = json.load(open(f"{DATA}/Phase_1_Pretraining/Test/test.json"))
VP, VR, VM = (json.load(open(f"{DATA}/val_{x}.json")) for x in ("prompts", "refs", "meta"))

# fixed decoding-study subset: 200 SC (test domain) + 50 from each other source
DEV = [k for k in VP if VM[k]["source"] == "supreme_court"][:200]
for s in ("acts", "tribunals", "high_courts"):
    DEV += [k for k in VP if VM[k]["source"] == s][:50]
GRP = {k: ("SC" if VM[k]["source"] == "supreme_court" else "other") for k in DEV}
print("dev subset:", len(DEV), "prompts")

# trivial baseline: repeat the last 128 words of the prompt (no model)
copy_last = lambda t: " ".join(t.split()[-128:])
base = {k: copy_last(VP[k]["text"]) for k in DEV}
res_base = eval_local.score(base, VR, by_group=GRP)
print("copy-last-128 baseline (val):", {k: round(v, 4) for k, v in res_base.items()})

# format-check submission on the real test set (same trivial baseline)
zp = infer.write_submission({k: copy_last(v["text"]) for k, v in test.items()}, test.keys(),
                            f"{SUB}/baseline_copy_last", "baseline_copy_last")
print("format-check zip:", zp)

dev subset: 350 prompts


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


copy-last-128 baseline (val): {'n': 350, 'bertscore_f1': 0.843, 'rougeL_f1': 0.2202, 'bleu4': 11.1995, 'bertscore_f1[SC]': 0.8412, 'bertscore_f1[other]': 0.8454}
format-check zip: /kaggle/working/submissions/baseline_copy_last/baseline_copy_last.zip


In [7]:
# ===== Cell 31: decoding study for M1 on the fixed validation subset =====
import pandas as pd
model, ck = infer.load_model(f"{WORK}/checkpoints/m1_dense_100m/best.pt")
print(f"loaded M1 best.pt (step {ck['step']}, SC val loss {ck['val']['supreme_court']['loss']:.4f})")
first_mask = infer.space_start_mask(tk, 32000, "cuda")

CONFIGS = {                                   # each row = one decoding strategy
    "greedy":                 dict(temperature=0),
    "greedy_norep3":          dict(temperature=0, no_repeat_ngram=3),
    "greedy_rep1.2":          dict(temperature=0, repetition_penalty=1.2),
    "greedy_rep1.2_norep4":   dict(temperature=0, repetition_penalty=1.2, no_repeat_ngram=4),
    "sample_t0.7_p0.9":       dict(temperature=0.7, top_p=0.9),
    "sample_t0.7_p0.9_rep1.1":dict(temperature=0.7, top_p=0.9, repetition_penalty=1.1),
}
dev_prompts = {k: VP[k]["text"] for k in DEV}
results, examples = {}, {}
for name, kw in CONFIGS.items():
    torch.manual_seed(1337); t0 = time.time()
    preds, st = infer.continue_prompts(model, tk, dev_prompts, first_mask=first_mask, **kw)
    r = eval_local.score(preds, VR, by_group=GRP)
    r["rep3"] = eval_local.repetition_rate(preds.values()); r["gen_sec"] = round(time.time() - t0, 1)
    results[name], examples[name] = r, preds[DEV[0]]
    print(f"{name:26s} BERT-F1 {r['bertscore_f1']:.4f} (SC {r['bertscore_f1[SC]']:.4f}) | "
          f"ROUGE-L {r['rougeL_f1']:.4f} | BLEU {r['bleu4']:.2f} | rep3 {r['rep3']:.3f} | {r['gen_sec']}s")

results["copy_last_baseline"] = res_base
df = pd.DataFrame(results).T.sort_values("bertscore_f1[SC]", ascending=False)
display(df.round(4))
os.makedirs(f"{WORK}/stats", exist_ok=True)
json.dump({"results": results, "examples": examples}, open(f"{WORK}/stats/decoding_m1.json", "w"), indent=1)

print("\nREFERENCE :", VR[DEV[0]]["text"][:300])
for name, ex in examples.items():
    print(f"\n{name}:", ex[:300])

loaded M1 best.pt (step 2670, SC val loss 2.3381)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


greedy                     BERT-F1 0.8206 (SC 0.8164) | ROUGE-L 0.2424 | BLEU 10.46 | rep3 0.585 | 207.4s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (11 outputs left out)

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


sample_t0.7_p0.9_rep1.1    BERT-F1 0.8430 (SC 0.8390) | ROUGE-L 0.2371 | BLEU 12.03 | rep3 0.056 | 128.0s


,n,bertscore_f1,rougeL_f1,bleu4,bertscore_f1[SC],bertscore_f1[other],rep3,gen_sec
copy_last_baseline,350.0,0.8430,0.2202,11.1995,0.8412,0.8454,NaN,NaN
greedy_rep1.2,350.0,0.8431,0.2312,12.6623,0.8397,0.8476,0.0543,155.0
sample_t0.7_p0.9_rep1.1,350.0,0.8430,0.2371,12.0338,0.8390,0.8483,0.0562,128.0
greedy_rep1.2_norep4,350.0,0.8393,0.2151,9.8860,0.8371,0.8422,0.0097,183.9
sample_t0.7_p0.9,350.0,0.8374,0.2453,11.3205,0.8351,0.8404,0.2148,178.9
greedy_norep3,350.0,0.8359,0.2304,8.2429,0.8340,0.8385,0.0000,199.2
greedy,350.0,0.8206,0.2424,10.4584,0.8164,0.8261,0.5850,207.4



REFERENCE : view, that, that alone will not justify a finding that the applicant is guilty of misconduct deserving departmental action and punishment." Immediately we prefer to record our total disapproval with the above observations of the Tribunal. We propose to deal with and rest our decision on the merits w

greedy: view that the applicant is entitled to be reinstated in service with all consequential benefits. The applicant is entitled to all consequential benefits. The respondents are directed to pay all the arrears of salary and other benefits to the applicant within a period of three months from the date of

greedy_norep3: view that the Tribunal was not justified in interfering with the order passed by the Disciplinary Authority." The Tribunal, therefore, set aside that order and directed the disciplinary authority to pass a fresh order after giving an opportunity to the delinquent to show cause as to why the order sh

greedy_rep1.2: view that the applicant is entitled to be r

In [9]:
# ===== Cell 32: M1 test predictions with the best decoding config =====
best_cfg = df.drop(index="copy_last_baseline").index[0]          # best SC BERTScore on val
print("best decoding config:", best_cfg, CONFIGS[best_cfg])
torch.manual_seed(1337)
preds, st = infer.continue_prompts(model, tk, {k: v["text"] for k, v in test.items()},
                                   first_mask=first_mask, **CONFIGS[best_cfg])
print("generation stats:", st)
name = f"m1_dense_{best_cfg}"
zp = infer.write_submission(preds, test.keys(), f"{SUB}/{name}", name)
json.dump({"model": "m1_dense_100m", "ckpt_step": ck["step"], "decoding": CONFIGS[best_cfg],
           "val_scores": results[best_cfg]}, open(f"{SUB}/{name}/meta.json", "w"), indent=1)
print("submission zip:", zp)
print("\nexample -- test id 1:\n", preds["1"])

best decoding config: greedy_rep1.2 {'temperature': 0, 'repetition_penalty': 1.2}
generation stats: {'n': 1000, 'left_truncated_in_batch': 832, 'needed_second_pass': 43, 'min_words': 90}


AssertionError: 3 predictions are not exactly 128 words

In [10]:
# ===== Cell 32b: guarantee exactly 128 words, then write the M1 submission =====
test_prompts = {k: v["text"] for k, v in test.items()}

def ensure_128(preds, prompts, cfg, max_rounds=3):
    """Extend short rows by generating more text; last resort: pad from the prompt's tail."""
    for r in range(max_rounds):
        short = [k for k in preds if len(preds[k].split()) < 128]
        if not short:
            break
        print(f"round {r+1}: extending {len(short)} short rows")
        more, _ = infer.continue_prompts(model, tk, {k: prompts[k] + " " + preds[k] for k in short},
                                         first_mask=first_mask, **cfg)
        for k in short:
            preds[k] = " ".join((preds[k] + " " + more[k]).split()[:128])
    for k in preds:                                               # fallback (should be rare/none)
        w = preds[k].split()
        if len(w) < 128:
            print(f"  padding id {k} ({len(w)} words) from prompt tail")
            preds[k] = " ".join(w + prompts[k].split()[-(128 - len(w)):])
    return preds

short_before = [k for k in preds if len(preds[k].split()) < 128]
for k in short_before[:1]:                                       # look at one problem case
    print(f"example short output (id {k}):", preds[k][:400], "\n")
preds = ensure_128(preds, test_prompts, CONFIGS[best_cfg])

zp = infer.write_submission(preds, test.keys(), f"{SUB}/{name}", name)
json.dump({"model": "m1_dense_100m", "ckpt_step": ck["step"], "decoding": CONFIGS[best_cfg],
           "val_scores": results[best_cfg], "short_rows_fixed": len(short_before)},
          open(f"{SUB}/{name}/meta.json", "w"), indent=1)
print("submission zip:", zp)

example short output (id 918): of the principles laid down in Masalti (supra), we are satisfied that the High Court was right in holding that the appellants were guilty of an offence under Section 302 read with [Section 149] IPC. 20.In view of the above, we do not find any merit in this appeal which is dismissed accordingly. J. (Dr. ARIJIT PASAYAT) J. (ASOK KUMAR GANGULY) New Delhi, April 15, 2009 13-14-2009 14-14-2009 15-15-20 

round 1: extending 3 short rows
submission zip: /kaggle/working/submissions/m1_dense_greedy_rep1.2/m1_dense_greedy_rep1.2.zip


## Section 7: M2, DeepSeek-style decoder (Multi-head Latent Attention + DeepSeekMoE)
**Motivation (constraint-driven):** the 50–300M rule caps *total* parameters, but training cost scales with *active* parameters.
- **DeepSeekMoE** (1 shared expert + 16 fine-grained routed experts of width 512, top-3) gives about **270M total parameters** but only about **100M active per token**, the same compute as M1. M1 vs M2 is therefore a **compute-matched dense vs sparse comparison.**
- **Multi-head Latent Attention (MLA)** caches one low-rank latent (d_c = 256) plus a shared 32-dim RoPE key per token: **288 values per token per layer**, vs 512 for M1's GQA and 1,536 for full MHA. At decode time the up-projections are **absorbed** into the query and output, so keys and values are never rebuilt. That's what makes many-sample decoding (MBR) cheap.
- **Decoupled RoPE:** position information goes through a separate shared key, because rotary encoding can't pass through the low-rank projection.
- **Auxiliary-loss-free load balancing** (DeepSeek-V3): a per-expert bias affects expert *selection* only. After each step it moves each expert's bias by γ·sign(mean load − expert load), so balancing never competes with the language-modelling loss.
- **Static-shape expert dispatch:** tokens are grouped by expert with a sort, placed into fixed per-expert capacity slots (1.25× the average load; overflow is dropped), and all experts run as **batched matmuls**. That keeps it compatible with `torch.compile`, instead of 16 small GEMMs in a Python loop.
- Layer 0 keeps a dense FFN, as DeepSeek does, to stabilise early routing.

**Same controlled recipe as M1:** 700M tokens, the same batch size, LR schedule, curriculum and evaluation.

In [7]:
%%writefile /kaggle/working/src/model_mla_moe.py
"""
model_mla_moe.py -- M2: DeepSeek-V2/V3-style decoder: Multi-head Latent Attention (MLA) +
DeepSeekMoE (fine-grained routed experts + shared expert) with auxiliary-loss-free balancing.
MLA:  q = W_q x -> [q_nope | q_rope]; c_kv = RMSNorm(W_dkv x); k_rope = RoPE(W_kr x) (shared);
      k_nope = W_uk c_kv; v = W_uv c_kv.  Decode (absorbed): score = (W_uk^T q_nope).c_kv + q_rope.k_rope
MoE:  s_i = sigmoid(u.e_i); top-k by (s_i + b_i); gate = s / sum(s_topk); y = Shared(x) + sum g_i E_i(x)
      after each optimizer step: b_i += gamma * sign(mean_load - load_i)   (no auxiliary loss)
"""
from __future__ import annotations
import math
from dataclasses import dataclass
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from lm_common import RMSNorm, SwiGLU, rope_cache, apply_rope


@dataclass
class MLAMoEConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    qk_nope_dim: int = 32          # per-head key/query part without position (from the latent)
    qk_rope_dim: int = 32          # per-head decoupled RoPE part (key shared across heads)
    v_head_dim: int = 64           # nope + rope == v dim -> SDPA's fused kernel applies
    kv_latent_dim: int = 256       # d_c: the only K/V state that is cached
    n_routed_experts: int = 16
    n_shared_experts: int = 1
    top_k: int = 3                 # shared(1) + routed(3) = 4 x 512 = 2048 active hidden (= M1)
    expert_hidden: int = 512       # fine-grained experts (M1's 2048-wide FFN split 4 ways)
    first_dense_layers: int = 1    # layer 0 keeps a dense FFN (stabilises early routing)
    dense_ffn_hidden: int = 2048
    capacity_factor: float = 1.25  # training-time slots per expert = 1.25 x average load
    bias_update_rate: float = 1e-3 # gamma of the aux-loss-free balancing
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    tie_embeddings: bool = True
    init_std: float = 0.02


@dataclass
class MLACache:
    c: torch.Tensor                # (B, max_T, d_c)        compressed KV latent
    kr: torch.Tensor               # (B, 1, max_T, rope)    shared RoPE key


class MLAttention(nn.Module):
    def __init__(self, c: MLAMoEConfig):
        super().__init__()
        self.h, self.nope, self.rope, self.vd, self.dc = (c.n_heads, c.qk_nope_dim, c.qk_rope_dim,
                                                          c.v_head_dim, c.kv_latent_dim)
        qk = self.nope + self.rope
        assert qk == self.vd, "keep qk head dim == v head dim so SDPA can use its fused kernel"
        self.wq = nn.Linear(c.d_model, self.h * qk, bias=False)
        self.w_dkv = nn.Linear(c.d_model, self.dc, bias=False)          # down-projection to latent
        self.w_kr = nn.Linear(c.d_model, self.rope, bias=False)         # decoupled RoPE key
        self.w_uk = nn.Linear(self.dc, self.h * self.nope, bias=False)  # latent -> keys
        self.w_uv = nn.Linear(self.dc, self.h * self.vd, bias=False)    # latent -> values
        self.wo = nn.Linear(self.h * self.vd, c.d_model, bias=False)
        self.q_norm, self.kv_norm, self.kr_norm = RMSNorm(qk), RMSNorm(self.dc), RMSNorm(self.rope)
        self.scale = qk ** -0.5

    def forward(self, x, cos, sin, cache: MLACache | None = None, pos: int = 0):
        B, T, _ = x.shape
        q = self.q_norm(self.wq(x).view(B, T, self.h, self.nope + self.rope).transpose(1, 2))
        q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
        q_rope = apply_rope(q_rope, cos[pos:pos + T], sin[pos:pos + T])
        c_kv = self.kv_norm(self.w_dkv(x))                                      # (B, T, d_c)
        k_rope = apply_rope(self.kr_norm(self.w_kr(x)).unsqueeze(1),
                            cos[pos:pos + T], sin[pos:pos + T])                 # (B, 1, T, rope)
        if cache is not None:                                                   # cache latents only
            cache.c[:, pos:pos + T] = c_kv
            cache.kr[:, :, pos:pos + T] = k_rope

        if T > 1 or cache is None:       # training / prefill: materialise K,V once, fused SDPA
            assert pos == 0, "chunked prefill is not supported"
            k_nope = self.w_uk(c_kv).view(B, T, self.h, self.nope).transpose(1, 2)
            v = self.w_uv(c_kv).view(B, T, self.h, self.vd).transpose(1, 2)
            k = torch.cat([k_nope, k_rope.expand(B, self.h, T, self.rope)], dim=-1)
            y = F.scaled_dot_product_attention(torch.cat([q_nope, q_rope], -1), k, v, is_causal=T > 1)
        else:                            # decoding: ABSORBED form, attends straight to the latents
            C = cache.c[:, :pos + 1]                                            # (B, S, d_c)
            Kr = cache.kr[:, 0, :pos + 1]                                       # (B, S, rope)
            w_uk = self.w_uk.weight.view(self.h, self.nope, self.dc)
            w_uv = self.w_uv.weight.view(self.h, self.vd, self.dc)

Overwriting /kaggle/working/src/model_mla_moe.py


In [8]:
%%writefile -a /kaggle/working/src/model_mla_moe.py


class Block(nn.Module):
    def __init__(self, c: MLAMoEConfig, dense: bool):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(c.d_model), RMSNorm(c.d_model)
        self.attn = MLAttention(c)
        self.ffn = SwiGLU(c.d_model, c.dense_ffn_hidden) if dense else DeepSeekMoE(c)

    def forward(self, x, cos, sin, cache=None, pos=0):
        x = x + self.attn(self.norm1(x), cos, sin, cache, pos)
        return x + self.ffn(self.norm2(x))


class MLAMoEGPT(nn.Module):
    def __init__(self, c: MLAMoEConfig):
        super().__init__()
        self.cfg = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c, dense=i < c.first_dense_layers) for i in range(c.n_layers)])
        self.norm_f = RMSNorm(c.d_model)
        self.lm_head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        if c.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = rope_cache(c.max_seq_len, c.qk_rope_dim, c.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.moe_layers = [b.ffn for b in self.blocks if isinstance(b.ffn, DeepSeekMoE)]
        self.apply(self._init)
        for n, p in self.named_parameters():               # expert weights + scaled residual outputs
            if n.endswith((".w1", ".w3")):
                nn.init.normal_(p, 0.0, c.init_std)
            if n.endswith((".w2", "wo.weight", "w2.weight")):
                nn.init.normal_(p, 0.0, c.init_std / math.sqrt(2 * c.n_layers))

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, self.cfg.init_std)

    def active_params(self):
        # parameters used per token: shared + top-k routed experts (not all 16)
        c = self.cfg
        total = sum(p.numel() for p in {id(p): p for p in self.parameters()}.values())
        inactive = len(self.moe_layers) * (c.n_routed_experts - c.top_k) * 3 * c.d_model * c.expert_hidden
        return total - inactive

    def make_caches(self, B, max_T, device, dtype=torch.float16):
        c = self.cfg
        return [MLACache(torch.zeros(B, max_T, c.kv_latent_dim, device=device, dtype=dtype),
                         torch.zeros(B, 1, max_T, c.qk_rope_dim, device=device, dtype=dtype))
                for _ in self.blocks]

    @torch.no_grad()
    def after_step(self):
        # Aux-loss-free balancing: loads summed over DDP ranks -> identical bias update everywhere
        if not self.moe_layers:
            return {}
        loads = torch.stack([m.load_accum for m in self.moe_layers])          # (L, E)
        drops = torch.stack([m.drop_accum for m in self.moe_layers]).sum(0)   # (dropped, total)
        if dist.is_available() and dist.is_initialized():
            dist.all_reduce(loads); dist.all_reduce(drops)
        mean = loads.mean(1, keepdim=True)
        for m, l, mu in zip(self.moe_layers, loads, mean):
            m.router_bias += self.cfg.bias_update_rate * torch.sign(mu - l)
            m.load_accum.zero_(); m.drop_accum.zero_()
        max_over_mean = (loads.max(1).values / mean.squeeze(1).clamp_min(1)).mean()
        return {"moe_max/mean": round(max_over_mean.item(), 3),
                "drop%": round(100 * (drops[0] / drops[1].clamp_min(1)).item(), 2)}

    def forward(self, idx, targets=None, caches=None, pos: int = 0):
        x = self.tok_emb(idx)
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope_cos, self.rope_sin, None if caches is None else caches[i], pos)
        logits = self.lm_head(self.norm_f(x))
        if targets is None:
            return logits
        return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-1)

Appending to /kaggle/working/src/model_mla_moe.py


In [11]:
# ===== Cell 33c: verify the written modules compile and are complete =====
import py_compile
for f in ["model_mla_moe.py", "train.py"]:
    p = f"/kaggle/working/src/{f}"
    py_compile.compile(p, doraise=True)                    # raises if the file is truncated/broken
    src = open(p).read()
    print(f"{f}: {src.count(chr(10))} lines, compiles OK | last line: {src.strip().splitlines()[-1][:60]!r}")

model_mla_moe.py: 164 lines, compiles OK | last line: '        return F.cross_entropy(logits.float().view(-1, logit'
train.py: 242 lines, compiles OK | last line: '    main()'


In [12]:
%%writefile -a /kaggle/working/src/model_mla_moe.py


class DeepSeekMoE(nn.Module):
    # 1 shared expert + E fine-grained routed experts, top-k routing with aux-loss-free bias
    def __init__(self, c: MLAMoEConfig):
        super().__init__()
        E, d, h = c.n_routed_experts, c.d_model, c.expert_hidden
        self.E, self.k, self.cap = E, c.top_k, c.capacity_factor
        self.shared = SwiGLU(d, h * c.n_shared_experts)
        self.router = nn.Linear(d, E, bias=False)
        self.w1 = nn.Parameter(torch.empty(E, d, h))     # gate  (all experts stacked)
        self.w3 = nn.Parameter(torch.empty(E, d, h))     # up
        self.w2 = nn.Parameter(torch.empty(E, h, d))     # down
        self.register_buffer("router_bias", torch.zeros(E))                   # saved, not trained
        self.register_buffer("load_accum", torch.zeros(E), persistent=False)  # tokens per expert
        self.register_buffer("drop_accum", torch.zeros(2), persistent=False)  # (dropped, total)

    def forward(self, x):
        B, T, d = x.shape
        N, E, k = B * T, self.E, self.k
        xf = x.reshape(N, d)
        with torch.autocast("cuda", enabled=False):                          # routing in fp32
            aff = torch.sigmoid(xf.float() @ self.router.weight.float().t())  # (N, E)
        _, idx = torch.topk(aff + self.router_bias, k, dim=-1)                # bias only SELECTS
        gate = aff.gather(1, idx)
        gate = gate / gate.sum(-1, keepdim=True)                              # (N, k)

        flat_e = idx.reshape(-1)                                              # (N*k,)
        flat_tok = torch.arange(N, device=x.device).repeat_interleave(k)
        C = N if not self.training else int(math.ceil(self.cap * N * k / E))  # no drops at eval
        counts = torch.zeros(E, device=x.device).scatter_add_(0, flat_e, torch.ones_like(flat_e, dtype=torch.float))
        order = torch.argsort(flat_e, stable=True)                            # group by expert
        se, tok = flat_e[order], flat_tok[order]
        starts = torch.cumsum(counts, 0) - counts
        slot = torch.arange(N * k, device=x.device) - starts[se].long()       # position in expert
        keep = slot < C
        dest = torch.where(keep, se * C + slot, torch.full_like(se, E * C))   # overflow -> dummy row

        buf = xf.new_zeros(E * C + 1, d)
        buf[dest] = xf[tok]                                                   # dispatch
        h = buf[:E * C].view(E, C, d)
        h = F.silu(torch.bmm(h, self.w1)) * torch.bmm(h, self.w3)              # all experts, 1 bmm
        o = torch.bmm(h, self.w2).reshape(E * C, d)
        o = torch.cat([o, o.new_zeros(1, d)], 0)
        wts = (gate.reshape(-1)[order] * keep).to(o.dtype).unsqueeze(1)
        y = torch.zeros(N, d, device=x.device, dtype=o.dtype).index_add_(0, tok, o[dest] * wts)  # combine

        if self.training:                                                     # stats for balancing
            self.load_accum += counts
            self.drop_accum += torch.stack([(~keep).sum().float(), xf.new_tensor(float(N * k))])
        return self.shared(x) + y.view(B, T, d).to(x.dtype)

Appending to /kaggle/working/src/model_mla_moe.py


In [13]:
# ===== Cell 33e: verify model_mla_moe.py has every class exactly once and compiles =====
import py_compile, re
p = "/kaggle/working/src/model_mla_moe.py"
py_compile.compile(p, doraise=True)
src = open(p).read()
for name in ["MLAMoEConfig", "MLACache", "MLAttention", "DeepSeekMoE", "Block", "MLAMoEGPT"]:
    n = len(re.findall(rf"^class {name}\b", src, flags=re.M))
    print(f"class {name:14s}: {'OK' if n == 1 else f'PROBLEM ({n} copies)'}")
for fn in ["def after_step", "def make_caches", "def active_params", "absorbed", "def forward(self, idx"]:
    print(f"{fn:22s}: {'present' if fn in src else 'MISSING'}")
print(f"{src.count(chr(10))} lines total")

class MLAMoEConfig  : OK
class MLACache      : OK
class MLAttention   : OK
class DeepSeekMoE   : OK
class Block         : OK
class MLAMoEGPT     : OK
def after_step        : present
def make_caches       : present
def active_params     : present
absorbed              : present
def forward(self, idx : present
215 lines total


In [2]:
%%writefile -a /kaggle/working/src/model_mla_moe.py


class MLAttention(nn.Module):
    # complete definition (replaces the truncated one above when the module loads)
    def __init__(self, c: MLAMoEConfig):
        super().__init__()
        self.h, self.nope, self.rope, self.vd, self.dc = (c.n_heads, c.qk_nope_dim, c.qk_rope_dim,
                                                          c.v_head_dim, c.kv_latent_dim)
        qk = self.nope + self.rope
        assert qk == self.vd, "keep qk head dim == v head dim so SDPA can use its fused kernel"
        self.wq = nn.Linear(c.d_model, self.h * qk, bias=False)
        self.w_dkv = nn.Linear(c.d_model, self.dc, bias=False)          # down-projection to latent
        self.w_kr = nn.Linear(c.d_model, self.rope, bias=False)         # decoupled RoPE key
        self.w_uk = nn.Linear(self.dc, self.h * self.nope, bias=False)  # latent -> keys
        self.w_uv = nn.Linear(self.dc, self.h * self.vd, bias=False)    # latent -> values
        self.wo = nn.Linear(self.h * self.vd, c.d_model, bias=False)
        self.q_norm, self.kv_norm, self.kr_norm = RMSNorm(qk), RMSNorm(self.dc), RMSNorm(self.rope)
        self.scale = qk ** -0.5

    def forward(self, x, cos, sin, cache=None, pos=0):
        B, T, _ = x.shape
        q = self.q_norm(self.wq(x).view(B, T, self.h, self.nope + self.rope).transpose(1, 2))
        q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
        q_rope = apply_rope(q_rope, cos[pos:pos + T], sin[pos:pos + T])
        c_kv = self.kv_norm(self.w_dkv(x))                                      # (B, T, d_c)
        k_rope = apply_rope(self.kr_norm(self.w_kr(x)).unsqueeze(1),
                            cos[pos:pos + T], sin[pos:pos + T])                 # (B, 1, T, rope)
        if cache is not None:                                                   # cache latents only
            cache.c[:, pos:pos + T] = c_kv
            cache.kr[:, :, pos:pos + T] = k_rope

        if T > 1 or cache is None:       # training / prefill: materialise K,V once, fused SDPA
            assert pos == 0, "chunked prefill is not supported"
            k_nope = self.w_uk(c_kv).view(B, T, self.h, self.nope).transpose(1, 2)
            v = self.w_uv(c_kv).view(B, T, self.h, self.vd).transpose(1, 2)
            k = torch.cat([k_nope, k_rope.expand(B, self.h, T, self.rope)], dim=-1)
            y = F.scaled_dot_product_attention(torch.cat([q_nope, q_rope], -1), k, v, is_causal=T > 1)
        else:                            # decoding: ABSORBED form, attends straight to the latents
            Cc = cache.c[:, :pos + 1]                                           # (B, S, d_c)
            Kr = cache.kr[:, 0, :pos + 1]                                       # (B, S, rope)
            w_uk = self.w_uk.weight.view(self.h, self.nope, self.dc)
            w_uv = self.w_uv.weight.view(self.h, self.vd, self.dc)
            q_lat = torch.einsum("bhtn,hnc->bhtc", q_nope, w_uk.to(q_nope.dtype))   # W_uk^T q
            s = (torch.einsum("bhtc,bsc->bhts", q_lat, Cc.to(q_lat.dtype)) +
                 torch.einsum("bhtr,bsr->bhts", q_rope, Kr.to(q_rope.dtype)))
            p = torch.softmax(s.float() * self.scale, dim=-1).to(q_lat.dtype)
            ctx = torch.einsum("bhts,bsc->bhtc", p, Cc.to(p.dtype))            # (B, H, 1, d_c)
            y = torch.einsum("bhtc,hvc->bhtv", ctx, w_uv.to(ctx.dtype))         # W_uv . ctx
        return self.wo(y.transpose(1, 2).reshape(B, T, self.h * self.vd))

Appending to /kaggle/working/src/model_mla_moe.py


In [9]:
%%writefile /kaggle/working/src/train.py
"""
train.py -- Phase-1 pre-training loop (single GPU or DDP on Kaggle 2xT4).
Launch:  torchrun --standalone --nproc_per_node=2 /kaggle/working/src/train.py --config <run.json>
fp16 autocast + GradScaler | DDP + grad accumulation (no_sync) | torch.compile | WSD or cosine LR
| curriculum mixture | per-source val loss + bits/word | atomic latest.pt / best.pt | exact
auto-resume | clean stop before the Kaggle session limit | per-step model hook (MoE balancing).
"""
from __future__ import annotations
import argparse, csv, json, math, os, sys, time
from contextlib import nullcontext
import torch
import torch.distributed as dist

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from data import ShardSampler, fixed_val_windows, mixture_weights, SOURCES   # noqa: E402
from lm_common import count_params                                         # noqa: E402


def build_model(arch: str, model_cfg: dict):
    """Model registry (M3, M4 added later)."""
    if arch == "dense":
        from model_dense import DenseGPT, DenseConfig
        return DenseGPT(DenseConfig(**model_cfg))
    if arch == "mla_moe":
        from model_mla_moe import MLAMoEGPT, MLAMoEConfig
        return MLAMoEGPT(MLAMoEConfig(**model_cfg))
    raise ValueError(f"unknown arch {arch}")


def lr_at(step, total, cfg):
    """Warmup, then WSD (constant -> cosine decay over the last decay_frac) or plain cosine."""
    peak, floor = cfg["lr"], cfg["lr"] * cfg["min_lr_frac"]
    warm = max(1, int(cfg["warmup_frac"] * total))
    if step < warm:
        return peak * (step + 1) / warm
    if cfg["schedule"] == "cosine":
        prog = (step - warm) / max(1, total - warm)
        return floor + 0.5 * (peak - floor) * (1 + math.cos(math.pi * prog))
    decay_start = int((1 - cfg["decay_frac"]) * total)
    if step < decay_start:
        return peak
    prog = (step - decay_start) / max(1, total - decay_start)
    return floor + 0.5 * (peak - floor) * (1 + math.cos(math.pi * prog))


@torch.no_grad()
def evaluate(model, val_windows, tokens_per_word, device, micro_bs):
    """Mean loss per source on fixed windows -> loss, perplexity, bits per word."""
    model.eval()
    res = {}
    for s, W in val_windows.items():
        tot, n = 0.0, 0
        for i in range(0, len(W), micro_bs):
            b = W[i:i + micro_bs].to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(b[:, :-1], b[:, 1:])
            tot += loss.item() * len(b); n += len(b)
        nats = tot / max(1, n)
        res[s] = {"loss": nats, "ppl": math.exp(min(nats, 20)),
                  "bits_per_word": nats * tokens_per_word.get(s, 1.3) / math.log(2)}
    model.train()
    return res


def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)                     # never leaves a half-written checkpoint


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    cfg = json.load(open(ap.parse_args().config))

    # ---- distributed setup -----------------------------------------------------------------
    ddp = int(os.environ.get("WORLD_SIZE", "1")) > 1
    if ddp:
        dist.init_process_group("nccl")
        rank, world = dist.get_rank(), dist.get_world_size()
        local = int(os.environ["LOCAL_RANK"])
    else:
        rank, world, local = 0, 1, 0
    torch.cuda.set_device(local)
    device = torch.device("cuda", local)
    master = rank == 0
    torch.manual_seed(cfg["seed"])
    out_dir = cfg["out_dir"]
    os.makedirs(out_dir, exist_ok=True)
    log = (lambda *a: print(*a, flush=True)) if master else (lambda *a: None)

    # ---- batch geometry --------------------------------------------------------------------
    T, mbs = cfg["seq_len"], cfg["micro_bs"]
    accum = cfg["global_bs_seqs"] // (mbs * world)
    assert accum >= 1 and accum * mbs * world == cfg["global_bs_seqs"], "global batch must divide evenly"
    tokens_per_step = cfg["global_bs_seqs"] * T
    total_steps = cfg["max_tokens"] // tokens_per_step

    # ---- data ------------------------------------------------------------------------------
    sampler = ShardSampler(cfg["data_dir"], "train", T)
    val_windows = fixed_val_windows(cfg["data_dir"], T, cfg["val_windows"])
    tpw = {s: v["tokens_per_word"] for s, v in
           json.load(open(f"{cfg['data_dir']}/val_loss_stats.json")).items()}

    # ---- model / optimiser (weight decay on matrices only) ----------------------------------
    raw = build_model(cfg["arch"], cfg["model"]).to(device)
    n_params = count_params(raw)
    n_active = raw.active_params() if hasattr(raw, "active_params") else n_params
    decay = [p for n, p in raw.named_parameters() if p.dim() >= 2]
    no_decay = [p for n, p in raw.named_parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": cfg["weight_decay"]},
                             {"params": no_decay, "weight_decay": 0.0}],
                            lr=cfg["lr"], betas=tuple(cfg["betas"]), eps=1e-8, fused=True)
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 14)

    # ---- resume ----------------------------------------------------------------------------
    start_step, tokens_seen, best_val, train_hours = 0, 0, float("inf"), 0.0
    ck_path = f"{out_dir}/latest.pt"
    if os.path.exists(ck_path):
        ck = torch.load(ck_path, map_location=device, weights_only=False)
        raw.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start_step, tokens_seen = ck["step"], ck["tokens_seen"]
        best_val, train_hours = ck.get("best_val", float("inf")), ck.get("train_hours", 0.0)
        log(f"RESUMED from step {start_step} ({tokens_seen/1e6:.0f}M tokens, {train_hours:.2f} h so far)")

    model = raw
    if cfg.get("compile", True):
        try:
            model = torch.compile(raw)
        except Exception as e:
            log("torch.compile failed, running eager:", repr(e)[:200])
    if ddp:
        # broadcast_buffers=False: MoE load counters are per-rank; router biases stay identical
        # because every rank applies the same all-reduced update in after_step()
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[local], broadcast_buffers=False)

    log(f"arch={cfg['arch']} params={n_params/1e6:.1f}M (active/token {n_active/1e6:.1f}M) | world={world} "
        f"micro_bs={mbs} accum={accum} | {tokens_per_step/1e3:.0f}k tokens/step | "
        f"{total_steps} steps = {cfg['max_tokens']/1e9:.2f}B tokens")
    json.dump({**cfg, "n_params": n_params, "n_active_params": n_active, "total_steps": total_steps},
              open(f"{out_dir}/run_config.json", "w"), indent=1)
    csv_path = f"{out_dir}/train_log.csv"
    if master and not os.path.exists(csv_path):
        with open(csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["step", "tokens_M", "loss", "lr", "grad_norm", "scale",
                                    "tok_per_s", "sc_frac", "hours"])

    def save_latest(step):
        if master:
            atomic_save({"model": raw.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                         "step": step, "tokens_seen": tokens_seen, "best_val": best_val,
                         "train_hours": train_hours + (time.time() - t_start) / 3600, "cfg": cfg}, ck_path)

    # ---- training loop ---------------------------------------------------------------------
    model.train()
    t_start = t_log = t_ck = time.time()
    tok_since_log, n_acc = 0, 0
    loss_acc = torch.zeros((), device=device)          # accumulate on GPU: no per-micro-step sync
    stop = False
    for step in range(start_step, total_steps):
        frac = step / total_steps
        lr = lr_at(step, total_steps, cfg)
        for g in opt.param_groups:
            g["lr"] = lr
        w = mixture_weights(cfg["mixture"], frac, sampler.n_tokens, 1 - cfg["decay_frac"])

        for micro in range(accum):
            x, y, _ = sampler.batch(w, mbs, cfg["seed"], step, rank, micro)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            sync = nullcontext() if (not ddp or micro == accum - 1) else model.no_sync()
            with sync:                                  # all-reduce grads only on last micro-step
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = model(x, y) / accum
                scaler.scale(loss).backward()
            loss_acc += loss.detach(); tok_since_log += x.numel() * world
        n_acc += 1
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw.parameters(), cfg["grad_clip"]).item()
        scaler.step(opt); scaler.update()               # skips the step if grads overflowed
        opt.zero_grad(set_to_none=True)
        extra = raw.after_step() if hasattr(raw, "after_step") else {}   # MoE bias balancing
        tokens_seen += tokens_per_step

        # ---- logging -------------------------------------------------------------------------
        if master and ((step + 1) % cfg["log_every"] == 0 or step == start_step):
            dt = time.time() - t_log
            row = [step + 1, round(tokens_seen / 1e6, 1), round(loss_acc.item() / n_acc, 4), f"{lr:.2e}",
                   round(gnorm, 3), int(scaler.get_scale()), int(tok_since_log / dt),
                   round(w["supreme_court"], 2), round(train_hours + (time.time() - t_start) / 3600, 3)]
            with open(csv_path, "a", newline="") as f:
                csv.writer(f).writerow(row)
            log(f"step {row[0]:6d}/{total_steps} | {row[1]:7.1f}M tok | loss {row[2]:.4f} | lr {row[3]} | "
                f"gnorm {row[4]:.2f} | scale {row[5]} | {row[6]/1e3:.1f}k tok/s | SC w {row[7]} | {row[8]:.2f} h"
                + "".join(f" | {k} {v}" for k, v in extra.items()))
            t_log, tok_since_log, n_acc = time.time(), 0, 0
            loss_acc.zero_()

        # ---- evaluation + best checkpoint ------------------------------------------------------
        last = step + 1 == total_steps
        if (step + 1) % cfg["eval_every"] == 0 or last:
            if master:
                res = evaluate(raw, val_windows, tpw, device, mbs)
                sc = res["supreme_court"]["loss"]
                with open(f"{out_dir}/eval_log.jsonl", "a") as f:
                    f.write(json.dumps({"step": step + 1, "tokens_M": tokens_seen / 1e6, **res}) + "\n")
                log("  EVAL " + " | ".join(f"{s[:5]} loss {r['loss']:.3f} bpw {r['bits_per_word']:.3f}"
                                           for s, r in res.items()))
                if sc < best_val:
                    best_val = sc
                    atomic_save({"model": {k: v.half() for k, v in raw.state_dict().items()},
                                 "arch": cfg["arch"], "model_cfg": cfg["model"], "step": step + 1,
                                 "val": res}, f"{out_dir}/best.pt")
                    log(f"  new best SC val loss {sc:.4f} -> best.pt")
            if ddp:
                dist.barrier()

        # ---- periodic checkpoint + time limit (decisions all-reduced -> ranks stay in lockstep)
        elapsed_h = (time.time() - t_start) / 3600
        flags = torch.tensor([float(elapsed_h > cfg["time_limit_hours"]),
                              float((time.time() - t_ck) / 60 > cfg["ckpt_every_min"])], device=device)
        if ddp:
            dist.all_reduce(flags, op=dist.ReduceOp.MAX)
        stop, want_ck = flags[0].item() > 0, flags[1].item() > 0
        if last or stop or want_ck:
            save_latest(step + 1)
            t_ck = time.time()
            log(f"  checkpoint saved at step {step + 1}")
            if ddp:
                dist.barrier()
        if stop:
            log(f"Time limit reached ({elapsed_h:.2f} h) -> stopped cleanly at step {step + 1}; "
                f"re-run the same command to resume.")
            break

    if master and not stop:
        log(f"TRAINING COMPLETE: {tokens_seen/1e9:.2f}B tokens, best SC val loss {best_val:.4f}")
    if ddp:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/src/train.py


In [3]:
# ===== Cell 35: M2 self-tests + single-GPU throughput =====
import sys, time, importlib, gc, torch, torch.nn.functional as F
WORK = "/kaggle/working"
sys.path.insert(0, f"{WORK}/src")
import lm_common, model_mla_moe as mm, data
for m in (lm_common, mm, data):
    importlib.reload(m)
from lm_common import count_params
from data import ShardSampler, mixture_weights
dev = "cuda"

# [1] size rule + KV-cache footprint
m2 = mm.MLAMoEGPT(mm.MLAMoEConfig())
c = m2.cfg
print(f"[1] M2 total {count_params(m2)/1e6:.1f}M | active/token {m2.active_params()/1e6:.1f}M")
print(f"    KV cache values per token per layer: MLA {c.kv_latent_dim + c.qk_rope_dim} | M1 GQA {2*4*64} | full MHA {2*768}")
assert 50e6 <= count_params(m2) <= 300e6
del m2

# [2] MLA absorbed decoding must reproduce the full forward pass (fp32 tiny model, eval = no drops)
torch.manual_seed(0)
tc = mm.MLAMoEConfig(d_model=128, n_layers=3, n_heads=4, qk_nope_dim=16, qk_rope_dim=16, v_head_dim=32,
                     kv_latent_dim=64, n_routed_experts=8, top_k=2, expert_hidden=64,
                     dense_ffn_hidden=256, max_seq_len=256)
tiny = mm.MLAMoEGPT(tc).to(dev).float().eval()
x = torch.randint(16, 32000, (2, 40), device=dev)
with torch.no_grad():
    full = tiny(x)
    caches = tiny.make_caches(2, 40, dev, dtype=torch.float32)
    parts = [tiny(x[:, :30], caches=caches, pos=0)] + \
            [tiny(x[:, t:t + 1], caches=caches, pos=t) for t in range(30, 40)]
err = (torch.cat(parts, 1) - full).abs().max().item()
print(f"[2] MLA absorbed decode vs full forward: max |diff| = {err:.2e}")
assert err < 1e-3

# [3] sort/scatter expert dispatch must equal a naive per-token loop
moe = tiny.blocks[1].ffn
hx = torch.randn(2, 40, 128, device=dev)
with torch.no_grad():
    fast = moe(hx)
    xf = hx.reshape(-1, 128)
    aff = torch.sigmoid(xf @ moe.router.weight.t())
    _, idx = torch.topk(aff + moe.router_bias, moe.k, -1)
    g = aff.gather(1, idx); g = g / g.sum(-1, keepdim=True)
    slow = torch.zeros_like(xf)
    for n in range(xf.size(0)):
        for j in range(moe.k):
            e = idx[n, j]
            slow[n] += g[n, j] * ((F.silu(xf[n] @ moe.w1[e]) * (xf[n] @ moe.w3[e])) @ moe.w2[e])
    slow = slow.view_as(hx) + moe.shared(hx)
err3 = (fast - slow).abs().max().item()
print(f"[3] MoE batched dispatch vs naive loop: max |diff| = {err3:.2e}")
assert err3 < 1e-4
moe.train(); _ = moe(hx)                                    # training mode: capacity + counters
print(f"    train-mode counters: load sum {moe.load_accum.sum().item():.0f} (= N*k = {80*moe.k}), "
      f"dropped {moe.drop_accum[0].item():.0f}/{moe.drop_accum[1].item():.0f}")
assert moe.load_accum.sum().item() == 80 * moe.k
del tiny, moe; gc.collect(); torch.cuda.empty_cache()

# [4] real M2: compiled fp16 training steps on real data, balancing hook, speed + memory
m2 = mm.MLAMoEGPT(mm.MLAMoEConfig()).to(dev)
opt = torch.optim.AdamW(m2.parameters(), lr=3e-4, fused=True)
scaler = torch.amp.GradScaler("cuda")
cm = torch.compile(m2)
sampler = ShardSampler(f"{WORK}/data", "train", 1024)
w = mixture_weights("curriculum", 0.0, sampler.n_tokens)
losses, torch_ok = [], True
torch.cuda.reset_peak_memory_stats()
for i in range(13):
    if i == 3:
        torch.cuda.synchronize(); t0 = time.time()
    xb, yb, _ = sampler.batch(w, 8, 1337, i, 0, 0)
    with torch.autocast("cuda", dtype=torch.float16):
        loss = cm(xb.to(dev), yb.to(dev))
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    info = m2.after_step()
    losses.append(loss.item())
torch.cuda.synchronize()
tps = 8 * 1024 * 10 / (time.time() - t0)
peak = torch.cuda.max_memory_allocated() / 1e9
bias = torch.stack([m.router_bias for m in m2.moe_layers])
print(f"[4] compiled fp16 training: loss {losses[0]:.2f} -> {losses[-1]:.2f} | {tps:,.0f} tok/s on 1 GPU | "
      f"peak mem {peak:.1f} GB | balance {info}")
print(f"    router bias range after 13 steps: [{bias.min().item():.3f}, {bias.max().item():.3f}]")
print(f"    projected 700M tokens on 2xT4: {7e8 / (tps * 1.85) / 3600:.1f} h (M1 took 4.9 h)")
assert 9.0 < losses[0] < 11.5 and losses[-1] < losses[0]
del m2, cm, opt; gc.collect(); torch.cuda.empty_cache()
print("\nALL M2 SELF-TESTS PASSED")

[1] M2 total 270.4M | active/token 101.7M
    KV cache values per token per layer: MLA 288 | M1 GQA 512 | full MHA 1536
[2] MLA absorbed decode vs full forward: max |diff| = 9.54e-07
[3] MoE batched dispatch vs naive loop: max |diff| = 2.33e-09
    train-mode counters: load sum 160 (= N*k = 160), dropped 5/160


W0929 02:08:54.089000 58 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


[4] compiled fp16 training: loss 10.51 -> 7.36 | 17,380 tok/s on 1 GPU | peak mem 8.7 GB | balance {'moe_max/mean': 5.316, 'drop%': 71.96}
    router bias range after 13 steps: [-0.013, 0.013]
    projected 700M tokens on 2xT4: 6.0 h (M1 took 4.9 h)

ALL M2 SELF-TESTS PASSED


In [4]:
# ===== Cell 36: launch M2 training in the BACKGROUND (notebook stays usable) =====
import json, os, subprocess, gc, torch
gc.collect(); torch.cuda.empty_cache()           # free GPU memory held by the self-test cell

WORK = "/kaggle/working"
RUN = "m2_mla_moe"
run_cfg = {
    "arch": "mla_moe", "model": {},                  # MLAMoEConfig defaults: 270M total / 102M active
    "run_name": RUN, "out_dir": f"{WORK}/checkpoints/{RUN}", "data_dir": f"{WORK}/data",
    # ---- identical budget + recipe to M1 (controlled comparison) -------------------------
    "seq_len": 1024, "micro_bs": 8, "global_bs_seqs": 256, "max_tokens": 700_000_000,
    "lr": 6e-4, "min_lr_frac": 0.1, "warmup_frac": 0.03, "schedule": "wsd", "decay_frac": 0.2,
    "weight_decay": 0.1, "betas": [0.9, 0.95], "grad_clip": 1.0, "compile": True, "seed": 1337,
    "mixture": "curriculum",
    "log_every": 20, "eval_every": 200, "val_windows": 64,
    "ckpt_every_min": 15, "time_limit_hours": 11.0,
}
os.makedirs(run_cfg["out_dir"], exist_ok=True)
cfg_path = f"{run_cfg['out_dir']}/config.json"
json.dump(run_cfg, open(cfg_path, "w"), indent=1)
LOG = f"{run_cfg['out_dir']}/stdout.log"

# nohup + '&': training runs as its own process; output goes to LOG instead of this cell
cmd = (f"cd {WORK} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True "
       f"nohup torchrun --standalone --nproc_per_node=2 {WORK}/src/train.py --config {cfg_path} "
       f"> {LOG} 2>&1 &")
subprocess.run(cmd, shell=True)
print("M2 launched in background. Log file:", LOG)

M2 launched in background. Log file: /kaggle/working/checkpoints/m2_mla_moe/stdout.log


In [1]:
# ===== Quick resume: M2 after a session restart (background, appends to the same log) =====
import os, subprocess
WORK = "/kaggle/working"
RUN_DIR = f"{WORK}/checkpoints/m2_mla_moe"
ck = f"{RUN_DIR}/latest.pt"

# ---- 1. confirm the checkpoint survived --------------------------------------------------
if os.path.exists(ck):
    print(f"latest.pt found ({os.path.getsize(ck)/1e9:.2f} GB) -> training will RESUME")
else:
    raise SystemExit("No latest.pt found -- do NOT relaunch; tell Claude (training would restart from 0)")

# ---- 2. relaunch with the SAME saved config; '>>' appends so the earlier log is kept ------
cmd = (f"cd {WORK} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True "
       f"nohup torchrun --standalone --nproc_per_node=2 {WORK}/src/train.py "
       f"--config {RUN_DIR}/config.json >> {RUN_DIR}/stdout.log 2>&1 &")
subprocess.run(cmd, shell=True)
print("M2 relaunched in background.")

latest.pt found (3.25 GB) -> training will RESUME
M2 relaunched in background.


In [14]:
# ===== Cell 37: show the latest M2 progress =====
!tail -n 8 /kaggle/working/checkpoints/m2_mla_moe/stdout.log
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used,memory.total --format=csv

  new best SC val loss 2.2959 -> best.pt
step   2620/2670 |   686.8M tok | loss 2.1505 | lr 7.21e-05 | gnorm 0.15 | scale 32768 | 28.0k tok/s | SC w 0.75 | 6.10 h | moe_max/mean 1.066 | drop% 1.01
step   2640/2670 |   692.1M tok | loss 2.1433 | lr 6.45e-05 | gnorm 0.14 | scale 32768 | 30.5k tok/s | SC w 0.75 | 6.15 h | moe_max/mean 1.064 | drop% 1.08
step   2660/2670 |   697.3M tok | loss 2.1435 | lr 6.06e-05 | gnorm 0.14 | scale 32768 | 30.5k tok/s | SC w 0.75 | 6.20 h | moe_max/mean 1.061 | drop% 0.82
  EVAL acts loss 2.049 bpw 3.806 | supre loss 2.288 bpw 4.066 | tribu loss 1.963 bpw 4.135 | high_ loss 1.754 bpw 3.368
  new best SC val loss 2.2884 -> best.pt
  checkpoint saved at step 2670
TRAINING COMPLETE: 0.70B tokens, best SC val loss 2.2884
index, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
0, 0 %, 3855 MiB, 15360 MiB
1, 0 %, 3509 MiB, 15360 MiB


In [13]:
# ===== Cell 38x: training summary across finished runs (report table) =====
import json, os, pandas as pd
WORK = "/kaggle/working"
rows = []
for run in ["m1_dense_100m", "m2_mla_moe"]:
    d = f"{WORK}/checkpoints/{run}"
    cfg = json.load(open(f"{d}/run_config.json"))
    ev = [json.loads(l) for l in open(f"{d}/eval_log.jsonl")][-1]          # final evaluation
    log = pd.read_csv(f"{d}/train_log.csv")
    rows.append({"run": run, "total params (M)": round(cfg["n_params"] / 1e6, 1),
                 "active params (M)": round(cfg.get("n_active_params", cfg["n_params"]) / 1e6, 1),
                 "tokens (M)": round(ev["tokens_M"]), "train hours": round(log["hours"].max(), 2),
                 "median tok/s": int(log["tok_per_s"].median()),
                 **{f"{s[:5]} bpw": round(ev[s]["bits_per_word"], 3) for s in ["supreme_court", "acts", "tribunals", "high_courts"]}})
df = pd.DataFrame(rows).set_index("run")
display(df)
df.to_csv(f"{WORK}/stats/training_summary.csv")

,total params (M),active params (M),tokens (M),train hours,median tok/s,supre bpw,acts bpw,tribu bpw,high_ bpw
run,,,,,,,,,
m1_dense_100m,100.1,100.1,700,4.92,39058,4.154,3.914,4.280,3.497
m2_mla_moe,270.4,101.7,700,6.20,31459,4.066,3.806,4.135,3.368


## Section 8: Copy-aware + MBR decoding (M1)
Motivation: the trivial copy-last-128 baseline scored **0.8412** officially, beating M1 (0.8364). Legal continuations reuse prompt content heavily.
- **Prompt copy bonus:** add β to the logits of every token that occurs in the prompt. This is a fixed decoding-time copy prior; M4 will *learn* when to copy instead.
- **MBR (Minimum Bayes Risk):** sample n candidates, then pick the one with the highest average n-gram F1 against the others, i.e. the most "central" continuation. Uses no pretrained model at inference.
- **Shared-prefix KV caching:** each prompt is prefilled once and its cache copied n times, so n samples cost n × decoding but only 1 × prefill.
- **128-word guarantee:** short candidates are extended automatically (fixes last night's 3 short rows).

In [14]:
%%writefile /kaggle/working/src/generation.py
# generation.py -- KV-cached decoding with copy bonus, n-sample shared-prefix caching, penalties
from __future__ import annotations
import dataclasses
import torch


def _expand_cache(c, n):
    # repeat every batch row of a KV cache n times (shared-prompt caching for MBR sampling)
    return type(c)(**{f.name: getattr(c, f.name).repeat_interleave(n, 0) for f in dataclasses.fields(c)})


def _apply_sampling(logits, temperature, top_k, top_p):
    # greedy if temperature <= 0, else temperature + optional top-k / nucleus sampling
    if temperature <= 0:
        return logits.argmax(-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1).values[..., -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p and top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = sl.softmax(-1).cumsum(-1)
        remove = cp - sl.softmax(-1) > top_p                          # always keep the top token
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(-1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1)

Overwriting /kaggle/working/src/generation.py


In [15]:
%%writefile -a /kaggle/working/src/generation.py


@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=0.0, top_k=0, top_p=1.0,
             repetition_penalty=1.0, banned_ids=(0,), no_repeat_ngram=0,
             first_allowed=None, ngram_window=384, prompt_bonus=0.0, num_return=1):
    # idx (B,T) equal-length prompts -> (B*num_return, max_new_tokens), rows grouped per prompt.
    # repetition_penalty: generated tokens only | prompt_bonus: +beta on prompt tokens (copy prior)
    # num_return: prompt prefilled ONCE, KV cache copied n times (shared-prefix caching)
    model.eval()
    B, T = idx.shape
    cdt = torch.float16 if torch.is_autocast_enabled() else next(model.parameters()).dtype
    caches = model.make_caches(B, T + max_new_tokens, idx.device, dtype=cdt)
    logits = model(idx, caches=caches, pos=0)[:, -1].float()          # prefill once per prompt
    in_prompt = torch.zeros(B, logits.size(-1), dtype=torch.bool, device=idx.device).scatter_(1, idx, True)
    in_prompt[:, :16] = False                                         # never boost special tokens
    if num_return > 1:
        caches = [_expand_cache(c, num_return) for c in caches]
        logits = logits.repeat_interleave(num_return, 0)
        in_prompt = in_prompt.repeat_interleave(num_return, 0)
        idx = idx.repeat_interleave(num_return, 0)
        B = B * num_return
    out, seen = [], idx.clone()
    for i in range(max_new_tokens):
        if repetition_penalty != 1.0 and i > 0:
            gen = seen[:, T:]
            g = logits.gather(1, gen)
            logits.scatter_(1, gen, torch.where(g > 0, g / repetition_penalty, g * repetition_penalty))
        if prompt_bonus:
            logits = logits + prompt_bonus * in_prompt
        if banned_ids:
            logits[:, list(banned_ids)] = float("-inf")
        if i == 0 and first_allowed is not None:                      # start a new word
            logits[:, ~first_allowed] = float("-inf")
        if no_repeat_ngram and seen.size(1) >= no_repeat_ngram:
            n = no_repeat_ngram
            for b in range(B):
                s = seen[b, -ngram_window:].tolist(); prefix = tuple(s[-(n - 1):])
                bad = [s[j + n - 1] for j in range(len(s) - n + 1) if tuple(s[j:j + n - 1]) == prefix]
                if bad:
                    logits[b, bad] = float("-inf")
        nxt = _apply_sampling(logits, temperature, top_k, top_p)
        out.append(nxt)
        seen = torch.cat([seen, nxt], 1)
        if i + 1 < max_new_tokens:
            logits = model(nxt, caches=caches, pos=T + i)[:, -1].float()
    return torch.cat(out, 1)

Appending to /kaggle/working/src/generation.py


In [16]:
%%writefile /kaggle/working/src/infer.py
# infer.py -- checkpoint -> 128-word continuations (optionally n candidates + MBR) -> submission
from __future__ import annotations
import json, os, sys, zipfile
from collections import Counter
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common                                   # noqa: E402
from generation import generate                 # noqa: E402

N_WORDS = 128


def load_model(ckpt_path, device="cuda"):
    # best.pt (fp16 weights + arch/model_cfg) -> fp32 module; inference runs under fp16 autocast
    from train import build_model
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model = build_model(ck["arch"], ck["model_cfg"])
    model.load_state_dict({k: v.float() for k, v in ck["model"].items()})
    return model.to(device).eval(), ck


def load_tokenizer(path):
    from tokenizers import Tokenizer
    return Tokenizer.from_file(path)


def space_start_mask(tk, vocab_size, device):
    # bool (V,): token text starts with a space -> allowed as the first generated token
    m = torch.zeros(vocab_size, dtype=torch.bool)
    for i in range(16, vocab_size):
        m[i] = tk.decode([i]).startswith(" ")
    return m.to(device)


def _ensure_128(model, tk, prompt_ids, text, max_prompt, max_new_tokens, device, gen_kw, rounds=3):
    # extend a short continuation by generating more; last resort: pad from the prompt's tail
    for _ in range(rounds):
        if len(text.split()) >= N_WORDS:
            return text
        ctx = (prompt_ids + tk.encode(" " + text, add_special_tokens=False).ids)[-max_prompt:]
        with torch.autocast("cuda", dtype=torch.float16):
            y = generate(model, torch.tensor([ctx], device=device), max_new_tokens, **gen_kw)
        text = common.normalise(text + " " + tk.decode(y[0].tolist()))
    w = text.split()
    if len(w) < N_WORDS:
        w = w + tk.decode(prompt_ids).split()[-(N_WORDS - len(w)):]
    return " ".join(w)

Overwriting /kaggle/working/src/infer.py


In [17]:
%%writefile -a /kaggle/working/src/infer.py


@torch.no_grad()
def continue_prompts(model, tk, prompts, device="cuda", batch_size=32, max_ctx=1024,
                     max_new_tokens=200, first_mask=None, num_return=1, **gen_kw):
    # {id: text} -> {id: text} (num_return=1) or {id: [n candidates]}; every text = 128 words
    ids = {k: tk.encode(v, add_special_tokens=False).ids for k, v in prompts.items()}
    max_prompt = max_ctx - max_new_tokens
    ids = {k: v[-max_prompt:] for k, v in ids.items()}
    order = sorted(ids, key=lambda k: len(ids[k]))              # similar lengths share a batch
    bs = max(1, batch_size // num_return)
    out, n_trunc = {}, 0
    for b in range(0, len(order), bs):
        keys = order[b:b + bs]
        L = min(len(ids[k]) for k in keys)                       # cut batch to shortest (left)
        n_trunc += sum(len(ids[k]) > L for k in keys)
        x = torch.tensor([ids[k][-L:] for k in keys], device=device)
        with torch.autocast("cuda", dtype=torch.float16):
            y = generate(model, x, max_new_tokens, first_allowed=first_mask, num_return=num_return, **gen_kw)
        rows = [common.normalise(tk.decode(r)) for r in y.tolist()]
        for j, k in enumerate(keys):
            out[k] = rows[j * num_return:(j + 1) * num_return]
    n_short = 0
    for k in out:
        for j, t in enumerate(out[k]):
            if len(t.split()) < N_WORDS:
                n_short += 1
                out[k][j] = _ensure_128(model, tk, ids[k], t, max_prompt, max_new_tokens, device, gen_kw)
    final = {k: [" ".join(t.split()[:N_WORDS]) for t in v] for k, v in out.items()}
    stats = {"n": len(final), "left_truncated_in_batch": n_trunc, "short_rows_extended": n_short}
    return ({k: v[0] for k, v in final.items()} if num_return == 1 else final), stats


def ngram_f1(a, b, orders=(1, 2)):
    # symmetric unigram+bigram F1: cheap, model-free MBR utility
    wa, wb, f = a.split(), b.split(), 0.0
    for n in orders:
        A, B = Counter(zip(*[wa[i:] for i in range(n)])), Counter(zip(*[wb[i:] for i in range(n)]))
        ov = sum((A & B).values())
        p, r = ov / max(1, sum(A.values())), ov / max(1, sum(B.values()))
        f += 2 * p * r / max(1e-9, p + r)
    return f / len(orders)


def mbr_select(cands):
    # Minimum Bayes Risk: candidate with the highest mean utility against all the others
    if len(cands) == 1:
        return cands[0], 0
    sc = [sum(ngram_f1(c, o) for j, o in enumerate(cands) if j != i) / (len(cands) - 1)
          for i, c in enumerate(cands)]
    best = max(range(len(cands)), key=sc.__getitem__)
    return cands[best], best


def write_submission(preds, test_ids, out_dir, name):
    # strict checks, then prediction.json at the ROOT of <name>.zip
    test_ids = set(map(str, test_ids))
    assert set(preds) == test_ids, f"ID mismatch: missing {len(test_ids - set(preds))}, extra {len(set(preds) - test_ids)}"
    bad = [k for k, v in preds.items() if len(v.split()) != N_WORDS]
    assert not bad, f"{len(bad)} predictions are not exactly {N_WORDS} words"
    os.makedirs(out_dir, exist_ok=True)
    js = {k: {"text": preds[k]} for k in sorted(preds, key=lambda s: int(s) if s.isdigit() else s)}
    pj = f"{out_dir}/prediction.json"
    json.dump(js, open(pj, "w"), ensure_ascii=True)
    zp = f"{out_dir}/{name}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    return zp

Appending to /kaggle/working/src/infer.py


In [11]:
# ===== Cell 38e: verify generation.py and infer.py are complete =====
import py_compile
for f, must in [("generation.py", ["def _expand_cache", "def _apply_sampling", "def generate", "prompt_bonus"]),
                ("infer.py", ["def load_model", "def _ensure_128", "def continue_prompts", "def mbr_select", "def write_submission"])]:
    p = f"/kaggle/working/src/{f}"
    py_compile.compile(p, doraise=True)
    src = open(p).read()
    print(f, "compiles |", {m: (m in src) for m in must})

generation.py compiles | {'def _expand_cache': True, 'def _apply_sampling': True, 'def generate': True, 'prompt_bonus': True}
infer.py compiles | {'def load_model': True, 'def _ensure_128': True, 'def continue_prompts': True, 'def mbr_select': True, 'def write_submission': True}


In [2]:
# ===== Cell 39: copy bonus + MBR decoding study for M1 (fixed 350-prompt validation subset) =====
import sys, json, time, importlib, torch, pandas as pd
WORK = "/kaggle/working"; DATA = f"{WORK}/data"; SUB = f"{WORK}/submissions"
sys.path.insert(0, f"{WORK}/src")
import generation, infer, eval_local
for m in (generation, infer, eval_local):
    importlib.reload(m)

# ---- use the GPU with the most free memory (M2 is training on both) --------------------------
free = [torch.cuda.mem_get_info(i)[0] / 1e9 for i in range(torch.cuda.device_count())]
gpu = max(range(len(free)), key=free.__getitem__)
print("free GPU memory (GB):", [round(f, 1) for f in free], "-> using GPU", gpu)
assert free[gpu] > 4.0, "not enough free GPU memory next to M2 -- run this after M2 finishes"
torch.cuda.set_device(gpu)

# ---- data: same fixed subset as Cell 30 (200 SC + 50 per other source) -------------------------
tk = infer.load_tokenizer(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
VP, VR, VM = (json.load(open(f"{DATA}/val_{x}.json")) for x in ("prompts", "refs", "meta"))
DEV = [k for k in VP if VM[k]["source"] == "supreme_court"][:200]
for s in ("acts", "tribunals", "high_courts"):
    DEV += [k for k in VP if VM[k]["source"] == s][:50]
GRP = {k: ("SC" if VM[k]["source"] == "supreme_court" else "other") for k in DEV}
dev_prompts = {k: VP[k]["text"] for k in DEV}

model, ck = infer.load_model(f"{WORK}/checkpoints/m1_dense_100m/best.pt")
first_mask = infer.space_start_mask(tk, 32000, "cuda")

def copy_rate(preds):
    # share of generated word-bigrams that also occur in the prompt (how much the output copies)
    r = []
    for k, t in preds.items():
        pw, gw = VP[k]["text"].split(), t.split()
        P = set(zip(pw, pw[1:])); G = list(zip(gw, gw[1:]))
        r.append(sum(g in P for g in G) / max(1, len(G)))
    return sum(r) / len(r)

def evaluate(name, preds, t0):
    r = eval_local.score(preds, VR, by_group=GRP)
    r.update(rep3=eval_local.repetition_rate(preds.values()), copy=copy_rate(preds), sec=round(time.time() - t0))
    print(f"{name:24s} BERT {r['bertscore_f1']:.4f} (SC {r['bertscore_f1[SC]']:.4f}) | ROUGE-L {r['rougeL_f1']:.4f} "
          f"| BLEU {r['bleu4']:.2f} | copy {r['copy']:.2f} | rep3 {r['rep3']:.3f} | {r['sec']}s", flush=True)
    return r

results, greedy_out = {}, {}
# ---- 1) greedy + repetition penalty with increasing prompt copy bonus ---------------------------
for beta in (0.0, 0.5, 1.0, 2.0):
    t0 = time.time(); torch.manual_seed(1337)
    p, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                                  temperature=0, repetition_penalty=1.2, prompt_bonus=beta)
    greedy_out[beta] = p
    results[f"greedy_rep1.2_copy{beta}"] = evaluate(f"greedy_rep1.2_copy{beta}", p, t0)

# ---- 2) MBR over 8 samples (+ the matching greedy output as a 9th candidate) -------------------
for beta in (0.0, 1.0):
    t0 = time.time(); torch.manual_seed(1337)
    cands, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                                      num_return=8, temperature=0.7, top_p=0.9,
                                      repetition_penalty=1.1, prompt_bonus=beta)
    picks = {k: infer.mbr_select(cands[k] + [greedy_out[beta][k]])[0] for k in DEV}
    results[f"mbr9_copy{beta}"] = evaluate(f"mbr9_copy{beta}", picks, t0)

# ---- reference rows + table ----------------------------------------------------------------
results["copy_last_128 (baseline)"] = eval_local.score({k: " ".join(VP[k]["text"].split()[-128:]) for k in DEV}, VR, by_group=GRP)
df = pd.DataFrame(results).T.sort_values("bertscore_f1[SC]", ascending=False)
display(df[["bertscore_f1", "bertscore_f1[SC]", "rougeL_f1", "bleu4", "copy", "rep3", "sec"]].round(4))
json.dump(results, open(f"{WORK}/stats/decoding_m1_v2.json", "w"), indent=1)

free GPU memory (GB): [15.5, 15.5] -> using GPU 0


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (19 outputs left out)

mbr9_copy1.0             BERT 0.8495 (SC 0.8462) | ROUGE-L 0.2599 | BLEU 14.40 | copy 0.47 | rep3 0.086 | 1355s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,bertscore_f1,bertscore_f1[SC],rougeL_f1,bleu4,copy,rep3,sec
mbr9_copy1.0,0.8495,0.8462,0.2599,14.3971,0.4694,0.0862,1355.0
mbr9_copy0.0,0.8486,0.8448,0.2521,13.7172,0.3419,0.0513,1351.0
greedy_rep1.2_copy1.0,0.8453,0.8432,0.2429,13.2072,0.4137,0.0852,213.0
greedy_rep1.2_copy2.0,0.8447,0.8423,0.2444,12.4861,0.4949,0.1150,188.0
greedy_rep1.2_copy0.5,0.8437,0.8419,0.2392,12.4211,0.3636,0.0690,235.0
copy_last_128 (baseline),0.8430,0.8412,0.2202,11.1995,NaN,NaN,NaN
greedy_rep1.2_copy0.0,0.8438,0.8405,0.2348,12.4406,0.2966,0.0525,214.0


In [3]:
# ===== Cell 39b: test-set predictions for one model + one decoding config -> Codabench zip =====
# ---- settings (from the Cell 39 table: best SC row that is not the baseline) ----------------
RUN_NAME = "m1_dense_100m"                  # later: "m2_mla_moe"
TAG      = "mbr9_copy1.0"                   # label used in the file name
USE_MBR  = True                             # winning row starts with "mbr9"
DEC_KW   = dict(prompt_bonus=1.0)           # MBR branch uses greedy rep1.2 + 8 samples (T0.7, p0.9, rep1.1)

import gc, json, time, torch
assert all(v in globals() for v in ("tk", "first_mask", "infer")), "run Cell 39 first (kernel was restarted)"

# ---- free the previous model (if any) and load the chosen checkpoint ------------------------
if "model" in globals():
    del model; gc.collect(); torch.cuda.empty_cache()
test = json.load(open(f"{DATA}/Phase_1_Pretraining/Test/test.json"))
test_prompts = {k: v["text"] for k, v in test.items()}
model, ck = infer.load_model(f"{WORK}/checkpoints/{RUN_NAME}/best.pt")
print(f"{RUN_NAME}: step {ck['step']} | {len(test_prompts)} test prompts")
t0 = time.time(); torch.manual_seed(1337)

# ---- generate ---------------------------------------------------------------------------------
if not USE_MBR:
    # single decoding pass with the given settings
    preds, st = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask, **DEC_KW)
else:
    beta = DEC_KW.get("prompt_bonus", 0.0)
    # candidate 9: greedy + repetition penalty 1.2 + copy bonus
    greedy, _ = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask,
                                       temperature=0, repetition_penalty=1.2, prompt_bonus=beta)
    print(f"greedy done ({(time.time()-t0)/60:.1f} min), sampling 8 candidates per prompt ...", flush=True)
    # candidates 1-8: nucleus samples sharing one prefilled prompt cache
    cands, st = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask,
                                       num_return=8, temperature=0.7, top_p=0.9,
                                       repetition_penalty=1.1, prompt_bonus=beta)
    # MBR: keep the candidate that agrees most (n-gram F1) with the other 8
    preds = {k: infer.mbr_select(cands[k] + [greedy[k]])[0] for k in test_prompts}
print("generation stats:", st, f"| {(time.time()-t0)/60:.1f} min")

# ---- write + verify the submission zip (exactly 128 words, all ids, prediction.json at root) --
name = f"{RUN_NAME}_{TAG}"
zp = infer.write_submission(preds, test.keys(), f"{SUB}/{name}", name)
json.dump({"model": RUN_NAME, "ckpt_step": ck["step"], "mbr": USE_MBR, "decoding": DEC_KW},
          open(f"{SUB}/{name}/meta.json", "w"), indent=1)            # record of what was submitted
print("submission zip:", zp)
print("example id 1:", preds[next(iter(preds))][:300])

m1_dense_100m: step 2670 | 1000 test prompts
greedy done (6.2 min), sampling 8 candidates per prompt ...
generation stats: {'n': 1000, 'left_truncated_in_batch': 87, 'short_rows_extended': 226} | 52.1 min
submission zip: /kaggle/working/submissions/m1_dense_100m_mbr9_copy1.0/m1_dense_100m_mbr9_copy1.0.zip
example id 1: not a case where the petitioner has been deprived of his right to have the acquisition proceedings quashed. The writ petition is, therefore, liable to be dismissed on this ground alone." 11. In [State of Maharashtra Vs. L. Krishnan & Ors.] (1996) 6 SCC 445, this Court observed as under:- "It is true


In [21]:
# ===== Cell 39c: M2 decoding check -- only the two best settings from the M1 study =====
import gc, json, time, torch, pandas as pd
assert all(v in globals() for v in ("tk", "first_mask", "evaluate", "dev_prompts", "results")), \
    "run Cell 39 first (kernel was restarted)"

# ---- swap M1 for M2 on the GPU ----------------------------------------------------------------
if "model" in globals():
    del model; gc.collect(); torch.cuda.empty_cache()
model, ck = infer.load_model(f"{WORK}/checkpoints/m2_mla_moe/best.pt")
res2 = {}

# ---- 1) greedy + repetition penalty 1.2 + copy bonus 1.0 (also MBR's 9th candidate) ----------
t0 = time.time(); torch.manual_seed(1337)
g2, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                               temperature=0, repetition_penalty=1.2, prompt_bonus=1.0)
res2["m2_greedy_rep1.2_copy1.0"] = evaluate("m2_greedy_rep1.2_copy1.0", g2, t0)

# ---- 2) MBR over 8 samples + the greedy output, copy bonus 1.0 --------------------------------
t0 = time.time(); torch.manual_seed(1337)
c2, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                               num_return=8, temperature=0.7, top_p=0.9,
                               repetition_penalty=1.1, prompt_bonus=1.0)
picks2 = {k: infer.mbr_select(c2[k] + [g2[k]])[0] for k in DEV}
res2["m2_mbr9_copy1.0"] = evaluate("m2_mbr9_copy1.0", picks2, t0)

# ---- compare with M1's best two rows (same 350 prompts), then save ---------------------------
both = {**{f"m1_{k}": results[k] for k in ("mbr9_copy1.0", "greedy_rep1.2_copy1.0")}, **res2}
display(pd.DataFrame(both).T[["bertscore_f1", "bertscore_f1[SC]", "rougeL_f1", "bleu4",
                              "copy", "rep3", "sec"]].round(4))
json.dump(res2, open(f"{WORK}/stats/decoding_m2_v2.json", "w"), indent=1)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


m2_greedy_rep1.2_copy1.0 BERT 0.8469 (SC 0.8433) | ROUGE-L 0.2467 | BLEU 13.67 | copy 0.42 | rep3 0.081 | 608s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


m2_mbr9_copy1.0          BERT 0.8499 (SC 0.8460) | ROUGE-L 0.2597 | BLEU 14.56 | copy 0.45 | rep3 0.081 | 4107s


,bertscore_f1,bertscore_f1[SC],rougeL_f1,bleu4,copy,rep3,sec
m1_mbr9_copy1.0,0.8495,0.8462,0.2599,14.3971,0.4694,0.0862,1355.0
m1_greedy_rep1.2_copy1.0,0.8453,0.8432,0.2429,13.2072,0.4137,0.0852,213.0
m2_greedy_rep1.2_copy1.0,0.8469,0.8433,0.2467,13.6700,0.4200,0.0812,608.0
m2_mbr9_copy1.0,0.8499,0.8460,0.2597,14.5642,0.4520,0.0806,4107.0


Section 8b markdown
Cell 39b  – M1 test submission (MBR-9 + copy 1.0)   ← leave as it is (record of submission #2)
Cell 39c  – M2 decoding check
Cell 39d  – zip comparison (optional; delete if you like)
Cell 39e  – M2 test submission                        ← NEW, paste here
Section 9b markdown
Cell 42, Cell 43, Cell 37 (M3) …

In [23]:
# ===== Cell 39e: M2 test-set predictions (MBR-9 + copy bonus 1.0) -> Codabench zip =====
# Same decoding as M1's 0.8443 submission, so the two architectures are compared like-for-like.
RUN_NAME = "m2_mla_moe"
TAG      = "mbr9_copy1.0"
BETA     = 1.0                                   # prompt copy bonus (best in Cells 39 / 39c)

import gc, json, time, torch
assert all(v in globals() for v in ("tk", "first_mask", "infer", "DATA", "SUB", "WORK")), \
    "kernel was restarted -- run Cell 26, restart, then Cell 39 without its two for-loops"

# ---- free whatever model is on the GPU, load M2's best checkpoint ------------------------------
if "model" in globals():
    del model; gc.collect(); torch.cuda.empty_cache()
test = json.load(open(f"{DATA}/Phase_1_Pretraining/Test/test.json"))
test_prompts = {k: v["text"] for k, v in test.items()}
model, ck = infer.load_model(f"{WORK}/checkpoints/{RUN_NAME}/best.pt")
print(f"{RUN_NAME}: step {ck['step']} | {len(test_prompts)} test prompts", flush=True)
t0 = time.time(); torch.manual_seed(1337)

# ---- candidate 9: greedy + repetition penalty 1.2 + copy bonus --------------------------------
greedy, _ = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask,
                                   temperature=0, repetition_penalty=1.2, prompt_bonus=BETA)
print(f"greedy done ({(time.time()-t0)/60:.1f} min), sampling 8 candidates per prompt ...", flush=True)

# ---- candidates 1-8: nucleus samples sharing one prefilled prompt cache ------------------------
cands, st = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask,
                                   num_return=8, temperature=0.7, top_p=0.9,
                                   repetition_penalty=1.1, prompt_bonus=BETA)

# ---- MBR: keep the candidate that agrees most (n-gram F1) with the other 8 ---------------------
preds = {k: infer.mbr_select(cands[k] + [greedy[k]])[0] for k in test_prompts}
print("generation stats:", st, f"| {(time.time()-t0)/60:.1f} min")

# ---- write + verify the zip (1000 ids, exactly 128 words, prediction.json at root) ------------
name = f"{RUN_NAME}_{TAG}"
zp = infer.write_submission(preds, test.keys(), f"{SUB}/{name}", name)
json.dump({"model": RUN_NAME, "ckpt_step": ck["step"], "mbr": True,
           "decoding": {"prompt_bonus": BETA, "greedy_rep": 1.2, "samples": 8, "T": 0.7, "top_p": 0.9, "sample_rep": 1.1}},
          open(f"{SUB}/{name}/meta.json", "w"), indent=1)          # record of what was submitted
print("submission zip:", zp)
print("example:", preds[next(iter(preds))][:300])

m2_mla_moe: step 2670 | 1000 test prompts
greedy done (20.6 min), sampling 8 candidates per prompt ...
generation stats: {'n': 1000, 'left_truncated_in_batch': 87, 'short_rows_extended': 220} | 150.0 min
submission zip: /kaggle/working/submissions/m2_mla_moe_mbr9_copy1.0/m2_mla_moe_mbr9_copy1.0.zip
example: a well-settled principle that delay in challenging the acquisition proceedings is not a ground for quashing the notification. It is true that the writ petition can be dismissed on the ground of laches only, but there is no indication in the Constitution Bench decision as to when and under what circu


In [4]:
# ===== Cell 39s: setup shared by Cells 39b/39c/39e/43b/43c (run once after any kernel restart) =====
import sys, json, time, importlib, gc, os, torch, pandas as pd
WORK = "/kaggle/working"; DATA = f"{WORK}/data"; SUB = f"{WORK}/submissions"
sys.path.insert(0, f"{WORK}/src")
import generation, infer
for m in (generation, infer):
    importlib.reload(m)

# ---- use the GPU with the most free memory (training may be running on both) ----------------
free = [torch.cuda.mem_get_info(i)[0] / 1e9 for i in range(torch.cuda.device_count())]
gpu = max(range(len(free)), key=free.__getitem__); torch.cuda.set_device(gpu)
print("free GPU memory (GB):", [round(f, 1) for f in free], "-> using GPU", gpu)

# ---- tokenizer, first-token mask, fixed 350-prompt validation subset ------------------------
tk = infer.load_tokenizer(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
first_mask = infer.space_start_mask(tk, 32000, "cuda")
VP, VR, VM = (json.load(open(f"{DATA}/val_{x}.json")) for x in ("prompts", "refs", "meta"))
DEV = [k for k in VP if VM[k]["source"] == "supreme_court"][:200]
for s in ("acts", "tribunals", "high_courts"):
    DEV += [k for k in VP if VM[k]["source"] == s][:50]
GRP = {k: ("SC" if VM[k]["source"] == "supreme_court" else "other") for k in DEV}
dev_prompts = {k: VP[k]["text"] for k in DEV}

# ---- local metrics (need Cell 26 packages; test submissions work without them) --------------
try:
    import eval_local; importlib.reload(eval_local)
except ImportError:
    eval_local = None; print("eval packages missing -> run Cell 26 + restart for decoding studies")

def copy_rate(preds):
    # share of generated word-bigrams that also occur in the prompt
    r = []
    for k, t in preds.items():
        pw, gw = VP[k]["text"].split(), t.split()
        P = set(zip(pw, pw[1:])); G = list(zip(gw, gw[1:]))
        r.append(sum(g in P for g in G) / max(1, len(G)))
    return sum(r) / len(r)

def evaluate(name, preds, t0):
    r = eval_local.score(preds, VR, by_group=GRP)
    r.update(rep3=eval_local.repetition_rate(preds.values()), copy=copy_rate(preds), sec=round(time.time() - t0))
    print(f"{name:24s} BERT {r['bertscore_f1']:.4f} (SC {r['bertscore_f1[SC]']:.4f}) | ROUGE-L {r['rougeL_f1']:.4f} "
          f"| BLEU {r['bleu4']:.2f} | copy {r['copy']:.2f} | rep3 {r['rep3']:.3f} | {r['sec']}s", flush=True)
    return r

# ---- earlier study results (so comparison tables work after a restart) ----------------------
results = json.load(open(f"{WORK}/stats/decoding_m1_v2.json"))
print("setup done |", len(DEV), "validation prompts")

free GPU memory (GB): [9.0, 9.0] -> using GPU 0
setup done | 350 validation prompts


## Section 9: M3, Encoder–Decoder ("read the prompt, write the continuation")
- Each 1,024-token training window is **split**: the first **640 tokens** go to a **bidirectional encoder**; the **decoder** (causal self-attention + cross-attention to the encoder) predicts the **next 384 tokens**. This mirrors the task itself: read a 512-word prompt (about 610 tokens), then write what follows.
- Unlike the decoder-only models, **every prompt token sees the whole prompt** (no causal mask in the encoder). The encoder is reused in Phases 2–3 (NER, rhetorical roles), where bidirectional context matters.
- 6 encoder + 6 decoder layers, d = 768, GQA 12/4 heads, RoPE in self-attention (none in cross-attention), QK-norm, SwiGLU, embeddings shared by encoder, decoder and output. **About 110M parameters**, matching M1's size.
- **Generation:** the prompt is encoded **once**; each decoder layer's cross-attention K/V is computed once and cached; decoder self-attention uses a normal KV cache.
- **Trade-off to analyse:** only the 384 continuation tokens per window receive a loss (about 37.5%, vs 100% for decoder-only models), but those are exactly the kind of predictions the task needs. The encoder side costs no loss computation, so M3 trains in roughly 55% of M1's compute at the same 700M-token budget.
- The validation loss here is a **continuation loss** (next 384 tokens given 640) and **isn't directly comparable** to M1/M2's full-sequence loss. Across all four models we compare **BERTScore on the same validation prompts**.

In [4]:
%%writefile /kaggle/working/src/model_encdec.py
# model_encdec.py -- M3: encoder-decoder Transformer trained on "prompt -> continuation".
# Each 1,024-token window: first enc_len=640 tokens -> BIDIRECTIONAL encoder; decoder (causal
# self-attn + cross-attn) predicts the next 384. Generation: encode prompt once, cache each
# decoder layer's cross-attention K/V once, normal KV cache for decoder self-attention.
from __future__ import annotations
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from lm_common import RMSNorm, SwiGLU, rope_cache, apply_rope


@dataclass
class EncDecConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_enc_layers: int = 6
    n_dec_layers: int = 6
    n_heads: int = 12
    n_kv_heads: int = 4
    ffn_hidden: int = 2048
    enc_len: int = 640             # encoder part of each training window (prompt side)
    bos_id: int = 5                # <|bos|> starts every decoder sequence
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    init_std: float = 0.02


@dataclass
class DecCache:
    k: torch.Tensor                # decoder self-attention keys   (B, kv, max_T, hd)
    v: torch.Tensor
    ck: torch.Tensor = None        # cross-attention keys/values from the encoder (B, kv, Te, hd)
    cv: torch.Tensor = None

Writing /kaggle/working/src/model_encdec.py


In [5]:
%%writefile -a /kaggle/working/src/model_encdec.py


class SelfAttn(nn.Module):
    # GQA self-attention with QK-norm + RoPE; causal (decoder) or bidirectional (encoder)
    def __init__(self, c: EncDecConfig, causal: bool):
        super().__init__()
        self.h, self.kv, self.hd, self.causal = c.n_heads, c.n_kv_heads, c.d_model // c.n_heads, causal
        self.wq = nn.Linear(c.d_model, self.h * self.hd, bias=False)
        self.wk = nn.Linear(c.d_model, self.kv * self.hd, bias=False)
        self.wv = nn.Linear(c.d_model, self.kv * self.hd, bias=False)
        self.wo = nn.Linear(self.h * self.hd, c.d_model, bias=False)
        self.q_norm, self.k_norm = RMSNorm(self.hd), RMSNorm(self.hd)

    def forward(self, x, cos, sin, cache=None, pos=0):
        B, T, _ = x.shape
        q = self.q_norm(self.wq(x).view(B, T, self.h, self.hd).transpose(1, 2))
        k = self.k_norm(self.wk(x).view(B, T, self.kv, self.hd).transpose(1, 2))
        v = self.wv(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T]), apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
        if cache is not None:                                        # decoder generation
            cache.k[:, :, pos:pos + T], cache.v[:, :, pos:pos + T] = k, v
            k, v = cache.k[:, :, :pos + T], cache.v[:, :, :pos + T]
        rep = self.h // self.kv
        k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        assert T == 1 or pos == 0
        y = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and T > 1)
        return self.wo(y.transpose(1, 2).reshape(B, T, self.h * self.hd))


class CrossAttn(nn.Module):
    # decoder queries attend to encoder states (no mask, no RoPE); K/V computable once per prompt
    def __init__(self, c: EncDecConfig):
        super().__init__()
        self.h, self.kv, self.hd = c.n_heads, c.n_kv_heads, c.d_model // c.n_heads
        self.wq = nn.Linear(c.d_model, self.h * self.hd, bias=False)
        self.wk = nn.Linear(c.d_model, self.kv * self.hd, bias=False)
        self.wv = nn.Linear(c.d_model, self.kv * self.hd, bias=False)
        self.wo = nn.Linear(self.h * self.hd, c.d_model, bias=False)
        self.q_norm, self.k_norm = RMSNorm(self.hd), RMSNorm(self.hd)

    def kv(self, mem):
        B, S, _ = mem.shape
        k = self.k_norm(self.wk(mem).view(B, S, self.kv, self.hd).transpose(1, 2))
        v = self.wv(mem).view(B, S, self.kv, self.hd).transpose(1, 2)
        return k, v

    def forward(self, x, k, v):
        B, T, _ = x.shape
        q = self.q_norm(self.wq(x).view(B, T, self.h, self.hd).transpose(1, 2))
        rep = self.h // self.kv
        y = F.scaled_dot_product_attention(q, k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1))
        return self.wo(y.transpose(1, 2).reshape(B, T, self.h * self.hd))

Appending to /kaggle/working/src/model_encdec.py


In [6]:
%%writefile -a /kaggle/working/src/model_encdec.py


class EncLayer(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.n2 = RMSNorm(c.d_model), RMSNorm(c.d_model)
        self.attn, self.mlp = SelfAttn(c, causal=False), SwiGLU(c.d_model, c.ffn_hidden)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)                      # bidirectional
        return x + self.mlp(self.n2(x))


class DecLayer(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.n2, self.n3 = RMSNorm(c.d_model), RMSNorm(c.d_model), RMSNorm(c.d_model)
        self.self_attn, self.cross = SelfAttn(c, causal=True), CrossAttn(c)
        self.mlp = SwiGLU(c.d_model, c.ffn_hidden)

    def forward(self, x, cos, sin, ck, cv, cache=None, pos=0):
        x = x + self.self_attn(self.n1(x), cos, sin, cache, pos)    # causal over the continuation
        x = x + self.cross(self.n2(x), ck, cv)                        # look at the encoded prompt
        return x + self.mlp(self.n3(x))

Appending to /kaggle/working/src/model_encdec.py


In [7]:
%%writefile -a /kaggle/working/src/model_encdec.py


class EncDecLM(nn.Module):
    def __init__(self, c: EncDecConfig):
        super().__init__()
        self.cfg = c
        self.max_prompt_tokens = c.enc_len                     # inference keeps the last 640 tokens
        self.tok_emb = nn.Embedding(c.vocab_size, c.d_model)  # shared by encoder, decoder, output
        self.enc = nn.ModuleList([EncLayer(c) for _ in range(c.n_enc_layers)])
        self.dec = nn.ModuleList([DecLayer(c) for _ in range(c.n_dec_layers)])
        self.enc_norm, self.dec_norm = RMSNorm(c.d_model), RMSNorm(c.d_model)
        self.lm_head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        cos, sin = rope_cache(c.max_seq_len, c.d_model // c.n_heads, c.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        L = c.n_enc_layers + c.n_dec_layers
        for n, p in self.named_parameters():                   # scaled residual-output init
            if n.endswith(("wo.weight", "w2.weight")):
                nn.init.normal_(p, 0.0, c.init_std / math.sqrt(2 * L))
        self._prompt_len = 0

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, self.cfg.init_std)

    def encode(self, src):
        x = self.tok_emb(src)
        for layer in self.enc:
            x = layer(x, self.rope_cos, self.rope_sin)
        return self.enc_norm(x)

    def decode_tf(self, mem, dec_in):
        # teacher-forced decoder pass over the whole target sequence -> logits (B, Td, V)
        x = self.tok_emb(dec_in)
        for layer in self.dec:
            ck, cv = layer.cross.kv(mem)
            x = layer(x, self.rope_cos, self.rope_sin, ck, cv)
        return self.lm_head(self.dec_norm(x))

    def make_caches(self, B, max_T, device, dtype=torch.float16):
        kv, hd = self.cfg.n_kv_heads, self.cfg.d_model // self.cfg.n_heads
        return [DecCache(torch.zeros(B, kv, max_T, hd, device=device, dtype=dtype),
                         torch.zeros(B, kv, max_T, hd, device=device, dtype=dtype)) for _ in self.dec]

    def _gen_step(self, idx, caches, pos):
        # pos == 0: idx is the PROMPT -> encode once, cache cross K/V, decode <|bos|>
        # pos  > 0: idx is the last generated token (generation.py passes pos = prompt_len + i)
        if pos == 0:
            mem = self.encode(idx)
            for layer, c in zip(self.dec, caches):
                c.ck, c.cv = layer.cross.kv(mem)
            self._prompt_len = idx.size(1)
            tok, dpos = torch.full((idx.size(0), 1), self.cfg.bos_id, device=idx.device), 0
        else:
            tok, dpos = idx, pos - self._prompt_len + 1
        x = self.tok_emb(tok)
        for layer, c in zip(self.dec, caches):
            x = layer(x, self.rope_cos, self.rope_sin, c.ck, c.cv, c, dpos)
        return self.lm_head(self.dec_norm(x))

    def forward(self, idx, targets=None, caches=None, pos: int = 0):
        if caches is not None:
            return self._gen_step(idx, caches, pos)
        # training / eval: rebuild the (T+1)-token window, split it into prompt | continuation
        full = torch.cat([idx[:, :1], targets], dim=1)
        T, s = idx.size(1), self.cfg.enc_len
        src, tgt = full[:, :s], full[:, s:T]                                   # 640 | 384 tokens
        bos = torch.full((idx.size(0), 1), self.cfg.bos_id, device=idx.device, dtype=idx.dtype)
        logits = self.decode_tf(self.encode(src), torch.cat([bos, tgt[:, :-1]], 1))
        return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-1)

Appending to /kaggle/working/src/model_encdec.py


In [10]:
# ===== Cell 40e: small patches instead of re-pasting long files =====
import py_compile, re
# 1) model registry: add "encdec"
p = "/kaggle/working/src/train.py"; s = open(p).read()
if '"encdec"' not in s:
    s = s.replace('    raise ValueError(f"unknown arch {arch}")',
                  '    if arch == "encdec":\n'
                  '        from model_encdec import EncDecLM, EncDecConfig\n'
                  '        return EncDecLM(EncDecConfig(**model_cfg))\n'
                  '    raise ValueError(f"unknown arch {arch}")')
    open(p, "w").write(s)
# 2) inference: respect a model's own prompt limit (M3's encoder was trained on 640 tokens)
p2 = "/kaggle/working/src/infer.py"; s2 = open(p2).read()
if "max_prompt_tokens" not in s2:
    s2 = s2.replace("    max_prompt = max_ctx - max_new_tokens\n",
                    "    max_prompt = min(max_ctx - max_new_tokens, getattr(model, 'max_prompt_tokens', 10**9))\n")
    open(p2, "w").write(s2)
# 3) verify everything compiles and has what we expect
for f in ["train.py", "infer.py", "model_encdec.py"]:
    py_compile.compile(f"/kaggle/working/src/{f}", doraise=True)
src_m = open("/kaggle/working/src/model_encdec.py").read()
print("train.py registry has encdec:", '"encdec"' in open(p).read())
print("infer.py respects max_prompt_tokens:", "max_prompt_tokens" in open(p2).read())
print("model_encdec classes:", re.findall(r"^class (\w+)", src_m, flags=re.M))

train.py registry has encdec: True
infer.py respects max_prompt_tokens: True
model_encdec classes: ['EncDecConfig', 'DecCache', 'SelfAttn', 'CrossAttn', 'EncLayer', 'DecLayer', 'EncDecLM']


In [9]:
# ===== Cell 40f: fix name clash in model_encdec.py (CrossAttn.kv was both an int and a method) =====
p = "/kaggle/working/src/model_encdec.py"
s = open(p).read()
n_def, n_call = s.count("def kv(self, mem):"), s.count("layer.cross.kv(mem)")
print("found:", n_def, "definition,", n_call, "calls (expected 1 and 2)")
# rename the method to mem_kv; self.kv stays the number of K/V heads
s = s.replace("def kv(self, mem):", "def mem_kv(self, mem):").replace("layer.cross.kv(mem)", "layer.cross.mem_kv(mem)")
open(p, "w").write(s)
assert "cross.kv(" not in s and s.count("mem_kv") == 3, "patch incomplete -- send me this output"
print("patched OK")

found: 1 definition, 2 calls (expected 1 and 2)
patched OK


In [11]:
# ===== Cell 41: M3 correctness tests on a tiny model (few MB of GPU memory) =====
import sys, importlib, torch
sys.path.insert(0, "/kaggle/working/src")
import lm_common, model_encdec as ed, generation
for m in (lm_common, ed, generation):
    importlib.reload(m)
from lm_common import count_params
dev = "cuda:1"                                        # the GPU with more free memory

# [1] real config size (built on CPU, not moved to GPU)
full_m = ed.EncDecLM(ed.EncDecConfig())
print(f"[1] M3 parameters: {count_params(full_m)/1e6:.1f}M")
assert 50e6 <= count_params(full_m) <= 300e6
del full_m

# [2] cached generation must equal the teacher-forced decoder (fp32 tiny model)
torch.manual_seed(0)
tc = ed.EncDecConfig(d_model=128, n_enc_layers=2, n_dec_layers=2, n_heads=4, n_kv_heads=2,
                     ffn_hidden=256, enc_len=24, max_seq_len=256)
tiny = ed.EncDecLM(tc).to(dev).float().eval()
p = torch.randint(16, 32000, (2, 24), device=dev)      # "prompt"
c = torch.randint(16, 32000, (2, 10), device=dev)      # "continuation"
with torch.no_grad():
    bos = torch.full((2, 1), tc.bos_id, device=dev)
    tf = tiny.decode_tf(tiny.encode(p), torch.cat([bos, c[:, :-1]], 1))          # (2, 10, V)
    caches = tiny.make_caches(2, 24 + 10, dev, dtype=torch.float32)
    steps = [tiny(p, caches=caches, pos=0)] + [tiny(c[:, i:i + 1], caches=caches, pos=24 + i) for i in range(9)]
err = (torch.cat(steps, 1) - tf).abs().max().item()
print(f"[2] cached generation vs teacher-forced decoder: max |diff| = {err:.2e}")
assert err < 1e-3

# [3] generation.py drives the enc-dec model correctly, incl. shared-prefix n-sample expansion
with torch.no_grad():
    g = generation.generate(tiny, p, 12, temperature=0.8, top_p=0.9, num_return=3)
print(f"[3] generate(num_return=3) -> {tuple(g.shape)} (expected (6, 12))")
assert g.shape == (6, 12) and not (g == 0).any()

# [4] training forward: loss at init ~ ln(32000) and gradients flow to encoder AND decoder
tiny.train()
window = torch.randint(16, 32000, (2, 41), device=dev)
tc2 = ed.EncDecConfig(d_model=128, n_enc_layers=2, n_dec_layers=2, n_heads=4, n_kv_heads=2,
                      ffn_hidden=256, enc_len=24, max_seq_len=256)
t2 = ed.EncDecLM(tc2).to(dev)
loss = t2(window[:, :-1], window[:, 1:])
loss.backward()
g_enc = t2.enc[0].attn.wq.weight.grad.abs().sum().item(); g_dec = t2.dec[0].cross.wq.weight.grad.abs().sum().item()
print(f"[4] init loss {loss.item():.2f} (~10.4) | grad enc {g_enc:.2e} | grad dec {g_dec:.2e}")
assert 9.5 < loss.item() < 11.5 and g_enc > 0 and g_dec > 0
del tiny, t2; torch.cuda.empty_cache()
print("\nM3 CORRECTNESS TESTS PASSED")

[1] M3 parameters: 109.5M
[2] cached generation vs teacher-forced decoder: max |diff| = 5.96e-07
[3] generate(num_return=3) -> (6, 12) (expected (6, 12))
[4] init loss 10.44 (~10.4) | grad enc 2.19e+01 | grad dec 4.12e+01

M3 CORRECTNESS TESTS PASSED


In [12]:
# ===== Cell 42: M3 full-size throughput + memory test (1 GPU, eager vs compiled, ~5 min) =====
import sys, gc, time, importlib, torch
sys.path.insert(0, "/kaggle/working/src")
import lm_common, model_encdec as ed
for m in (lm_common, ed):
    importlib.reload(m)                                   # pick up the patched file
from lm_common import count_params

# ---- pick the emptier GPU; this test needs almost a whole T4 ---------------------------------
free = [torch.cuda.mem_get_info(i)[0] / 1e9 for i in range(torch.cuda.device_count())]
gpu = max(range(len(free)), key=free.__getitem__); dev = f"cuda:{gpu}"
torch.cuda.set_device(gpu)
print("free GPU memory (GB):", [round(f, 1) for f in free], "-> using GPU", gpu)
assert free[gpu] > 12, "run this when nothing else uses the GPU (no training / generation running)"

def bench_m3(mbs, use_compile, steps=8, T=1024):
    """Training tokens/s and peak memory for the real M3 config at micro-batch `mbs`."""
    torch.manual_seed(0)
    model = ed.EncDecLM(ed.EncDecConfig()).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, fused=True)
    scaler = torch.amp.GradScaler("cuda")                  # fp16 training, same as M1/M2
    m = torch.compile(model) if use_compile else model
    w = torch.randint(16, 32000, (mbs, T + 1), device=dev) # one (T+1)-token window per row
    torch.cuda.reset_peak_memory_stats(dev)
    for i in range(steps + 3):                             # first 3 iterations = warm-up/compile
        if i == 3:
            torch.cuda.synchronize(); t0 = time.time()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = m(w[:, :-1], w[:, 1:])                  # same call train.py makes
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    tps = mbs * T * steps / (time.time() - t0)             # window tokens/s (comparable to M1's 23.6k)
    mem = torch.cuda.max_memory_allocated(dev) / 1e9
    n, ok = count_params(model), bool(torch.isfinite(loss))
    del model, m, opt, w; gc.collect(); torch.cuda.empty_cache()
    return tps, mem, n, ok

# ---- table: hours = time for the 700M-token budget on 2xT4 (~1.85x one GPU) ------------------
print(f"{'mbs':>4s} {'compile':>8s} {'tok/s':>8s} {'peak GB':>8s} {'h/700M (2xT4)':>14s}  finite")
for mbs, c in [(8, False), (8, True), (16, True)]:
    try:
        tps, mem, n, ok = bench_m3(mbs, c)
        print(f"{mbs:4d} {str(c):>8s} {tps:8.0f} {mem:8.2f} {700e6 / (tps * 1.85) / 3600:14.1f}  {ok}")
    except torch.cuda.OutOfMemoryError:
        print(f"{mbs:4d} {str(c):>8s}      OOM"); gc.collect(); torch.cuda.empty_cache()
print(f"M3 parameters: {n/1e6:.1f}M   (reference: M1 compiled mbs8 = 23.6k tok/s, 6.0 GB)")

free GPU memory (GB): [15.0, 15.4] -> using GPU 1
 mbs  compile    tok/s  peak GB  h/700M (2xT4)  finite
   8    False    22222     5.83            4.7  True


W0930 10:41:45.656000 58 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


   8     True    34442     3.69            3.1  True
  16     True    36754     5.65            2.9  True
M3 parameters: 109.5M   (reference: M1 compiled mbs8 = 23.6k tok/s, 6.0 GB)


In [20]:
# ===== Cell 43: launch M3 (encoder-decoder) in the BACKGROUND -- same budget/recipe as M1, M2 =====
import json, os, subprocess, gc, torch
gc.collect(); torch.cuda.empty_cache()                   # release memory held by Cell 42

WORK = "/kaggle/working"
RUN = "m3_encdec"
run_cfg = {
    "arch": "encdec", "model": {},                        # EncDecConfig defaults: 6+6 layers, enc_len 640
    "run_name": RUN, "out_dir": f"{WORK}/checkpoints/{RUN}", "data_dir": f"{WORK}/data",
    # ---- identical budget + recipe to M1/M2 (controlled comparison) --------------------------
    "seq_len": 1024, "micro_bs": 8, "global_bs_seqs": 256, "max_tokens": 700_000_000,
    "lr": 6e-4, "min_lr_frac": 0.1, "warmup_frac": 0.03, "schedule": "wsd", "decay_frac": 0.2,
    "weight_decay": 0.1, "betas": [0.9, 0.95], "grad_clip": 1.0, "compile": True, "seed": 1337,
    "mixture": "curriculum",
    "log_every": 20, "eval_every": 200, "val_windows": 64,
    "ckpt_every_min": 15, "time_limit_hours": 11.0,
}
os.makedirs(run_cfg["out_dir"], exist_ok=True)
cfg_path = f"{run_cfg['out_dir']}/config.json"
json.dump(run_cfg, open(cfg_path, "w"), indent=1)
LOG = f"{run_cfg['out_dir']}/stdout.log"

# '>>' appends, so a resumed run keeps its earlier log; resumes from latest.pt automatically
# setsid: training gets its own process session, so stopping a notebook cell can't kill it
cmd = (f"cd {WORK} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True "
       f"setsid nohup torchrun --standalone --nproc_per_node=2 {WORK}/src/train.py --config {cfg_path} "
       f">> {LOG} 2>&1 < /dev/null &")
subprocess.run(cmd, shell=True)
print("M3 launched in background. Log file:", LOG)

M3 launched in background. Log file: /kaggle/working/checkpoints/m3_encdec/stdout.log


In [1]:
# ===== (M3): latest progress of the background M3 run =====
!tail -n 8 /kaggle/working/checkpoints/m3_encdec/stdout.log
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used,memory.total --format=csv

  new best SC val loss 2.5231 -> best.pt
step   2620/2670 |   686.8M tok | loss 2.5002 | lr 7.21e-05 | gnorm 0.25 | scale 32768 | 52.7k tok/s | SC w 0.75 | 3.39 h
step   2640/2670 |   692.1M tok | loss 2.4978 | lr 6.45e-05 | gnorm 0.25 | scale 32768 | 55.0k tok/s | SC w 0.75 | 3.42 h
step   2660/2670 |   697.3M tok | loss 2.5021 | lr 6.06e-05 | gnorm 0.24 | scale 32768 | 55.1k tok/s | SC w 0.75 | 3.44 h
  EVAL acts loss 2.400 bpw 4.457 | supre loss 2.515 bpw 4.468 | tribu loss 2.346 bpw 4.942 | high_ loss 2.065 bpw 3.966
  new best SC val loss 2.5146 -> best.pt
  checkpoint saved at step 2670
TRAINING COMPLETE: 0.70B tokens, best SC val loss 2.5146
index, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
0, 0 %, 0 MiB, 15360 MiB
1, 0 %, 0 MiB, 15360 MiB


In [19]:
# ===== Cell 43a: release the model Cell 39c loaded, check nothing is training =====
import gc, torch
if "model" in globals():
    del model
gc.collect(); torch.cuda.empty_cache()
!ps aux | grep -c "[t]rain.py"          # 0 = no training process alive
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

0
index, utilization.gpu [%], memory.used [MiB]
0, 0 %, 159 MiB
1, 92 %, 1393 MiB


In [11]:
# ===== Cell 43b: M3 decoding check -- the two best settings from the M1 study =====
assert "evaluate" in globals() and eval_local is not None, "run Cell 26 + restart, then Cell 39s"
if "model" in globals():
    del model; gc.collect(); torch.cuda.empty_cache()
model, ck = infer.load_model(f"{WORK}/checkpoints/m3_encdec/best.pt")     # keeps the last 640 prompt tokens
res3 = {}

# ---- 1) greedy + repetition penalty 1.2 + copy bonus 1.0 (also MBR's 9th candidate) ----------
t0 = time.time(); torch.manual_seed(1337)
g3, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                               temperature=0, repetition_penalty=1.2, prompt_bonus=1.0)
res3["m3_greedy_rep1.2_copy1.0"] = evaluate("m3_greedy_rep1.2_copy1.0", g3, t0)

# ---- 2) MBR over 8 samples + the greedy output, copy bonus 1.0 --------------------------------
t0 = time.time(); torch.manual_seed(1337)
c3, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                               num_return=8, temperature=0.7, top_p=0.9,
                               repetition_penalty=1.1, prompt_bonus=1.0)
picks3 = {k: infer.mbr_select(c3[k] + [g3[k]])[0] for k in DEV}
res3["m3_mbr9_copy1.0"] = evaluate("m3_mbr9_copy1.0", picks3, t0)

# ---- compare with M1 and M2 (same prompts, same decoding), then save -------------------------
m2r = json.load(open(f"{WORK}/stats/decoding_m2_v2.json"))
table = {"m1_mbr9_copy1.0": results["mbr9_copy1.0"], "m2_mbr9_copy1.0": m2r["m2_mbr9_copy1.0"], **res3}
display(pd.DataFrame(table).T[["bertscore_f1", "bertscore_f1[SC]", "rougeL_f1", "bleu4",
                               "copy", "rep3", "sec"]].round(4))
json.dump(res3, open(f"{WORK}/stats/decoding_m3_v2.json", "w"), indent=1)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (4 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


m3_mbr9_copy1.0          BERT 0.8377 (SC 0.8380) | ROUGE-L 0.2247 | BLEU 7.94 | copy 0.31 | rep3 0.094 | 1979s


,bertscore_f1,bertscore_f1[SC],rougeL_f1,bleu4,copy,rep3,sec
m1_mbr9_copy1.0,0.8495,0.8462,0.2599,14.3971,0.4694,0.0862,1355.0
m2_mbr9_copy1.0,0.8499,0.8460,0.2597,14.5642,0.4520,0.0806,4107.0
m3_greedy_rep1.2_copy1.0,0.8316,0.8316,0.2069,7.0734,0.2648,0.1354,361.0
m3_mbr9_copy1.0,0.8377,0.8380,0.2247,7.9404,0.3109,0.0943,1979.0


In [5]:
# ===== Cell 43c: M3 test-set predictions (MBR-9 + copy bonus 1.0) -> Codabench zip =====
# Same decoding as the M1 (0.8443) and M2 submissions -> like-for-like architecture comparison.
RUN_NAME = "m3_encdec"
TAG      = "mbr9_copy1.0"
BETA     = 1.0

assert all(v in globals() for v in ("tk", "first_mask", "infer", "DATA", "SUB", "WORK")), \
    "kernel was restarted -- run Cell 39s first"

# ---- free whatever model is on the GPU, load M3's best checkpoint ------------------------------
if "model" in globals():
    del model; gc.collect(); torch.cuda.empty_cache()
test = json.load(open(f"{DATA}/Phase_1_Pretraining/Test/test.json"))
test_prompts = {k: v["text"] for k, v in test.items()}
model, ck = infer.load_model(f"{WORK}/checkpoints/{RUN_NAME}/best.pt")   # encoder reads last 640 tokens
print(f"{RUN_NAME}: step {ck['step']} | {len(test_prompts)} test prompts", flush=True)
t0 = time.time(); torch.manual_seed(1337)

# ---- candidate 9: greedy + repetition penalty 1.2 + copy bonus --------------------------------
greedy, _ = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask,
                                   temperature=0, repetition_penalty=1.2, prompt_bonus=BETA)
print(f"greedy done ({(time.time()-t0)/60:.1f} min), sampling 8 candidates per prompt ...", flush=True)

# ---- candidates 1-8: nucleus samples sharing one encoded prompt (cross-attn K/V computed once) --
cands, st = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask,
                                   num_return=8, temperature=0.7, top_p=0.9,
                                   repetition_penalty=1.1, prompt_bonus=BETA)

# ---- MBR: keep the candidate that agrees most (n-gram F1) with the other 8 ---------------------
preds = {k: infer.mbr_select(cands[k] + [greedy[k]])[0] for k in test_prompts}
print("generation stats:", st, f"| {(time.time()-t0)/60:.1f} min")

# ---- write + verify the zip (1000 ids, exactly 128 words, prediction.json at root) ------------
name = f"{RUN_NAME}_{TAG}"
zp = infer.write_submission(preds, test.keys(), f"{SUB}/{name}", name)
json.dump({"model": RUN_NAME, "ckpt_step": ck["step"], "mbr": True,
           "decoding": {"prompt_bonus": BETA, "greedy_rep": 1.2, "samples": 8, "T": 0.7, "top_p": 0.9, "sample_rep": 1.1}},
          open(f"{SUB}/{name}/meta.json", "w"), indent=1)          # record of what was submitted
print("submission zip:", zp)
print("example:", preds[next(iter(preds))][:300])

m3_encdec: step 2670 | 1000 test prompts
greedy done (10.7 min), sampling 8 candidates per prompt ...
generation stats: {'n': 1000, 'left_truncated_in_batch': 43, 'short_rows_extended': 352} | 80.3 min
submission zip: /kaggle/working/submissions/m3_encdec_mbr9_copy1.0/m3_encdec_mbr9_copy1.0.zip
example: no reason to hold that the judgment of this Court in [State of Rajasthan v. Raghubir Singh] (1997) 4 SCC 648 is not a correct view. It is only when there are two reasonable and valid grounds to quash the acquisition proceedings and then the acquisition proceedings should be quashed. The same, as con


## Section 10. M4: Dense decoder + learned legal copy head (pointer–generator)

**Motivation.** The decoding study (Section 8) showed that copying from the prompt helps, but a *fixed* copy bonus helps only up to $\beta \approx 1$ and then causes loops. M4 **learns** when and what to copy.

**Architecture.** At every position $t$, a scalar gate mixes two distributions:

$$
g_t = \sigma(\mathbf{w}^\top \mathbf{h}_t + b)
$$

$$
p(y_t) \;=\; (1 - g_t)\,\underbrace{\mathrm{softmax}(W\mathbf{h}_t)[y_t]}_{\text{generate from vocabulary}} \;+\; g_t \underbrace{\sum_{i<t,\; y_i = y_t} a_{t,i}}_{\text{copy (pointer)}}
$$

- **Vocabulary distribution:** the usual softmax over the 32k tokens (tied embeddings, as in M1).
- **Pointer distribution:** attention over earlier positions,
  $$a_{t,i} = \mathrm{softmax}_i\!\left(\frac{\mathbf{q}_t^\top \mathbf{k}_i}{\sqrt{d_c}}\right),\quad i < t,\qquad \mathbf{q}_t = W_q\mathbf{h}_t,\; \mathbf{k}_i = W_k\mathbf{h}_i,\; d_c = 128$$
  Pointing at position $i$ proposes the token that **followed** it ($y_i = x_{i+1}$). This is an *induction-style* copy: "the current context looks like the context before that earlier token, so copy it". Only $i < t$ is allowed, so the target is never visible.

**Training objective.** The exact negative log-likelihood of the mixture (computed stably in log-space with `logaddexp` / `logsumexp`). It is a proper probability, so **M4's loss is directly comparable to M1's**.

**Cost.** Backbone identical to M1 (100.1M parameters), plus only $2 \times 768 \times 128 + 769 \approx 0.2$M parameters for the head (**+0.2%**).

**Training.** The same 700M-token recipe as M1–M3 (WSD schedule, curriculum mix, 256 × 1024 tokens per step). Because of the GPU quota, training ran in **two parts**: 3.6 h, then an exact resume from `latest.pt` on a second session (deterministic data sampling makes the resume seamless).

| Model | SC val loss | Params (active) |
|---|---|---|
| M1 dense | 2.338 | 100.1M |
| **M4 dense + copy head** | **2.301** | **100.3M** |
| M2 MLA + MoE | 2.288 | 270M (102M) |

In [2]:
%%writefile /kaggle/working/src/model_copy.py
# model_copy.py -- M4: dense decoder (M1 backbone) + a learned LEGAL COPY HEAD.
# Legal continuations reuse the prompt heavily (party names, section numbers, quoted phrases);
# the decoding study showed a fixed copy bonus helps up to beta~1 and then hurts. M4 LEARNS it.
# Output = mixture of two distributions, mixed by a per-token gate g_t = sigmoid(w . h_t + b):
#   p(y_t) = (1 - g_t) * softmax(W h_t)[y_t]            (generate from the vocabulary)
#          +      g_t  * sum_{i<t, y_i = y_t} a_{t,i}     (copy: point at an earlier position)
# a_{t,i} = softmax_i(q_t . k_i / sqrt(d_c)) over i < t, q_t = Wq h_t, k_i = Wk h_i.
# Induction-style pointer: key i is the state that PRECEDED token y_i = x_{i+1}, so the head
# learns "the context now looks like the context before that earlier token -> copy it".
# Strictly i < t, so the target y_t is never visible. Extra parameters: 2*768*128 + 769 ~ 0.2M.
from __future__ import annotations
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from model_dense import DenseGPT, DenseConfig

NEG = -1e4                                    # finite "minus infinity" (keeps gradients NaN-free)


@dataclass
class CopyConfig(DenseConfig):
    copy_dim: int = 128                       # query/key size of the pointer attention
    gate_bias_init: float = -2.0              # sigmoid(-2) ~ 0.12: start mostly generating


@dataclass
class CopyCache:
    k: torch.Tensor                           # pointer keys  (B, max_T, copy_dim)
    ids: torch.Tensor                         # input token ids (B, max_T); value of key i = ids[i+1]

Writing /kaggle/working/src/model_copy.py


In [3]:
%%writefile -a /kaggle/working/src/model_copy.py


class CopyGPT(DenseGPT):
    def __init__(self, c: CopyConfig):
        super().__init__(c)                                          # M1 backbone + its init
        self.copy_q = nn.Linear(c.d_model, c.copy_dim, bias=False)
        self.copy_k = nn.Linear(c.d_model, c.copy_dim, bias=False)
        self.gate = nn.Linear(c.d_model, 1)
        for m in (self.copy_q, self.copy_k, self.gate):
            nn.init.normal_(m.weight, 0.0, c.init_std)
        nn.init.constant_(self.gate.bias, c.gate_bias_init)

    def _hidden(self, idx, caches=None, pos=0):
        x = self.tok_emb(idx)
        for i, blk in enumerate(self.blocks):
            x = blk(x, self.rope_cos, self.rope_sin, None if caches is None else caches[i], pos)
        return self.norm_f(x)

    def make_caches(self, B, max_T, device, dtype=torch.float16):
        # M1's per-layer KV caches + one pointer cache (keys + token ids) at the end
        return super().make_caches(B, max_T, device, dtype) + [
            CopyCache(torch.zeros(B, max_T, self.cfg.copy_dim, device=device, dtype=dtype),
                      torch.zeros(B, max_T, dtype=torch.long, device=device))]

    def copy_loss(self, h, targets):
        """Mean NLL of the gated mixture (a proper probability -> comparable to M1's loss)."""
        B, T, _ = h.shape
        z = self.lm_head(h)
        lp_vocab = -F.cross_entropy(z.float().view(-1, z.size(-1)), targets.reshape(-1),
                                    reduction="none").view(B, T)           # log p_vocab(y_t)
        with torch.autocast("cuda", enabled=False):                        # pointer math in fp32
            hf = h.float()
            allowed = torch.ones(T, T, dtype=torch.bool, device=h.device).tril(-1)  # i < t
            g = self.gate(hf).squeeze(-1).masked_fill(~allowed.any(-1), NEG)  # t=0: nothing to copy
            s = self.copy_q(hf) @ self.copy_k(hf).transpose(1, 2) / math.sqrt(self.cfg.copy_dim)
            la = torch.log_softmax(s.masked_fill(~allowed, NEG), -1)       # log a_{t,i}
            match = targets[:, None, :] == targets[:, :, None]              # [b,t,i]: y_i == y_t
            lp_copy = torch.logsumexp(la.masked_fill(~(match & allowed), NEG), -1)   # (B, T)
            lp = torch.logaddexp(F.logsigmoid(-g) + lp_vocab, F.logsigmoid(g) + lp_copy)
        return -lp.mean()

    def forward(self, idx, targets=None, caches=None, pos: int = 0):
        if caches is None:
            h = self._hidden(idx)
            if targets is None:
                return self.lm_head(h)                                      # vocab-only logits
            return self.copy_loss(h, targets)
        # ---- generation: returns LOG-PROBS of the mixture for the last position, (B, 1, V) ----
        h = self._hidden(idx, caches, pos)
        cc, T = caches[-1], idx.size(1)
        cc.k[:, pos:pos + T] = self.copy_k(h).to(cc.k.dtype)
        cc.ids[:, pos:pos + T] = idx
        t = pos + T - 1                                                     # current position
        hl = h[:, -1:]
        z = self.lm_head(hl).float()
        with torch.autocast("cuda", enabled=False):
            out = torch.log_softmax(z, -1)                                  # t=0: nothing to copy
            if t >= 1:
                hf = hl.float()
                g = self.gate(hf)                                           # (B, 1, 1)
                out = out + F.logsigmoid(-g)
                a = torch.softmax(self.copy_q(hf) @ cc.k[:, :t].float().transpose(1, 2)
                                  / math.sqrt(self.cfg.copy_dim), -1)       # (B, 1, t), keys i < t
                pc = torch.zeros_like(out).scatter_add_(2, cc.ids[:, None, 1:t + 1], a)
                out = torch.logaddexp(out, F.logsigmoid(g) + torch.log(pc))  # log 0 = -inf is fine
        return out

Appending to /kaggle/working/src/model_copy.py


In [4]:
# ===== Cell 44c: add arch "copy" to train.py's model registry + verify model_copy.py compiles =====
import py_compile
p = "/kaggle/working/src/train.py"
s = open(p).read()
if 'arch == "copy"' in s:
    print("train.py already knows arch 'copy'")
else:
    old = 'raise ValueError(f"unknown arch {arch}")'
    assert s.count(old) == 1, "registry line not found -- send me this output"
    s = s.replace(old, 'if arch == "copy":\n        from model_copy import CopyGPT, CopyConfig\n'
                       '        return CopyGPT(CopyConfig(**model_cfg))\n    ' + old)
    open(p, "w").write(s); print("train.py patched: arch 'copy' registered")
print("registry has encdec:", 'arch == "encdec"' in s, "| copy:", 'arch == "copy"' in s)
py_compile.compile("/kaggle/working/src/model_copy.py", doraise=True)   # catches a truncated paste
print("model_copy.py compiles, lines:", sum(1 for _ in open("/kaggle/working/src/model_copy.py")))

train.py patched: arch 'copy' registered
registry has encdec: True | copy: True
model_copy.py compiles, lines: 97


In [5]:
# ===== Cell 45: M4 tests -- cached decoding == training loss, probabilities, learning, speed =====
import sys, gc, time, math, importlib, torch
sys.path.insert(0, "/kaggle/working/src")
import lm_common, model_dense, model_copy as mc, generation, train
for m in (lm_common, model_dense, mc, generation, train):
    importlib.reload(m)
from lm_common import count_params
dev = "cuda:0"

# [1] real size + registry works
full = train.build_model("copy", {})
n_head = sum(p.numel() for n, p in full.named_parameters() if n.startswith(("copy_", "gate")))
print(f"[1] M4 params {count_params(full)/1e6:.2f}M (copy head {n_head/1e3:.0f}k)"); del full

# [2]-[4] tiny fp32 model: training loss must equal token-by-token cached decoding
torch.manual_seed(0)
tc = mc.CopyConfig(vocab_size=200, d_model=64, n_layers=2, n_heads=4, n_kv_heads=2, ffn_hidden=128,
                   max_seq_len=128, copy_dim=16, gate_bias_init=0.0)
tiny = mc.CopyGPT(tc).to(dev).float().eval()
x = torch.randint(16, 200, (2, 33), device=dev)
x[:, 17:33] = x[:, 1:17]                                  # second half repeats -> copying matters
with torch.no_grad():
    nll_tf = tiny(x[:, :-1], x[:, 1:]).item()                                     # training path
    caches = tiny.make_caches(2, 32, dev, dtype=torch.float32)
    outs = [tiny(x[:, t:t + 1], caches=caches, pos=t)[:, -1] for t in range(32)]  # decode path
    lp = torch.stack([o.gather(1, x[:, t + 1:t + 2]).squeeze(1) for t, o in enumerate(outs)], 1)
    pre = tiny(x[:, :20], caches=tiny.make_caches(2, 32, dev, dtype=torch.float32), pos=0)[:, -1]
print(f"[2] NLL training path {nll_tf:.5f} vs cached token-by-token {-lp.mean().item():.5f}")
assert abs(nll_tf + lp.mean().item()) < 1e-4
err = (pre - outs[19]).abs().max().item()
print(f"[3] prefill vs step-by-step at t=19: max |diff| {err:.1e}"); assert err < 1e-4
tot = torch.stack(outs).exp().sum(-1)
print(f"[4] mixture probs sum to 1: {tot.min().item():.5f} .. {tot.max().item():.5f}")
assert (tot - 1).abs().max() < 1e-4

# [5] gradients reach the pointer and the gate; [6] generation.py drives it (incl. MBR expansion)
tiny.train(); loss = tiny(x[:, :-1], x[:, 1:]); loss.backward()
gq, gg = tiny.copy_q.weight.grad.abs().sum().item(), tiny.gate.weight.grad.abs().sum().item()
print(f"[5] grad copy_q {gq:.2e} | grad gate {gg:.2e}"); assert gq > 0 and gg > 0
with torch.no_grad():
    y = generation.generate(tiny, x[:, :20], 8, temperature=0.8, top_p=0.9, num_return=3)
print(f"[6] generate(num_return=3) -> {tuple(y.shape)} (expected (6, 8))"); assert y.shape == (6, 8)

# [7] can it learn to copy? 150 quick steps on "16 random tokens repeated 4 times"
torch.manual_seed(1)
t2 = mc.CopyGPT(tc).to(dev); opt = torch.optim.AdamW(t2.parameters(), lr=3e-3)
for _ in range(150):
    b = torch.randint(16, 200, (16, 16), device=dev).repeat(1, 4)
    loss = t2(b[:, :-1], b[:, 1:]); opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    g = torch.sigmoid(t2.gate(t2._hidden(b[:, :-1]))).squeeze(-1)
print(f"[7] loss {loss.item():.2f} (no copying >= {15/63*math.log(184):.2f} + repeats) | "
      f"gate: first pass {g[:, 1:15].mean():.2f} vs repeats {g[:, 16:].mean():.2f}")
assert loss.item() < 4.0 and g[:, 16:].mean() > g[:, 1:15].mean()
del tiny, t2, opt; gc.collect(); torch.cuda.empty_cache()

# [8] full-size training speed + memory (compiled, micro-batch 8). M1: 23.6k tok/s, 6.0 GB
model = mc.CopyGPT(mc.CopyConfig()).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, fused=True); scaler = torch.amp.GradScaler("cuda")
m = torch.compile(model); w = torch.randint(16, 32000, (8, 1025), device=dev)
torch.cuda.reset_peak_memory_stats(dev)
for i in range(11):
    if i == 3:
        torch.cuda.synchronize(); t0 = time.time()
    with torch.autocast("cuda", dtype=torch.float16):
        loss = m(w[:, :-1], w[:, 1:])
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
torch.cuda.synchronize(); tps = 8 * 1024 * 8 / (time.time() - t0)
print(f"[8] {tps:.0f} tok/s | peak {torch.cuda.max_memory_allocated(dev)/1e9:.1f} GB | "
      f"~{700e6/(tps*1.85)/3600:.1f} h for 700M tokens on 2xT4 | loss finite {bool(torch.isfinite(loss))}")
del model, m, opt, w; gc.collect(); torch.cuda.empty_cache()
print("\nM4 TESTS PASSED")

[1] M4 params 100.29M (copy head 197k)
[2] NLL training path 4.60449 vs cached token-by-token 4.60449
[3] prefill vs step-by-step at t=19: max |diff| 4.8e-07
[4] mixture probs sum to 1: 1.00000 .. 1.00000
[5] grad copy_q 3.03e+00 | grad gate 1.96e+00
[6] generate(num_return=3) -> (6, 8) (expected (6, 8))
[7] loss 1.40 (no copying >= 1.24 + repeats) | gate: first pass 0.02 vs repeats 0.99


W0930 22:18:32.897000 58 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


[8] 21446 tok/s | peak 5.0 GB | ~4.9 h for 700M tokens on 2xT4 | loss finite True

M4 TESTS PASSED


In [1]:
# ===== Cell 46: launch M4 (dense + learned copy head) -- same budget/recipe as M1, M2, M3 =====
import json, os, subprocess, gc, torch
gc.collect(); torch.cuda.empty_cache()

WORK = "/kaggle/working"
RUN = "m4_copy"
run_cfg = {
    "arch": "copy", "model": {},                          # CopyConfig defaults: M1 backbone + pointer head
    "run_name": RUN, "out_dir": f"{WORK}/checkpoints/{RUN}", "data_dir": f"{WORK}/data",
    # ---- identical budget + recipe (controlled comparison) -----------------------------------
    "seq_len": 1024, "micro_bs": 8, "global_bs_seqs": 256, "max_tokens": 700_000_000,
    "lr": 6e-4, "min_lr_frac": 0.1, "warmup_frac": 0.03, "schedule": "wsd", "decay_frac": 0.2,
    "weight_decay": 0.1, "betas": [0.9, 0.95], "grad_clip": 1.0, "compile": True, "seed": 1337,
    "mixture": "curriculum",
    "log_every": 20, "eval_every": 200, "val_windows": 64,
    # quota-limited: stops cleanly after 3.6 h; after the weekly reset set 11.0 and re-run to resume
    "ckpt_every_min": 15, "time_limit_hours": 11,
}
os.makedirs(run_cfg["out_dir"], exist_ok=True)
cfg_path = f"{run_cfg['out_dir']}/config.json"
json.dump(run_cfg, open(cfg_path, "w"), indent=1)
LOG = f"{run_cfg['out_dir']}/stdout.log"

# setsid: own process session -> stopping notebook cells can't kill training; '>>' keeps old logs
cmd = (f"cd {WORK} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True "
       f"setsid nohup torchrun --standalone --nproc_per_node=2 {WORK}/src/train.py --config {cfg_path} "
       f">> {LOG} 2>&1 < /dev/null &")
subprocess.run(cmd, shell=True)
print("M4 launched in background. Log file:", LOG)

M4 launched in background. Log file: /kaggle/working/checkpoints/m4_copy/stdout.log


In [3]:
# ===== Cell 46b: latest progress of the background M4 run =====
!tail -n 8 /kaggle/working/checkpoints/m4_copy/stdout.log
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used,memory.total --format=csv

[rank1]:W1001 19:40:47.776000 115 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
[rank0]:W1001 19:40:47.778000 114 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
step   1486/2670 |   389.5M tok | loss 2.1525 | lr 6.00e-04 | gnorm 0.23 | scale 16384 | 2.6k tok/s | SC w 0.3 | 3.63 h
step   1500/2670 |   393.2M tok | loss 2.2200 | lr 6.00e-04 | gnorm 0.28 | scale 16384 | 37.8k tok/s | SC w 0.3 | 3.66 h
step   1520/2670 |   398.5M tok | loss 2.2245 | lr 6.00e-04 | gnorm 0.22 | scale 16384 | 37.4k tok/s | SC w 0.3 | 3.70 h
step   1540/2670 |   403.7M tok | loss 2.2202 | lr 6.00e-04 | gnorm 0.22 | scale 16384 | 37.6k tok/s | SC w 0.3 | 3.73 h
step   1560/2670 |   408.9M tok | loss 2.1981 | lr 6.00e-04 | gnorm 0.20 | scale 16384 | 37.0k tok/s | SC w 0.3 | 3.77 h
step   1580/2670 |   414.2M tok | loss 2.2075 | lr 6.00e-04 | gnorm 0.23 | scale 16384 | 37.0k tok/s | SC w 0.3 | 3.81 h
index, utilization.gpu [%], memory.used [Mi

In [8]:
%%writefile /kaggle/working/src/backup_hf.py
# backup_hf.py -- upload everything needed to continue Phase 1 on another account to a PRIVATE HF repo
import os, sys, time
from huggingface_hub import HfApi
WORK = "/kaggle/working"
api = HfApi(token=os.environ["HF_WRITE_TOKEN"])
repo = f"{api.whoami()['name']}/cs781-phase1-work"
try:
    api.create_repo(repo, private=True, exist_ok=True)
    t0 = time.time()
    api.upload_folder(
        repo_id=repo, folder_path=WORK, commit_message="phase-1 working state",
        allow_patterns=["src/**", "tokenizer/**", "data/**", "stats/**", "submissions/**",
                        "checkpoints/*/best.pt", "checkpoints/*/config.json", "checkpoints/*/*.csv",
                        "checkpoints/*/*.jsonl", "checkpoints/*/stdout.log",
                        "checkpoints/m4_copy/latest.pt"],          # optimizer state -> exact resume
        ignore_patterns=["**/__pycache__/**", "**/*.tmp"])
    print(f"BACKUP DONE -> {repo} in {(time.time() - t0) / 60:.1f} min", flush=True)
except Exception as e:
    print("BACKUP FAILED:", repr(e), flush=True); sys.exit(1)

Writing /kaggle/working/src/backup_hf.py


In [6]:
# ===== Cell 46c: keep the session alive until M4 has stopped AND the backup has finished =====
import time, subprocess, os
from IPython.display import clear_output
LOG, BLOG = "/kaggle/working/checkpoints/m4_copy/stdout.log", "/kaggle/working/backup.log"
while True:
    out = subprocess.run(f"tail -n 3 {LOG}", shell=True, capture_output=True, text=True).stdout
    bk = open(BLOG).read()[-300:] if os.path.exists(BLOG) else ""
    clear_output(wait=True); print(time.strftime("%H:%M"), "\n" + out, "\nbackup:", bk.strip() or "waiting", flush=True)
    if "BACKUP DONE" in bk or "BACKUP FAILED" in bk:
        break                                   # safe to let the session end now
    time.sleep(300)

22:19 
  new best SC val loss 2.3010 -> best.pt
  checkpoint saved at step 2670
TRAINING COMPLETE: 0.70B tokens, best SC val loss 2.3010
 
backup: waiting


KeyboardInterrupt: 

In [9]:
# ===== Cell 46e: verify the HF write token, then launch a background job that backs up when M4 stops =====
import os, subprocess
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
tok = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
api = HfApi(token=tok)
repo = f"{api.whoami()['name']}/cs781-phase1-work"
api.create_repo(repo, private=True, exist_ok=True)          # fails now (not at 5 am) if the token is wrong
print("token OK, private repo ready:", repo)

# waits while training runs ('[s]rc' stops pgrep matching this bash line itself), then uploads
cmd = ("cd /kaggle/working && setsid nohup bash -c "
       "'while pgrep -f \"[s]rc/train.py\" > /dev/null; do sleep 60; done; python src/backup_hf.py' "
       "> /kaggle/working/backup.log 2>&1 < /dev/null &")
subprocess.run(cmd, shell=True, env={**os.environ, "HF_WRITE_TOKEN": tok})   # token via env, not in the command
print("auto-backup armed -> log: /kaggle/working/backup.log")

token OK, private repo ready: Utkarsh-1718-1/cs781-phase1-work
auto-backup armed -> log: /kaggle/working/backup.log


## Section 10b. M4 decoding check and submission

**Plan.** First, how strongly does the learned copy gate fire on validation prompts? Then a $2 \times 2$ study:

| | Greedy (rep 1.2) | MBR-9 |
|---|---|---|
| **Learned copy alone** ($\beta = 0$) | ✓ | ✓ |
| **Learned copy + decoding bonus** ($\beta = 1.0$) | ✓ | ✓ |

This tells us whether the copy head **replaces** the hand-tuned prompt bonus from Section 8. Finally, the test submission with the best setting.

**What actually happened.** The first two greedy rows (Cell 47a) degenerated: **rep3 ≈ 0.54–0.58**, i.e. more than half of all 3-grams repeated. The study was stopped there, and Cell 47b (automatic submission) was **not used**; it is superseded by Cells 47c–47d in Section 10c.

**Diagnosis (two separate causes).**
1. **Repetition-penalty scale bug (the main cause).** The CTRL-style penalty *divides positive logits* and *multiplies negative ones* by 1.2. M1 returns raw logits (a likely token is about $+15 \to 12.5$, a strong penalty). M4 returned **log-probabilities** ($\le 0$; a likely token is about $-0.1 \to -0.12$, almost no penalty), so M4 was effectively decoding **without** a repetition penalty. Fix (Cell 47q): return the mixture on the logit scale, $\log p_{\text{mix}} + \mathrm{logsumexp}(z)$, which equals M1's logits exactly when $g_t = 0$, and leaves argmax and sampling unchanged.
2. **Self-copying.** An unrestricted pointer can also copy the model's *own* output. Fix (Cell 47p): an optional **prompt-only pointer** that attends only to prompt positions.

> **Lesson:** decoding heuristics that depend on the output scale (like the CTRL penalty) silently break when the output layer changes from logits to a mixture distribution.

In [9]:
# ===== Cell 47a: M4 decoding check -- learned copy alone vs + bonus, greedy vs MBR =====
assert "evaluate" in globals() and eval_local is not None, "run Cell 26 + restart, then Cell 39s"
if "model" in globals():
    del model; gc.collect(); torch.cuda.empty_cache()
model, ck = infer.load_model(f"{WORK}/checkpoints/m4_copy/best.pt")
print("M4 best checkpoint from step", ck["step"])

# ---- how much does the learned gate copy? mean g_t over 32 validation prompts --------------
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    ids = [tk.encode(VP[k]["text"], add_special_tokens=False).ids for k in DEV[:32]]
    L = min(map(len, ids))
    x = torch.tensor([i[-L:] for i in ids], device="cuda")
    gate = torch.sigmoid(model.gate(model._hidden(x).float())).float()
print(f"copy gate on validation prompts: mean {gate.mean():.3f} | "
      f"share of positions with g > 0.5: {(gate > 0.5).float().mean():.3f}")

res4, g4 = {}, {}
# ---- 1) greedy + rep 1.2, learned copy alone (beta 0) vs + decoding bonus (beta 1) -----------
for beta in (0.0, 1.0):
    t0 = time.time(); torch.manual_seed(1337)
    p, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                                  temperature=0, repetition_penalty=1.2, prompt_bonus=beta)
    g4[beta] = p
    res4[f"m4_greedy_rep1.2_copy{beta}"] = evaluate(f"m4_greedy_rep1.2_copy{beta}", p, t0)

# ---- 2) MBR over 8 samples + the matching greedy output -------------------------------------
for beta in (0.0, 1.0):
    t0 = time.time(); torch.manual_seed(1337)
    c, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                                  num_return=8, temperature=0.7, top_p=0.9,
                                  repetition_penalty=1.1, prompt_bonus=beta)
    picks = {k: infer.mbr_select(c[k] + [g4[beta][k]])[0] for k in DEV}
    res4[f"m4_mbr9_copy{beta}"] = evaluate(f"m4_mbr9_copy{beta}", picks, t0)

# ---- compare with M1 (same backbone, no copy head) and M2, then save -------------------------
m2r = json.load(open(f"{WORK}/stats/decoding_m2_v2.json"))
table = {"m1_greedy_rep1.2_copy0.0": results["greedy_rep1.2_copy0.0"],
         "m1_greedy_rep1.2_copy1.0": results["greedy_rep1.2_copy1.0"],
         "m1_mbr9_copy1.0": results["mbr9_copy1.0"], "m2_mbr9_copy1.0": m2r["m2_mbr9_copy1.0"], **res4}
display(pd.DataFrame(table).T[["bertscore_f1", "bertscore_f1[SC]", "rougeL_f1", "bleu4",
                               "copy", "rep3", "sec"]].round(4))
json.dump({"gate_mean": gate.mean().item(), **res4}, open(f"{WORK}/stats/decoding_m4_v2.json", "w"), indent=1)
best4 = max(res4, key=lambda k: res4[k]["bertscore_f1[SC]"])
print("best M4 setting on SC:", best4)

M4 best checkpoint from step 2670
copy gate on validation prompts: mean 0.097 | share of positions with g > 0.5: 0.059


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


m4_greedy_rep1.2_copy1.0 BERT 0.8237 (SC 0.8191) | ROUGE-L 0.2507 | BLEU 10.56 | copy 0.57 | rep3 0.584 | 260s


KeyboardInterrupt: 

In [ ]:
# ===== Cell 47b: M4 test-set predictions with the best setting from Cell 47a -> Codabench zip =====
RUN_NAME = "m4_copy"
USE_MBR  = best4.startswith("m4_mbr9")                   # chosen automatically from Cell 47a
BETA     = float(best4.split("copy")[-1])                 # 0.0 (learned copy only) or 1.0
TAG      = best4.replace("m4_", "")
print("submitting M4 with:", TAG)

test = json.load(open(f"{DATA}/Phase_1_Pretraining/Test/test.json"))
test_prompts = {k: v["text"] for k, v in test.items()}
t0 = time.time(); torch.manual_seed(1337)                 # model from Cell 47a is still loaded

# ---- greedy pass (final answer if no MBR, else the 9th candidate) -----------------------------
greedy, st = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask,
                                    temperature=0, repetition_penalty=1.2, prompt_bonus=BETA)
print(f"greedy done ({(time.time()-t0)/60:.1f} min)", flush=True)
preds = greedy
if USE_MBR:
    # 8 nucleus samples sharing one prefilled cache, then pick the most central candidate
    cands, st = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask,
                                       num_return=8, temperature=0.7, top_p=0.9,
                                       repetition_penalty=1.1, prompt_bonus=BETA)
    preds = {k: infer.mbr_select(cands[k] + [greedy[k]])[0] for k in test_prompts}
print("generation stats:", st, f"| {(time.time()-t0)/60:.1f} min")

# ---- write + verify the zip (1000 ids, exactly 128 words, prediction.json at root) ------------
name = f"{RUN_NAME}_{TAG}"
zp = infer.write_submission(preds, test.keys(), f"{SUB}/{name}", name)
json.dump({"model": RUN_NAME, "ckpt_step": ck["step"], "mbr": USE_MBR, "prompt_bonus": BETA},
          open(f"{SUB}/{name}/meta.json", "w"), indent=1)        # record of what was submitted
print("submission zip:", zp)
print("example:", preds[next(iter(preds))][:300])

In [10]:
# ===== Cell 47p: M4 decode-time controls -- copy only from the prompt, shift the copy gate =====
p = "/kaggle/working/src/model_copy.py"
s = open(p).read()
old1 = "        t = pos + T - 1                                                     # current position\n"
new1 = old1 + ("        if pos == 0:\n"
               "            self._prompt_len = T                                            # remember where the prompt ends\n")
old2 = '''            if t >= 1:
                hf = hl.float()
                g = self.gate(hf)                                           # (B, 1, 1)
                out = out + F.logsigmoid(-g)
                a = torch.softmax(self.copy_q(hf) @ cc.k[:, :t].float().transpose(1, 2)
                                  / math.sqrt(self.cfg.copy_dim), -1)       # (B, 1, t), keys i < t
                pc = torch.zeros_like(out).scatter_add_(2, cc.ids[:, None, 1:t + 1], a)
                out = torch.logaddexp(out, F.logsigmoid(g) + torch.log(pc))  # log 0 = -inf is fine
'''
new2 = '''            if t >= 1:
                hf = hl.float()
                g = self.gate(hf) + getattr(self, "copy_gate_bias", 0.0)   # decode-time gate shift
                out = out + F.logsigmoid(-g)
                # copy_prompt_only: point only at prompt tokens (no copying of the model's own output)
                n = min(t, self._prompt_len - 1) if getattr(self, "copy_prompt_only", False) else t
                if n >= 1:
                    a = torch.softmax(self.copy_q(hf) @ cc.k[:, :n].float().transpose(1, 2)
                                      / math.sqrt(self.cfg.copy_dim), -1)   # (B, 1, n), keys i < n
                    pc = torch.zeros_like(out).scatter_add_(2, cc.ids[:, None, 1:n + 1], a)
                    out = torch.logaddexp(out, F.logsigmoid(g) + torch.log(pc))
'''
if "copy_prompt_only" in s:
    print("already patched")
else:
    assert s.count(old1) == 1 and s.count(old2) == 1, "text not found -- send me this output"
    open(p, "w").write(s.replace(old1, new1).replace(old2, new2)); print("model_copy.py patched")
import py_compile; py_compile.compile(p, doraise=True); print("compiles OK")

model_copy.py patched
compiles OK


## Section 10c. Calibrating M4's copy head at decoding time

**Problem.** Free-running greedy decoding with M4 initially degenerated (**rep3 ≈ 0.54**). Two decode-time fixes, with **no retraining**:

1. **Logit-scale output** (Cell 47q, the main fix): the mixture is returned as $\log p_{\text{mix}} + \mathrm{logsumexp}(z)$, so the CTRL repetition penalty acts exactly as it does for M1.
2. **Decode-time controls** (Cell 47p):
   - **Prompt-only pointer:** the pointer attends only to prompt positions, never to the model's own output.
   - **Gate shift** $\delta$: added to the gate logit, $g_t = \sigma(\mathbf{w}^\top\mathbf{h}_t + b + \delta)$, so $\delta > 0$ means copy more and $\delta < 0$ means copy less.

**Results** (greedy, repetition penalty 1.2, 350 validation prompts; *copy* = share of generated bigrams found in the prompt):

| Setting | BERTScore F1 | SC | copy | rep3 |
|---|---|---|---|---|
| M1, no copy help | 0.8438 | 0.8405 | 0.30 | 0.052 |
| M1 + hand-tuned bonus $\beta = 1$ | 0.8453 | **0.8432** | 0.41 | 0.085 |
| M4 before fix (penalty bug) | 0.8243 | 0.8201 | 0.50 | **0.536** |
| M4 all-context pointer, $\delta = 0$ | 0.8418 | 0.8379 | 0.34 | 0.038 |
| M4 prompt-only, $\delta = -1$ | 0.8411 | 0.8373 | 0.24 | 0.011 |
| M4 prompt-only, $\delta = 0$ | 0.8437 | 0.8398 | 0.36 | 0.020 |
| **M4 prompt-only, $\delta = +1$** | **0.8460** | 0.8423 | 0.59 | 0.038 |
| M4 prompt-only, $\delta = +2$ | 0.8447 | 0.8410 | 0.77 | 0.061 |
| **M4 prompt-only, $\delta = +1$, MBR-9** | 0.8493 | **0.8459** | 0.64 | 0.054 |

**Findings.**
- **Fixing the penalty scale** removed the loops: rep3 went from 0.54 to 0.02–0.04, lower than M1's.
- **Prompt-only pointing beats the all-context pointer** (+0.002) and gives the least repetition of any model. It's the right inductive bias for continuing a given prompt.
- **The gate response is an inverted U** with its peak at $\delta = +1$. The likelihood-trained gate is slightly *under*-confident for BERTScore, and calibrating it by one scalar is enough.
- With $\delta = +1$ and MBR-9, M4 **matches the best hand-tuned decoding of M1 and M2** (SC 0.846, a three-way tie within noise) **without a hand-tuned prompt bonus**, and with about 35% less repetition.

In [12]:
# ===== Cell 47q: return M4's mixture on the same scale as M1's logits (makes rep-penalty work) =====
# CTRL repetition penalty divides POSITIVE logits; M4 returned log-probs (all <= 0), so the
# penalty was nearly a no-op. Adding logsumexp(z) gives exactly M1's logits when the gate is 0.
p = "/kaggle/working/src/model_copy.py"
s = open(p).read()
old = "                    out = torch.logaddexp(out, F.logsigmoid(g) + torch.log(pc))\n        return out\n"
new = ("                    out = torch.logaddexp(out, F.logsigmoid(g) + torch.log(pc))\n"
       "            out = out + torch.logsumexp(z, -1, keepdim=True)            # logit scale, as M1\n"
       "        return out\n")
if "logit scale, as M1" in s:
    print("already patched")
else:
    assert s.count(old) == 1, "text not found -- send me this output"
    open(p, "w").write(s.replace(old, new)); print("model_copy.py patched")
import py_compile; py_compile.compile(p, doraise=True); print("compiles OK")

model_copy.py patched
compiles OK


In [13]:
# ===== Cell 47c: M4 greedy with prompt-only copying and gate shifts (learned copy only, beta 0) =====
import importlib, model_copy
importlib.reload(model_copy)                               # pick up the patched forward()
if "model" in globals():
    del model; gc.collect(); torch.cuda.empty_cache()
model, ck = infer.load_model(f"{WORK}/checkpoints/m4_copy/best.pt")

res4b, g4b = {}, {}
for prompt_only, gb in [(False, 0.0), (True, 0.0), (True, -1.0), (False, -1.0)]:
    model.copy_prompt_only, model.copy_gate_bias = prompt_only, gb   # read inside forward()
    name = f"m4_greedy_{'prompt' if prompt_only else 'all'}copy_gate{gb:+.0f}"
    t0 = time.time(); torch.manual_seed(1337)
    p, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                                  temperature=0, repetition_penalty=1.2, prompt_bonus=0.0)
    g4b[(prompt_only, gb)] = p
    res4b[name] = evaluate(name, p, t0)

# ---- compare with the unrestricted M4 run and M1 (same backbone) -----------------------------
table = {"m1_greedy_rep1.2_copy0.0": results["greedy_rep1.2_copy0.0"],
         "m1_greedy_rep1.2_copy1.0": results["greedy_rep1.2_copy1.0"],
         "m1_mbr9_copy1.0": results["mbr9_copy1.0"], **res4, **res4b}
display(pd.DataFrame(table).T[["bertscore_f1", "bertscore_f1[SC]", "rougeL_f1", "bleu4",
                               "copy", "rep3", "sec"]].round(4))
json.dump({**res4, **res4b}, open(f"{WORK}/stats/decoding_m4_v2.json", "w"), indent=1)
best_cfg = max(g4b, key=lambda c: res4b[f"m4_greedy_{'prompt' if c[0] else 'all'}copy_gate{c[1]:+.0f}"]["bertscore_f1[SC]"])
print("best greedy config (prompt_only, gate_bias):", best_cfg)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


m4_greedy_allcopy_gate+0 BERT 0.8418 (SC 0.8379) | ROUGE-L 0.2281 | BLEU 12.13 | copy 0.34 | rep3 0.038 | 242s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


m4_greedy_promptcopy_gate+0 BERT 0.8437 (SC 0.8398) | ROUGE-L 0.2272 | BLEU 12.03 | copy 0.36 | rep3 0.020 | 199s


... (4 outputs left out)

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


m4_greedy_allcopy_gate-1 BERT 0.8404 (SC 0.8365) | ROUGE-L 0.2250 | BLEU 11.35 | copy 0.23 | rep3 0.026 | 231s


,bertscore_f1,bertscore_f1[SC],rougeL_f1,bleu4,copy,rep3,sec
m1_greedy_rep1.2_copy0.0,0.8438,0.8405,0.2348,12.4406,0.2966,0.0525,214.0
m1_greedy_rep1.2_copy1.0,0.8453,0.8432,0.2429,13.2072,0.4137,0.0852,213.0
m1_mbr9_copy1.0,0.8495,0.8462,0.2599,14.3971,0.4694,0.0862,1355.0
m4_greedy_rep1.2_copy0.0,0.8243,0.8201,0.2480,10.7647,0.4956,0.5356,293.0
m4_greedy_rep1.2_copy1.0,0.8237,0.8191,0.2507,10.5561,0.5735,0.5841,260.0
m4_greedy_allcopy_gate+0,0.8418,0.8379,0.2281,12.1250,0.3428,0.0383,242.0
m4_greedy_promptcopy_gate+0,0.8437,0.8398,0.2272,12.0338,0.3632,0.0196,199.0
m4_greedy_promptcopy_gate-1,0.8411,0.8373,0.2220,11.0846,0.2425,0.0109,221.0
m4_greedy_allcopy_gate-1,0.8404,0.8365,0.2250,11.3453,0.2335,0.0258,231.0


best greedy config (prompt_only, gate_bias): (True, 0.0)


In [16]:
# ===== Cell 47c2: push the learned gate UP (copy more), all-context pointer -- greedy =====
for prompt_only, gb in [(True, 1.0), (True, 2.0)]:
    model.copy_prompt_only, model.copy_gate_bias = prompt_only, gb
    name = f"m4_greedy_{'prompt' if prompt_only else 'all'}copy_gate{gb:+.0f}"
    t0 = time.time(); torch.manual_seed(1337)
    p, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                                  temperature=0, repetition_penalty=1.2, prompt_bonus=0.0)
    g4b[(prompt_only, gb)] = p
    res4b[name] = evaluate(name, p, t0)
json.dump({**res4, **res4b}, open(f"{WORK}/stats/decoding_m4_v2.json", "w"), indent=1)
best_cfg = max(g4b, key=lambda c: res4b[f"m4_greedy_{'prompt' if c[0] else 'all'}copy_gate{c[1]:+.0f}"]["bertscore_f1[SC]"])
print("best greedy config (prompt_only, gate_bias):", best_cfg)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


m4_greedy_promptcopy_gate+1 BERT 0.8460 (SC 0.8423) | ROUGE-L 0.2361 | BLEU 13.09 | copy 0.59 | rep3 0.038 | 197s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


m4_greedy_promptcopy_gate+2 BERT 0.8447 (SC 0.8410) | ROUGE-L 0.2371 | BLEU 13.21 | copy 0.77 | rep3 0.061 | 218s
best greedy config (prompt_only, gate_bias): (True, 1.0)


In [17]:
# ===== Cell 47d: MBR check with the best M4 config, then test predictions -> Codabench zip =====
po, gb = best_cfg
model.copy_prompt_only, model.copy_gate_bias = po, gb
cfg_tag = f"{'prompt' if po else 'all'}copy_gate{gb:+.0f}"
greedy_sc = res4b[f"m4_greedy_{cfg_tag}"]["bertscore_f1[SC]"]

# ---- 1) MBR on the validation subset (8 samples + the greedy output) ------------------------
t0 = time.time(); torch.manual_seed(1337)
c, _ = infer.continue_prompts(model, tk, dev_prompts, batch_size=16, first_mask=first_mask,
                              num_return=8, temperature=0.7, top_p=0.9, repetition_penalty=1.1, prompt_bonus=0.0)
picks = {k: infer.mbr_select(c[k] + [g4b[best_cfg][k]])[0] for k in DEV}
r = evaluate(f"m4_mbr9_{cfg_tag}", picks, t0)
json.dump({**res4, **res4b, f"m4_mbr9_{cfg_tag}": r}, open(f"{WORK}/stats/decoding_m4_v2.json", "w"), indent=1)
USE_MBR = r["bertscore_f1[SC]"] > greedy_sc
print(f"MBR SC {r['bertscore_f1[SC]']:.4f} vs greedy SC {greedy_sc:.4f} -> use MBR: {USE_MBR}", flush=True)

# ---- 2) test predictions with the winner ------------------------------------------------------
test = json.load(open(f"{DATA}/Phase_1_Pretraining/Test/test.json"))
test_prompts = {k: v["text"] for k, v in test.items()}
t0 = time.time(); torch.manual_seed(1337)
preds, st = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask,
                                   temperature=0, repetition_penalty=1.2, prompt_bonus=0.0)
if USE_MBR:
    cands, st = infer.continue_prompts(model, tk, test_prompts, batch_size=16, first_mask=first_mask,
                                       num_return=8, temperature=0.7, top_p=0.9,
                                       repetition_penalty=1.1, prompt_bonus=0.0)
    preds = {k: infer.mbr_select(cands[k] + [preds[k]])[0] for k in test_prompts}
print("generation stats:", st, f"| {(time.time()-t0)/60:.1f} min")

# ---- write + verify the zip ---------------------------------------------------------------------
name = f"m4_copy_{'mbr9' if USE_MBR else 'greedy'}_{cfg_tag}"
zp = infer.write_submission(preds, test.keys(), f"{SUB}/{name}", name)
json.dump({"model": "m4_copy", "ckpt_step": ck["step"], "mbr": USE_MBR,
           "copy_prompt_only": po, "copy_gate_bias": gb, "prompt_bonus": 0.0},
          open(f"{SUB}/{name}/meta.json", "w"), indent=1)
print("submission zip:", zp)
print("example:", preds[next(iter(preds))][:300])

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


m4_mbr9_promptcopy_gate+1 BERT 0.8493 (SC 0.8459) | ROUGE-L 0.2512 | BLEU 14.42 | copy 0.64 | rep3 0.054 | 1344s
MBR SC 0.8459 vs greedy SC 0.8423 -> use MBR: True
generation stats: {'n': 1000, 'left_truncated_in_batch': 87, 'short_rows_extended': 110} | 50.2 min
submission zip: /kaggle/working/submissions/m4_copy_mbr9_promptcopy_gate+1/m4_copy_mbr9_promptcopy_gate+1.zip
example: well settled that a person who has not approached the Court within reasonable time and, therefore, is not entitled to relief claimed in writ petition even if it is barred by limitation. (Emphasis added) 11. In [State of M.P. Vs. M.V. Joshi & Ors.] AIR 1997 SC 1468, this Court observed as under:- "In


In [18]:
import os, subprocess; from kaggle_secrets import UserSecretsClient
print(subprocess.run("python /kaggle/working/src/backup_hf.py", shell=True, capture_output=True, text=True,
      env={**os.environ, "HF_WRITE_TOKEN": UserSecretsClient().get_secret("HF_WRITE_TOKEN")}).stdout)

BACKUP DONE -> Utkarsh-1718-1/cs781-phase1-work in 0.5 min



In [2]:
   # ===== free ~3-4 GB: drop pretraining resume states of finished runs (best.pt files are kept) =====
   import os
   for r in ("m1_dense_100m", "m2_mla_moe", "m3_encdec"):
       p = f"/kaggle/working/checkpoints/{r}/latest.pt"
       if os.path.exists(p):
           print("removing", p, f"{os.path.getsize(p)/1e9:.1f} GB"); os.remove(p)
   !du -sh /kaggle/working/*

2.4G	/kaggle/working/checkpoints
3.1G	/kaggle/working/data
248K	/kaggle/working/src
357M	/kaggle/working/stats
5.7M	/kaggle/working/submissions
3.6M	/kaggle/working/tokenizer


In [3]:
# ===== Cell P0: list which phase folders / test files exist in the capstone dataset (no download) =====
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
from collections import Counter
api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
files = api.list_repo_files("Exploration-Lab/CS-781-Capstone", repo_type="dataset")
# group by top-level folder + split (Train/Test), so the output stays short
print(Counter("/".join(f.split("/")[:2]) for f in files))
for f in files:
    if "Phase_1" not in f:
        print(f)                                      # every non-Phase-1 file, in full

Counter({'Phase_2_LNER/Train': 5, 'Phase_3_RR/Train': 5, 'Phase_1_Pretraining/Train': 4, 'Phase_3_RR/Test': 2, 'Phase_4_CJPE/Train': 2, 'Phase_5_LSI/Train': 2, 'Phase_7_PCR/Dev': 2, 'Phase_7_PCR/Test': 2, 'Phase_7_PCR/Train': 2, '.gitattributes': 1, 'Phase_1_Pretraining/Test': 1, 'Phase_2_LNER/Test': 1, 'Phase_4_CJPE/Test': 1, 'Phase_5_LSI/Test': 1, 'Phase_6_Summ/Test': 1, 'Phase_6_Summ/Train': 1, 'README.md': 1})
.gitattributes
Phase_2_LNER/Test/fold3-text.json
Phase_2_LNER/Train/fold1-gold.json
Phase_2_LNER/Train/fold1-text.json
Phase_2_LNER/Train/fold2-gold.json
Phase_2_LNER/Train/fold2-text.json
Phase_2_LNER/Train/labels.txt
Phase_3_RR/Test/CL_test.json
Phase_3_RR/Test/IT_test.json
Phase_3_RR/Train/CL_dev.json
Phase_3_RR/Train/CL_train.json
Phase_3_RR/Train/IT_dev.json
Phase_3_RR/Train/IT_train.json
Phase_3_RR/Train/label_vocab.json
Phase_4_CJPE/Test/test.json
Phase_4_CJPE/Train/dev.json
Phase_4_CJPE/Train/train.json
Phase_5_LSI/Test/test.json
Phase_5_LSI/Train/dev.json
Phase_5_LSI

## Section 11. Ablations of M1's design choices

The course requires our own architectural decisions to be justified with ablations. M1 differs from a
plain LLaMA-style decoder in two choices we made, so we test each one by switching it off:

| Run | Change | Question |
|---|---|---|
| `abl_base` | none (M1 as designed) | reference at the reduced budget |
| `abl_noqk` | **QK-norm off** | does normalising queries/keys matter for fp16 stability and loss on a T4 (no bf16)? |
| `abl_untied` | **untied** input/output embeddings (+24.6M params) | does weight tying cost quality, or save parameters for free? |

**Controlled setup:** identical data, seed, curriculum and WSD schedule, but **100M tokens** each
(instead of 700M) to fit the compute budget. They're run on one GPU, in parallel with the Phase 2 experiments.
We compare SC validation loss (and bits per word) at equal tokens.

In [1]:
# ===== Cell 48: M1 ablations (QK-norm off, untied embeddings) -- background, GPU 1 only =====
import json, os, subprocess
WORK = "/kaggle/working"
base = {
    "arch": "dense", "data_dir": f"{WORK}/data",
    "seq_len": 1024, "micro_bs": 8, "global_bs_seqs": 256, "max_tokens": 100_000_000,   # reduced budget
    "lr": 6e-4, "min_lr_frac": 0.1, "warmup_frac": 0.03, "schedule": "wsd", "decay_frac": 0.2,
    "weight_decay": 0.1, "betas": [0.9, 0.95], "grad_clip": 1.0, "compile": True, "seed": 1337,
    "mixture": "curriculum", "log_every": 20, "eval_every": 50, "val_windows": 64,
    "ckpt_every_min": 15, "time_limit_hours": 11.0,
}
runs = {"abl_base": {}, "abl_noqk": {"qk_norm": False}, "abl_untied": {"tie_embeddings": False}}
steps = []
for name, model_cfg in runs.items():
    cfg = dict(base, run_name=name, out_dir=f"{WORK}/checkpoints/{name}", model=model_cfg)
    os.makedirs(cfg["out_dir"], exist_ok=True)
    json.dump(cfg, open(f"{cfg['out_dir']}/config.json", "w"), indent=1)
    steps.append(f"torchrun --standalone --nproc_per_node=1 src/train.py "
                 f"--config {cfg['out_dir']}/config.json >> {cfg['out_dir']}/stdout.log 2>&1")

# one background job: the three runs one after another ('; ' so a failure doesn't stop the rest)
script = "cd /kaggle/working && " + " ; ".join(steps)
subprocess.run(f"CUDA_VISIBLE_DEVICES=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True "
               f"setsid nohup bash -c '{script}' > /dev/null 2>&1 < /dev/null &", shell=True)
print("ablations launched on GPU 1:", list(runs))

ablations launched on GPU 1: ['abl_base', 'abl_noqk', 'abl_untied']


In [5]:
# ============================================================================================
# 48u (fixed) — Resume ONLY abl_untied on GPU 1 (auto-resumes from latest.pt at step 309)
#   '[t]rain.py' stops pgrep from matching its own shell command line.
# ============================================================================================
import os, subprocess, time
RUN = "/kaggle/working/checkpoints/abl_untied"
running = subprocess.run("pgrep -f '[t]rain.py --config'", shell=True, capture_output=True, text=True).stdout.split()
gpu1_busy = int(subprocess.run("nvidia-smi -i 1 --query-gpu=memory.used --format=csv,noheader,nounits",
                               shell=True, capture_output=True, text=True).stdout.strip() or 0) > 1000
if running or gpu1_busy:
    print("Training already active (PIDs", running, "| GPU1 busy:", gpu1_busy, ") -> NOT launching a second copy.")
else:
    cmd = (f"cd /kaggle/working/src && CUDA_VISIBLE_DEVICES=1 setsid nohup "
           f"python train.py --config {RUN}/run_config.json >> {RUN}/stdout.log 2>&1 < /dev/null &")
    subprocess.run(cmd, shell=True)
    print("launched abl_untied on GPU 1; waiting 90 s for start-up (torch.compile)...")
    time.sleep(90)

# ---- check: one process, RESUMED from step 309, GPU 1 in use -----------------------------------
print(subprocess.run(f"echo 'train.py processes:' $(pgrep -fc '[t]rain.py --config'); "
                     f"grep -n 'RESUMED' {RUN}/stdout.log | tail -1; tail -n 2 {RUN}/stdout.log; "
                     "nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv",
                     shell=True, capture_output=True, text=True).stdout)

launched abl_untied on GPU 1; waiting 90 s for start-up (torch.compile)...
train.py processes: 1
183:RESUMED from step 309 (81M tokens, 2.30 h so far)
W1004 10:27:32.521000 163 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
step    310/381 |    81.3M tok | loss 3.5465 | lr 5.94e-04 | gnorm 0.97 | scale 16384 | 4.3k tok/s | SC w 0.75 | 2.31 h
index, utilization.gpu [%], memory.used [MiB]
0, 0 %, 0 MiB
1, 100 %, 6379 MiB



In [10]:
# ===== Cell 48b: progress of the three ablation runs =====
for n in ("abl_base", "abl_noqk", "abl_untied"):
    print(f"--- {n}"); os.system(f"tail -n 2 /kaggle/working/checkpoints/{n}/stdout.log 2>/dev/null")
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

--- abl_base
    raise SignalException(f"Process {os.getpid()} got signal: {sigval}", sigval=sigval)
torch.distributed.elastic.multiprocessing.api.SignalException: Process 89 got signal: 15
--- abl_noqk
arch=dense params=100.1M (active/token 100.1M) | world=1 micro_bs=8 accum=32 | 262k tokens/step | 381 steps = 0.10B tokens
TRAINING COMPLETE: 0.10B tokens, best SC val loss 3.2698
--- abl_untied
  checkpoint saved at step 381
TRAINING COMPLETE: 0.10B tokens, best SC val loss 3.2925
index, utilization.gpu [%], memory.used [MiB]
0, 0 %, 155 MiB
1, 0 %, 3 MiB


In [2]:
# ============================================================================================
# R2 — Kill the accidentally relaunched ablations + inventory of local files and the HF backup
# ============================================================================================
import os, subprocess
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

# ---- 1. stop background training (setsid jobs survive a cell interrupt, so kill them by name) ----
print(subprocess.run("pkill -f 'train.py' ; pkill -f torchrun ; sleep 3 ; "
                     "ps aux | grep -E 'train.py|torchrun' | grep -v grep || echo 'no training processes left'",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv",
                     shell=True, capture_output=True, text=True).stdout)

# ---- 2. what exists locally right now ------------------------------------------------------------
for p in ["src", "tokenizer", "data_p2", "p2_ckpt", "submissions_p2", "checkpoints"]:
    full = f"/kaggle/working/{p}"
    print(f"local {p:<15}", sorted(os.listdir(full))[:12] if os.path.exists(full) else "MISSING")

# ---- 3. what is safe on HF (the backup is what we restore from) ----------------------------------
api = HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN"))
repo = f"{api.whoami()['name']}/cs781-phase1-work"
files = api.list_repo_files(repo)
print("\nHF p2_ckpt:      ", sorted(f.split('/')[-1] for f in files if f.startswith("p2_ckpt/")))
print("HF submissions:  ", sorted({f.split('/')[1] for f in files if f.startswith("submissions_p2/")}))
print("HF stats:        ", sorted(f for f in files if f.startswith("stats/p2")))
print("HF ablations:    ", sorted(f for f in files if f.startswith("checkpoints/abl_")))
last = api.list_repo_commits(repo)[0]
print("last backup commit:", last.created_at, "|", last.title)


index, utilization.gpu [%], memory.used [MiB]
0, 0 %, 0 MiB
1, 0 %, 105 MiB

local src             ['__pycache__', 'backup_hf.py', 'bpe_tokenizer.py', 'common.py', 'data.py', 'eval_local.py', 'generation.py', 'infer.py', 'lm_common.py', 'model_copy.py', 'model_dense.py', 'model_encdec.py']
local tokenizer       ['a1_bpe_v2_32k.json', 'hf_a1_bpe_v2_32k.json']
local data_p2         ['.cache', 'Phase_2_LNER', 'inlegalner', 'inlegalner_conv.json']
local p2_ckpt         ['kd_student_m1x6.pt', 'm1_full_bidir.pt', 'm1_full_bidir_ctrl.pt', 'm1_full_bidir_ext.pt', 'm1_full_bidir_ext_all70_s0.pt', 'm1_full_bidir_ext_all70_s1.pt', 'm1_full_bidir_ext_all70_s2.pt', 'm1_full_bidir_ext_s1.pt', 'm1_full_bidir_ext_s2.pt', 'm1_full_bidir_header.pt', 'm1_full_causal.pt', 'm1_lora.pt']
local submissions_p2  ['p2_ens3_full_bidir_ext', 'p2_ens4_final_all70', 'p2_ens4_full3_lora_ext', 'p2_kd_student_m1x6', 'p2_m1_full_bidir', 'p2_m1_full_bidir_ext', 'p2_m1_lora_cw', 'p2_m1_lora_cw_ext', 'p2_m1_lorma_pi', 'p

In [3]:
# ============================================================================================
# R3 — State of abl_untied: is there a resumable checkpoint, and does train.py support resuming?
# ============================================================================================
import os, subprocess
d = "/kaggle/working/checkpoints/abl_untied"
print("files:", {f: f"{os.path.getsize(os.path.join(d, f))/1e6:.0f} MB" for f in sorted(os.listdir(d))})
print("\n--- last lines of stdout.log ---")
print(subprocess.run(f"tail -n 5 {d}/stdout.log", shell=True, capture_output=True, text=True).stdout)
print("--- resume support in train.py ---")
print(subprocess.run("grep -n -i 'resume\\|latest.pt' /kaggle/working/src/train.py | head",
                     shell=True, capture_output=True, text=True).stdout)

files: {'best.pt': '249 MB', 'config.json': '0 MB', 'eval_log.jsonl': '0 MB', 'latest.pt': '1496 MB', 'run_config.json': '0 MB', 'stdout.log': '0 MB', 'train_log.csv': '0 MB'}

--- last lines of stdout.log ---
  new best SC val loss 3.5943 -> best.pt
  checkpoint saved at step 309
  checkpoint saved at step 309
step    320/381 |    83.9M tok | loss 3.4675 | lr 5.51e-04 | gnorm 0.35 | scale 16384 | 9.6k tok/s | SC w 0.75 | 2.37 h
step    320/381 |    83.9M tok | loss 3.4638 | lr 5.51e-04 | gnorm 0.41 | scale 16384 | 9.5k tok/s | SC w 0.75 | 2.38 h

--- resume support in train.py ---
5:| curriculum mixture | per-source val loss + bits/word | atomic latest.pt / best.pt | exact
6:auto-resume | clean stop before the Kaggle session limit | per-step model hook (MoE balancing).
122:    # ---- resume ----------------------------------------------------------------------------
124:    ck_path = f"{out_dir}/latest.pt"
130:        log(f"RESUMED from step {start_step} ({tokens_seen/1e6:.0f}M tokens

In [7]:
# ============================================================================================
# R4 — Rebuild the Python objects N17 needs after a session restart (NO training, nothing overwritten)
#   Same data split, class weights, metric helpers and hyper-parameters as N15b.
# ============================================================================================
import os, re, glob, json, time, random, sys
from collections import Counter
import numpy as np, torch
sys.path.insert(0, "/kaggle/working/src")
import p2_ner as ner, p2_model as pm, p2_peft as pp

WORK, DEV = "/kaggle/working", "cuda:0"; torch.cuda.set_device(0)
pm.HEADER = 0
BIDIR, METHOD = False, "lora"
LR, HEAD_LR, BS, EP_A, EP_B = 5e-4, 1e-3, 8, 2, 8

# ---- tokenizer (the HF-format file is the one tokenizers.Tokenizer can load) ----------------
from tokenizers import Tokenizer
tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")

# ---- our data: same folds + same deterministic dev split -------------------------------------
docs = []
for tp in sorted(glob.glob(f"{WORK}/data_p2/Phase_2_LNER/**/*text*.json", recursive=True)):
    if "/Test/" in tp:
        continue
    gp = next(tp.replace("text", x) for x in ("gold", "labels", "label", "ann")
              if os.path.exists(tp.replace("text", x)))
    docs += ner.load_fold(tp, gp, fold=os.path.basename(tp).split("-")[0])
train_docs, dev_docs = ner.dev_split(docs, 5)
ENC = pm.encode_all(train_docs + dev_docs, tk)
ours_ex = pm.make_examples(ENC, [d["id"] for d in train_docs], header=0)
print(f"{len(train_docs)} train / {len(dev_docs)} dev docs | {len(ours_ex)} windows (expect 60 / 10 / 1513)")

# ---- class weights (inverse sqrt frequency, capped) -------------------------------------------
cnt = np.bincount([t for _, ts in ours_ex for t in ts if t >= 0], minlength=len(ner.TAGS)).astype(float)
CW = torch.tensor(np.where(cnt > 0, np.sqrt(cnt.max() / np.maximum(cnt, 1)), 1.0), dtype=torch.float).clamp(max=10.0)

# ---- metric helpers (macro over labels present in dev gold) ----------------------------------
_strict = ner.strict_f1
def macro_of(r, gold):
    present = {e[2] for es in gold.values() for e in es}
    return float(np.mean([r["per_label_f1"][l] for l in ner.LABELS if l in present]))
def strict_f1_macro(pred, gold):
    r = _strict(pred, gold); r["micro"] = r["f1"]; r["f1"] = macro_of(r, gold); return r
def evaluate(model):
    gold = {d["id"]: d["ents"] for d in dev_docs}
    r = _strict(pm.predict_docs(model, dev_docs, ENC, device=DEV), gold)
    return {"macro": round(macro_of(r, gold), 4), "micro": round(r["f1"], 4), "per_label": r["per_label_f1"]}

# ---- sanity check: the saved LoRA control must reproduce its dev score (0.4945) -------------
m = pp.apply_method(pm.build_tagger(f"{WORK}/checkpoints/m1_dense_100m/best.pt", bidirectional=BIDIR), METHOD)
m.load_state_dict({k: v.float() for k, v in torch.load(f"{WORK}/p2_ckpt/m1_lora_cw_ctrl.pt", map_location="cpu").items()})
print("m1_lora_cw_ctrl dev re-check:", {k: v for k, v in evaluate(m.to(DEV)).items() if k != "per_label"}, "(expect macro 0.4945)")
del m; torch.cuda.empty_cache()

60 train / 10 dev docs | 1513 windows (expect 60 / 10 / 1513)
m1_lora_cw_ctrl dev re-check: {'macro': 0.4945, 'micro': 0.5872} (expect macro 0.4945)


In [26]:
# ===== Cell 48c: GPU usage, ablation process status, and the end of each ablation log =====
import subprocess, torch
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout
print(sh("nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv"))
print("notebook (N9) current CUDA device:", torch.cuda.current_device())
print("ablation process alive:", bool(sh("pgrep -f '[s]rc/train.py'").strip()))
for n in ("abl_base", "abl_noqk", "abl_untied"):
    print(f"--- {n}\n" + sh(f"tail -n 6 /kaggle/working/checkpoints/{n}/stdout.log 2>/dev/null"))

index, memory.used [MiB], utilization.gpu [%]
0, 181 MiB, 0 %
1, 3 MiB, 0 %

notebook (N9) current CUDA device: 0
ablation process alive: True
--- abl_base

--- abl_noqk

--- abl_untied

